# Clustering Evaluation: Complete Analysis

Unified evaluation notebook for the synthetic data clustering pipeline.
Compares K-Means and Hierarchical Clustering on real vs synthetic datasets
across Normal and Gamma distributions.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# Optional imports for statistical tests
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Plot style settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print('Libraries loaded successfully')

## 1. Load and Prepare Data

Load the simulation results from the parquet file and verify the data structure.

In [ ]:
# Load clustering results (parquet)
df = pd.read_parquet('../results/clustering_results.parquet')
print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
print("=" * 60)
print("DATA VALIDATION")
print("=" * 60)

print(f"\n✓ Total synthetic datasets processed: {len(df)}")

# --- Helper to convert numpy/pandas scalars to native Python types for nice printing ---
def _to_py_list(arr, sort=True):
    vals = np.asarray(arr)
    py = [v.item() if hasattr(v, "item") else v for v in vals]
    return sorted(py) if sort else py

N_vals = _to_py_list(df["N"].unique())
p_vals = _to_py_list(df["p"].unique())
k_vals = _to_py_list(df["k"].unique())
sep_vals = _to_py_list(df["sep"].unique())
rho_vals = _to_py_list(df["rho"].unique())
rep_vals = _to_py_list(df["rep"].unique())

num_real = len(N_vals) * len(p_vals) * len(k_vals) * len(sep_vals) * len(rho_vals)
num_rep = len(rep_vals)

# determine how many dataframe rows correspond to one (N,p,k,sep,rho,rep) entry
per_rep_counts = df.groupby(["N", "p", "k", "sep", "rho", "rep"]).size()
unique_per_rep = np.unique(per_rep_counts.values)
if unique_per_rep.size == 1:
    rows_per_rep = int(unique_per_rep[0])
    rows_info = f"{rows_per_rep} rows per (N,p,k,sep,rho,rep)"
else:
    rows_per_rep = None
    rows_info = f"varying rows per (N,p,k,sep,rho,rep): {unique_per_rep.tolist()}"

expected = num_real * num_rep * (rows_per_rep if rows_per_rep is not None else 1)

print(f"✓ Expected ({num_real} real × {num_rep} rep × {rows_info}): {expected}")
print(f"✓ Match: {'YES ✅' if len(df) == expected else 'NO ❌'}")

print("\n✓ Unique parameter combinations:")
print(f"  - N values: {N_vals}")
print(f"  - p values: {p_vals}")
print(f"  - k values: {k_vals}")
print(f"  - separation values: {sep_vals}")
print(f"  - rho values: {rho_vals}")

print(f"\n✓ Synthetic replications per real dataset (by rep value):")
print(f"  - Min rows per rep: {per_rep_counts.min()}")
print(f"  - Max rows per rep: {per_rep_counts.max()}")
print(f"  - Unique rep values: {num_rep}")

print("\n✅ Data validation complete")

---
## 2. Dimensionality Reduction: PCA Scatter Plots

2×3 grids showing Real vs Synthetic data with True Labels, K-Means,
and Hierarchical Clustering predictions — one grid per distribution.

In [ ]:
"""
Publication-Quality Visualization: Algorithm Comparison on Real vs Synthetic Data
=================================================================================

Generate two separate 2×3 grid visualizations:
- normal_analysis_grid.png: Normal distribution analysis
- gamma_analysis_grid.png: Gamma distribution analysis

Layout for each grid:
- Row 1: Real Data (Control)
- Row 2: Synthetic Data (Experiment)
- Column 1: True Labels (Ground Truth)
- Column 2: K-Means Predictions
- Column 3: Hierarchical Predictions
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# Data parameters
N = 1000
K = 2
SEPARATION = 2
RHO = 0
REP = 1
SYN_ID = 1

# File paths
BASE_PATH = '..'  # project root relative to 04_evaluation/
DATA_PATH_REAL = f'{BASE_PATH}/data/original'
DATA_PATH_SYN = f'{BASE_PATH}/data/synthetic'

# Output configuration
FIGSIZE = (15, 10)

# Styling
sns.set_style('whitegrid')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 10

# Distinct color schemes for different algorithms
colors_true = ['#2E86AB', '#A23B72']          # Blue/Purple for Ground Truth
colors_kmeans = ['#06A77D', '#D62246']        # Green/Red for K-Means
colors_hierarchical = ['#FF6B35', '#8B5A3C']  # Orange/Brown for Hierarchical
cmap_true = ListedColormap(colors_true)
cmap_kmeans = ListedColormap(colors_kmeans)
cmap_hierarchical = ListedColormap(colors_hierarchical)

print("="*70)
print("ALGORITHM COMPARISON: REAL vs SYNTHETIC DATA")
print("="*70)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def _fmt_rho(rho):
    """Format rho value to match filename convention (0.0 → '0', 0.4 → '0.4')"""
    return str(int(rho)) if rho == int(rho) else str(rho)

def load_dataset(distribution):
    """Load real and synthetic data for a given distribution"""
    rho_str = _fmt_rho(RHO)
    real_file = f'OD_N{N}_p10_k{K}_rho{rho_str}_sep{SEPARATION}_{distribution}.parquet'
    syn_file  = f'SD_cart_N{N}_p10_k{K}_rho{rho_str}_sep{SEPARATION}_{distribution}_syn{SYN_ID}.parquet'

    real_df = pd.read_parquet(f'{DATA_PATH_REAL}/{real_file}')
    real_df = real_df[real_df['rep'] == REP]

    syn_df = pd.read_parquet(f'{DATA_PATH_SYN}/{syn_file}')
    syn_df = syn_df[syn_df['rep'] == REP]

    feature_cols = [col for col in real_df.columns if col.startswith('X')]

    X_real = real_df[feature_cols].values
    y_real = real_df['group'].values.astype(int)

    X_syn = syn_df[feature_cols].values
    y_syn = syn_df['group'].values.astype(int)

    return X_real, y_real, X_syn, y_syn

def reduce_to_2d(X_real, X_syn):
    """Apply PCA to combined data for shared coordinate system"""
    X_combined = np.vstack([X_real, X_syn])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_combined)
    
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    
    n_real = X_real.shape[0]
    return X_pca[:n_real], X_pca[n_real:], pca.explained_variance_ratio_

def apply_kmeans(X_pca, k=2):
    """Apply K-Means clustering"""
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    return kmeans.fit_predict(X_pca)

def apply_hierarchical(X_pca, k=2):
    """Apply Hierarchical (Agglomerative) clustering"""
    hierarchical = AgglomerativeClustering(n_clusters=k, linkage='ward')
    return hierarchical.fit_predict(X_pca)

def plot_scatter(ax, X, labels, title, cmap, xlabel='PC1', ylabel='PC2', alpha=0.6):
    """Plot scatter with prominent centroids"""
    # Convert labels to 0-indexed if needed
    labels_indexed = labels - 1 if labels.min() == 1 else labels
    
    # Scatter plot
    scatter = ax.scatter(
        X[:, 0], X[:, 1],
        c=labels_indexed,
        cmap=cmap,
        alpha=alpha,
        s=30,
        edgecolors='white',
        linewidth=0.4
    )
    
    # Calculate and plot centroids
    for cluster_id in np.unique(labels):
        mask = labels == cluster_id
        centroid = X[mask].mean(axis=0)
        # PROMINENT CENTROID: Large white X with thick black outline
        ax.scatter(centroid[0], centroid[1], 
                  marker='X', s=500, c='white',
                  edgecolors='black', linewidth=3.5, zorder=10)
    
    ax.set_xlabel(xlabel, fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.grid(True, alpha=0.3)
    
    return scatter

def create_analysis_grid(X_real_pca, y_real, X_syn_pca, y_syn, distribution_name, output_file):
    """Create 2×3 grid for a single distribution"""
    
    print(f"\n{'='*60}")
    print(f"Creating {distribution_name.upper()} Distribution Grid")
    print(f"{'='*60}")
    
    # Apply clustering algorithms to both datasets
    print(f"  Running K-Means on Real data...")
    labels_kmeans_real = apply_kmeans(X_real_pca, K)
    
    print(f"  Running Hierarchical on Real data...")
    labels_hierarchical_real = apply_hierarchical(X_real_pca, K)
    
    print(f"  Running K-Means on Synthetic data...")
    labels_kmeans_syn = apply_kmeans(X_syn_pca, K)
    
    print(f"  Running Hierarchical on Synthetic data...")
    labels_hierarchical_syn = apply_hierarchical(X_syn_pca, K)
    
    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=FIGSIZE)
    
    # ========================================================================
    # ROW 1: REAL DATA (CONTROL)
    # ========================================================================
    
    # Column 1: True Labels
    plot_scatter(axes[0, 0], X_real_pca, y_real, 'True Labels', cmap_true)
    
    # Column 2: K-Means Predictions
    plot_scatter(axes[0, 1], X_real_pca, labels_kmeans_real + 1, 'K-Means Prediction', cmap_kmeans)
    
    # Column 3: Hierarchical Predictions
    plot_scatter(axes[0, 2], X_real_pca, labels_hierarchical_real + 1, 'Hierarchical Prediction', cmap_hierarchical)
    
    # Add row label
    axes[0, 0].annotate('REAL DATA\n(Control)', xy=(-0.25, 0.5), xycoords='axes fraction',
                        fontsize=12, fontweight='bold', rotation=90,
                        ha='center', va='center',
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.3))
    
    # ========================================================================
    # ROW 2: SYNTHETIC DATA (EXPERIMENT)
    # ========================================================================
    
    # Column 1: True Labels
    plot_scatter(axes[1, 0], X_syn_pca, y_syn, 'True Labels', cmap_true)
    
    # Column 2: K-Means Predictions
    plot_scatter(axes[1, 1], X_syn_pca, labels_kmeans_syn + 1, 'K-Means Prediction', cmap_kmeans)
    
    # Column 3: Hierarchical Predictions
    plot_scatter(axes[1, 2], X_syn_pca, labels_hierarchical_syn + 1, 'Hierarchical Prediction', cmap_hierarchical)
    
    # Add row label
    axes[1, 0].annotate('SYNTHETIC DATA\n(Experiment)', xy=(-0.25, 0.5), xycoords='axes fraction',
                        fontsize=12, fontweight='bold', rotation=90,
                        ha='center', va='center',
                        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.3))
    
    # ========================================================================
    # TITLE AND LEGEND
    # ========================================================================
    
    fig.suptitle(
        f'{distribution_name.upper()} Distribution: Clustering Algorithm Comparison\n'
        f'Real vs Synthetic Data (N={N}, k={K}, Separation={SEPARATION})',
        fontsize=16, fontweight='bold', y=0.98
    )
    
    # Create legend with three distinct colormaps
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    
    legend_elements = [
        Patch(facecolor=colors_true[0], edgecolor='white', label='True: C1'),
        Patch(facecolor=colors_true[1], edgecolor='white', label='True: C2'),
        Patch(facecolor=colors_kmeans[0], edgecolor='white', label='K-Means: C1'),
        Patch(facecolor=colors_kmeans[1], edgecolor='white', label='K-Means: C2'),
        Patch(facecolor=colors_hierarchical[0], edgecolor='white', label='Hierarchical: C1'),
        Patch(facecolor=colors_hierarchical[1], edgecolor='white', label='Hierarchical: C2'),
        Line2D([0], [0], marker='X', color='w', markerfacecolor='white', 
               markeredgecolor='black', markeredgewidth=2.5, markersize=15, label='Centroid')
    ]
    
    fig.legend(handles=legend_elements, loc='lower center', ncol=7, 
               fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.02),
               title='Legend', title_fontsize=10)
    
    plt.tight_layout(rect=[0.05, 0.02, 1, 0.95])
    
    return fig

# ============================================================================
# LOAD AND PROCESS NORMAL DISTRIBUTION
# ============================================================================

print("\n" + "="*70)
print("NORMAL DISTRIBUTION")
print("="*70)

print("\nLoading Normal distribution data...")
X_real_normal, y_real_normal, X_syn_normal, y_syn_normal = load_dataset('normal')
print(f"  Real: {X_real_normal.shape}, Synthetic: {X_syn_normal.shape}")

print("\nApplying PCA to Normal data...")
X_real_normal_pca, X_syn_normal_pca, var_normal = reduce_to_2d(X_real_normal, X_syn_normal)
print(f"  Variance explained: {var_normal.sum():.1%} (PC1: {var_normal[0]:.1%}, PC2: {var_normal[1]:.1%})")

fig_normal = create_analysis_grid(
    X_real_normal_pca, y_real_normal, 
    X_syn_normal_pca, y_syn_normal,
    'Normal', None
)

# ============================================================================
# LOAD AND PROCESS GAMMA DISTRIBUTION
# ============================================================================

print("\n" + "="*70)
print("GAMMA DISTRIBUTION")
print("="*70)

print("\nLoading Gamma distribution data...")
X_real_gamma, y_real_gamma, X_syn_gamma, y_syn_gamma = load_dataset('gamma')
print(f"  Real: {X_real_gamma.shape}, Synthetic: {X_syn_gamma.shape}")

print("\nApplying PCA to Gamma data...")
X_real_gamma_pca, X_syn_gamma_pca, var_gamma = reduce_to_2d(X_real_gamma, X_syn_gamma)
print(f"  Variance explained: {var_gamma.sum():.1%} (PC1: {var_gamma[0]:.1%}, PC2: {var_gamma[1]:.1%})")

fig_gamma = create_analysis_grid(
    X_real_gamma_pca, y_real_gamma, 
    X_syn_gamma_pca, y_syn_gamma,
    'Gamma', None
)

### 2.1 t-SNE Visualization

Same 2×3 grid layout using t-SNE instead of PCA.
Clustering is performed on the high-dimensional data, then projected.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

# Data parameters
N = 1000
K = 2
SEPARATION = 2
RHO = 0
REP = 1
SYN_ID = 1

# File paths (Adjust as needed)
BASE_PATH = '..'
DATA_PATH_REAL = f'{BASE_PATH}/data/original'
DATA_PATH_SYN = f'{BASE_PATH}/data/synthetic'

# Styling
sns.set_style('whitegrid')
FIGSIZE = (15, 10)

# Colors
colors_true = ['#2E86AB', '#A23B72']          # Blue/Purple
colors_kmeans = ['#06A77D', '#D62246']        # Green/Red
colors_hierarchical = ['#FF6B35', '#8B5A3C']  # Orange/Brown
cmap_true = ListedColormap(colors_true)
cmap_kmeans = ListedColormap(colors_kmeans)
cmap_hierarchical = ListedColormap(colors_hierarchical)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def _fmt_rho(rho):
    """Format rho value to match filename convention (0.0 → '0', 0.4 → '0.4')"""
    return str(int(rho)) if rho == int(rho) else str(rho)

def load_dataset(distribution):
    """Load real and synthetic data"""
    rho_str = _fmt_rho(RHO)
    real_file = f'OD_N{N}_p10_k{K}_rho{rho_str}_sep{SEPARATION}_{distribution}.parquet'
    syn_file  = f'SD_cart_N{N}_p10_k{K}_rho{rho_str}_sep{SEPARATION}_{distribution}_syn{SYN_ID}.parquet'

    real_df = pd.read_parquet(f'{DATA_PATH_REAL}/{real_file}')
    real_df = real_df[real_df['rep'] == REP]

    syn_df = pd.read_parquet(f'{DATA_PATH_SYN}/{syn_file}')
    syn_df = syn_df[syn_df['rep'] == REP]

    feature_cols = [col for col in real_df.columns if col.startswith('X')]

    X_real = real_df[feature_cols].values
    y_real = real_df['group'].values.astype(int)

    X_syn = syn_df[feature_cols].values
    y_syn = syn_df['group'].values.astype(int)

    return X_real, y_real, X_syn, y_syn

def compute_tsne(X_real, X_syn):
    """Compute t-SNE on COMBINED dataset"""
    X_combined = np.vstack([X_real, X_syn])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_combined)
    
    print("  Running t-SNE (this may take a moment)...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
    X_tsne = tsne.fit_transform(X_scaled)
    
    n_real = X_real.shape[0]
    return X_tsne[:n_real], X_tsne[n_real:]

def run_clustering_high_dim(X_real, X_syn, k=2):
    """Run clustering on HIGH-DIMENSIONAL data"""
    scaler = StandardScaler()
    X_combined = np.vstack([X_real, X_syn])
    scaler.fit(X_combined)
    
    X_real_scaled = scaler.transform(X_real)
    X_syn_scaled = scaler.transform(X_syn)
    
    # K-Means
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_real = kmeans.fit_predict(X_real_scaled)
    km_syn = kmeans.fit_predict(X_syn_scaled)
    
    # Hierarchical
    hc = AgglomerativeClustering(n_clusters=k, linkage='ward')
    hc_real = hc.fit_predict(X_real_scaled)
    hc_syn = hc.fit_predict(X_syn_scaled)
    
    return km_real, km_syn, hc_real, hc_syn

def plot_scatter(ax, X, labels, title, cmap, xlabel='t-SNE 1', ylabel='t-SNE 2'):
    labels_indexed = labels - 1 if labels.min() == 1 else labels
    
    ax.scatter(X[:, 0], X[:, 1], c=labels_indexed, cmap=cmap, 
               alpha=0.6, s=30, edgecolors='white', linewidth=0.3)
    
    for cluster_id in np.unique(labels_indexed):
        mask = labels_indexed == cluster_id
        if np.sum(mask) > 0:
            centroid = X[mask].mean(axis=0)
            ax.scatter(centroid[0], centroid[1], marker='X', s=400, c='white', 
                       edgecolors='black', linewidth=3, zorder=10)
            
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel(xlabel, fontweight='bold', fontsize=9)
    ax.set_ylabel(ylabel, fontweight='bold', fontsize=9)
    ax.grid(True, alpha=0.3)

def create_grid(X_real_tsne, y_real, X_syn_tsne, y_syn, 
                km_real, km_syn, hc_real, hc_syn, 
                dist_name):
    
    fig, axes = plt.subplots(2, 3, figsize=FIGSIZE)
    
    # ROW 1: REAL DATA
    plot_scatter(axes[0,0], X_real_tsne, y_real, "True Labels", cmap_true)
    plot_scatter(axes[0,1], X_real_tsne, km_real+1, "K-Means Pred", cmap_kmeans)
    plot_scatter(axes[0,2], X_real_tsne, hc_real+1, "Hierarchical Pred", cmap_hierarchical)
    
    axes[0,0].annotate("REAL DATA\n(Control)", xy=(-0.3, 0.5), xycoords='axes fraction', 
                       rotation=90, ha='center', va='center', fontweight='bold', fontsize=12,
                       bbox=dict(boxstyle="round,pad=0.3", fc="lightblue", alpha=0.3))

    # ROW 2: SYNTHETIC DATA
    plot_scatter(axes[1,0], X_syn_tsne, y_syn, "True Labels", cmap_true)
    plot_scatter(axes[1,1], X_syn_tsne, km_syn+1, "K-Means Pred", cmap_kmeans)
    plot_scatter(axes[1,2], X_syn_tsne, hc_syn+1, "Hierarchical Pred", cmap_hierarchical)
    
    axes[1,0].annotate("SYNTHETIC DATA\n(Experiment)", xy=(-0.3, 0.5), xycoords='axes fraction', 
                       rotation=90, ha='center', va='center', fontweight='bold', fontsize=12,
                       bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.3))

    # TITLES & LEGEND
    fig.suptitle(f"{dist_name} Distribution (t-SNE): Real vs Synthetic Clustering\n(N={N}, k={K}, Sep={SEPARATION})", 
                 fontsize=16, fontweight='bold', y=0.98)
    
    legend_elements = [
        Patch(facecolor=colors_true[0], label='True C1'), Patch(facecolor=colors_true[1], label='True C2'),
        Patch(facecolor=colors_kmeans[0], label='KM C1'), Patch(facecolor=colors_kmeans[1], label='KM C2'),
        Patch(facecolor=colors_hierarchical[0], label='HC C1'), Patch(facecolor=colors_hierarchical[1], label='HC C2'),
        Line2D([0], [0], marker='X', color='w', markerfacecolor='white', markeredgecolor='black', markersize=10, label='Centroid')
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=7, bbox_to_anchor=(0.5, 0.02))
    
    plt.tight_layout(rect=[0.05, 0.08, 1, 0.95])
    plt.show()  # Display directly in cell

# ============================================================================
# MAIN EXECUTION
# ============================================================================

print("Processing NORMAL Distribution...")
X_real_n, y_real_n, X_syn_n, y_syn_n = load_dataset('normal')
X_r_tsne_n, X_s_tsne_n = compute_tsne(X_real_n, X_syn_n)
km_r_n, km_s_n, hc_r_n, hc_s_n = run_clustering_high_dim(X_real_n, X_syn_n)
create_grid(X_r_tsne_n, y_real_n, X_s_tsne_n, y_syn_n, km_r_n, km_s_n, hc_r_n, hc_s_n, "NORMAL")

print("\nProcessing GAMMA Distribution...")
X_real_g, y_real_g, X_syn_g, y_syn_g = load_dataset('gamma')
X_r_tsne_g, X_s_tsne_g = compute_tsne(X_real_g, X_syn_g)
km_r_g, km_s_g, hc_r_g, hc_s_g = run_clustering_high_dim(X_real_g, X_syn_g)
create_grid(X_r_tsne_g, y_real_g, X_s_tsne_g, y_syn_g, km_r_g, km_s_g, hc_r_g, hc_s_g, "GAMMA")

---
## 3. Detection Success Rate Analysis

**Question:** How often does each algorithm correctly identify the true
number of clusters (k)?

In [ ]:
# Calculate success rates grouped by separation and k
success_rates = df.groupby(['sep', 'k']).agg({
    'success_kmeans': 'mean',
    'success_hc': 'mean'
}).reset_index()

# Pivot for heatmap format
heatmap_km = success_rates.pivot(index='k', columns='sep', values='success_kmeans')
heatmap_hc = success_rates.pivot(index='k', columns='sep', values='success_hc')

print("📊 Success Rates - K-Means:")
print(heatmap_km)
print("\n📊 Success Rates - Hierarchical Clustering:")
print(heatmap_hc)

In [ ]:
# Create side-by-side heatmap comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means Heatmap
sns.heatmap(heatmap_km, annot=True, fmt='.3f', cmap='BuGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Success Rate'},
            ax=axes[0], linewidths=0.5, linecolor='gray')
axes[0].set_title('K-Means: Detection Success Rate', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Cluster Separation', fontsize=12)
axes[0].set_ylabel('True Number of Clusters (k)', fontsize=12)

# Hierarchical Clustering Heatmap
sns.heatmap(heatmap_hc, annot=True, fmt='.3f', cmap='BuGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Success Rate'},
            ax=axes[1], linewidths=0.5, linecolor='gray')
axes[1].set_title('Hierarchical Clustering: Detection Success Rate', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Cluster Separation', fontsize=12)
axes[1].set_ylabel('True Number of Clusters (k)', fontsize=12)

plt.tight_layout()
plt.show()

### 3.1 Detection Success Rate — Normal Distributions

In [ ]:
# Calculate success rates grouped by separation and k
success_rates = df.groupby(['sep', 'k']).agg({
    'success_kmeans': 'mean',
    'success_hc': 'mean'
}).reset_index()

# Pivot for heatmap format
heatmap_km = success_rates.pivot(index='k', columns='sep', values='success_kmeans')
heatmap_hc = success_rates.pivot(index='k', columns='sep', values='success_hc')

# Create side-by-side heatmap comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means Heatmap
sns.heatmap(heatmap_km, annot=True, fmt='.3f', cmap='RdYlGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Success Rate'},
            ax=axes[0], linewidths=0.5, linecolor='gray', alpha = 0.8)
axes[0].set_title('K-Means', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Cluster Separation', fontsize=12)
axes[0].set_ylabel('True Number of Clusters (k)', fontsize=12)

# Hierarchical Clustering Heatmap
sns.heatmap(heatmap_hc, annot=True, fmt='.3f', cmap='RdYlGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Success Rate'},
            ax=axes[1], linewidths=0.5, linecolor='gray', alpha = 0.8)
axes[1].set_title('Hierarchical Clustering', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Cluster Separation', fontsize=12)
axes[1].set_ylabel('True Number of Clusters (k)', fontsize=12)
fig.suptitle('Detection Success Rate — Normal Distributions', fontsize=18, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()

### 3.2 Detection Success Rate — Gamma Distributions

In [ ]:
df_gamma = df[df['distribution'] == 'gamma'].copy()

# Calculate success rates for GAMMA distribution only
success_rates_gamma = df_gamma.groupby(['sep', 'k']).agg({
    'success_kmeans': 'mean',
    'success_hc': 'mean'
}).reset_index()

# Pivot for heatmap format
heatmap_km_gamma = success_rates_gamma.pivot(index='k', columns='sep', values='success_kmeans')
heatmap_hc_gamma = success_rates_gamma.pivot(index='k', columns='sep', values='success_hc')

# Create side-by-side heatmap comparison for GAMMA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means Heatmap (Gamma)
sns.heatmap(heatmap_km_gamma, annot=True, fmt='.3f', cmap='RdYlGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Success Rate'},
            ax=axes[0], linewidths=0.5, linecolor='gray', alpha = 0.8)
axes[0].set_title('K-Means', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Cluster Separation', fontsize=12)
axes[0].set_ylabel('True Number of Clusters (k)', fontsize=12)

# Hierarchical Clustering Heatmap (Gamma)
sns.heatmap(heatmap_hc_gamma, annot=True, fmt='.3f', cmap='RdYlGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Success Rate'},
            ax=axes[1], linewidths=0.5, linecolor='gray', alpha = 0.8)
axes[1].set_title('Hierarchical Clustering', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Cluster Separation', fontsize=12)
axes[1].set_ylabel('True Number of Clusters (k)', fontsize=12)
fig.suptitle('Detection Success Rate — Gamma Distributions', fontsize=18, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()

---
## 4. Performance Delta: HC vs K-Means

**Question:** Under what conditions does Hierarchical Clustering
outperform K-Means?

In [ ]:
# Calculate performance delta (HC - K-Means)
performance_delta = heatmap_hc - heatmap_km

print("📊 Performance Delta (HC - K-Means):")
print(performance_delta)
print("\n🔍 Interpretation:")
print("  Positive values = HC outperforms K-Means")
print("  Negative values = K-Means outperforms HC")
print("  Zero = Equal performance")

In [ ]:
# Create delta heatmap
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(performance_delta, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
            cbar_kws={'label': 'Performance Delta (HC - K-Means)'},
            linewidths=0.5, linecolor='gray', ax=ax)

ax.set_title('Hierarchical Clustering vs K-Means\nPerformance Advantage Map', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Cluster Separation', fontsize=12)
ax.set_ylabel('True Number of Clusters (k)', fontsize=12)

# Add interpretation text
fig.text(0.5, -0.05, 
         'Red = HC Outperforms | Blue = K-Means Outperforms | White = Equal',
         ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.show()

---
## 5. Quality Distortion: Silhouette Score Delta

**Question:** Does synthpop smooth/distort the cluster geometry?

In [ ]:
# Prepare data for boxplots
boxplot_data = df[['sep', 'k', 'diff_sil_km', 'diff_sil_hc']].copy()

# Reshape for easier plotting
boxplot_data_long = pd.melt(
    boxplot_data, 
    id_vars=['sep', 'k'], 
    value_vars=['diff_sil_km', 'diff_sil_hc'],
    var_name='Algorithm', 
    value_name='Silhouette_Delta'
)

# Rename for clarity
boxplot_data_long['Algorithm'] = boxplot_data_long['Algorithm'].map({
    'diff_sil_km': 'K-Means',
    'diff_sil_hc': 'Hierarchical'
})

print("📊 Sample of prepared data:")
print(boxplot_data_long.head(10))

In [ ]:
# Summary statistics
print("\n📊 QUALITY DISTORTION SUMMARY")
print("=" * 60)

for algo in ['K-Means', 'Hierarchical']:
    data = boxplot_data_long[boxplot_data_long['Algorithm'] == algo]['Silhouette_Delta']
    print(f"\n{algo}:")
    print(f"  Mean Delta: {data.mean():.4f}")
    print(f"  Median Delta: {data.median():.4f}")
    print(f"  Std Dev: {data.std():.4f}")
    print(f"  Min: {data.min():.4f} | Max: {data.max():.4f}")
    
print("\n🔍 Interpretation:")
print("  Negative values = Synthetic data has WORSE silhouette than real (expected)")
print("  Values near zero = Synthetic preserves cluster quality well")

In [ ]:
# Create boxplots by separation
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

separations = sorted(df['sep'].unique())

for idx, sep in enumerate(separations):
    subset = boxplot_data_long[boxplot_data_long['sep'] == sep]
    
    sns.boxplot(
        data=subset, 
        x='k', 
        y='Silhouette_Delta', 
        hue='Algorithm',
        palette={'K-Means': '#FF6B6B', 'Hierarchical': '#4ECDC4'},
        ax=axes[idx]
    )
    
    # Add reference line at 0
    axes[idx].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    axes[idx].set_title(f'Separation = {sep}', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('True Number of Clusters (k)', fontsize=11)
    axes[idx].set_ylabel('Silhouette Delta\n(Synthetic - Real)', fontsize=11)
    axes[idx].legend(title='Algorithm', loc='lower right')
    axes[idx].grid(axis='y', alpha=0.3)

fig.suptitle('Quality Distortion: How Much Does Synthpop Degrade Cluster Geometry?', 
             fontsize=18, fontweight='bold', y=1.00)

plt.tight_layout()
plt.show()

---
## 6. Summary Statistics & Key Findings

In [ ]:
# Generate comprehensive summary
print("=" * 70)
print("FINAL SUMMARY: SYNTHETIC DATA CLUSTERING ANALYSIS")
print("=" * 70)

# Overall success rates
overall_km = df['success_kmeans'].mean()
overall_hc = df['success_hc'].mean()

print(f"\n📊 OVERALL DETECTION SUCCESS RATES (across all 1800 synthetic datasets):")
print(f"   K-Means:              {overall_km:.2%}")
print(f"   Hierarchical:         {overall_hc:.2%}")
print(f"   Winner:               {'HC' if overall_hc > overall_km else 'K-Means' if overall_km > overall_hc else 'TIE'}")

# Best/worst scenarios
best_km = success_rates.loc[success_rates['success_kmeans'].idxmax()]
worst_km = success_rates.loc[success_rates['success_kmeans'].idxmin()]

print(f"\n🎯 K-MEANS PERFORMANCE:")
print(f"   Best:  k={int(best_km['k'])}, sep={best_km['sep']:.1f} → {best_km['success_kmeans']:.2%}")
print(f"   Worst: k={int(worst_km['k'])}, sep={worst_km['sep']:.1f} → {worst_km['success_kmeans']:.2%}")

best_hc = success_rates.loc[success_rates['success_hc'].idxmax()]
worst_hc = success_rates.loc[success_rates['success_hc'].idxmin()]

print(f"\n🎯 HIERARCHICAL CLUSTERING PERFORMANCE:")
print(f"   Best:  k={int(best_hc['k'])}, sep={best_hc['sep']:.1f} → {best_hc['success_hc']:.2%}")
print(f"   Worst: k={int(worst_hc['k'])}, sep={worst_hc['sep']:.1f} → {worst_hc['success_hc']:.2%}")

# Quality distortion
print(f"\n🔬 QUALITY DISTORTION (Silhouette Score Degradation):")
print(f"   K-Means Mean Delta:        {df['diff_sil_km'].mean():.4f}")
print(f"   Hierarchical Mean Delta:   {df['diff_sil_hc'].mean():.4f}")
print(f"   Interpretation: {'Synthetic data degrades cluster quality' if df['diff_sil_km'].mean() < 0 else 'Synthetic preserves quality'}")

# Statistical test
from scipy import stats
t_stat, p_value = stats.ttest_rel(df['diff_sil_km'], df['diff_sil_hc'])
print(f"\n📈 STATISTICAL SIGNIFICANCE (Paired t-test):")
print(f"   t-statistic: {t_stat:.4f}")
print(f"   p-value:     {p_value:.4e}")
print(f"   Result:      {'Algorithms differ significantly' if p_value < 0.05 else 'No significant difference'}")

print("\n" + "=" * 70)
print("=" * 70)

## 6. Research Conclusions

### Key Findings:

1. **Detection Capability**
   - Both algorithms struggle when cluster separation is low (< 2)
   - Performance improves dramatically as separation increases
   - Hierarchical clustering shows [better/worse/similar] performance overall

2. **Algorithm Comparison**
   - K-Means excels in scenarios with [describe conditions]
   - Hierarchical clustering dominates when [describe conditions]
   - The choice of algorithm matters most at [separation/k values]

3. **Synthetic Data Quality**
   - Synthpop (CART method) introduces measurable distortion
   - Silhouette scores typically [decrease/increase] by X%
   - Quality degradation is [more/less] severe at low separation values

### Implications for Practice:

- When using synthetic data for clustering research, expect [X]% reduction in detectability
- Researchers should validate findings on real data when separation < Y
- The CART synthesis method is [adequate/inadequate] for preserving cluster geometry

---

**End of Analysis**

In [ ]:
from scipy.stats import wilcoxon

# -------------------------------------------------------
# ANALYSIS 4: STATISTICAL SIGNIFICANCE (Wilcoxon Test)
# -------------------------------------------------------
print("\n" + "="*50)
print("📊 STATISTICAL SIGNIFICANCE TEST (K-Means vs HC)")
print("="*50)

# We compare the success rates row-by-row
# (Each row is a specific synthetic dataset)
stat, p_value = wilcoxon(df['success_kmeans'], df['success_hc'])

print(f"Overall Paired Comparison:")
print(f"  - Wilcoxon Statistic: {stat}")
print(f"  - P-Value: {p_value:.5e}")

if p_value < 0.05:
    print("✅ RESULT: The performance difference is STATISTICALLY SIGNIFICANT.")
else:
    print("❌ RESULT: The difference might be due to chance.")

# Breakdown by k
print("\nBreakdown by Cluster Count (k):")
for k_val in sorted(df['k'].unique()):
    subset = df[df['k'] == k_val]
    # Only test if there are differences to avoid errors
    if not (subset['success_kmeans'] == subset['success_hc']).all():
        _, p = wilcoxon(subset['success_kmeans'], subset['success_hc'])
        print(f"  - k={k_val}: p={p:.5e} {'*' if p<0.05 else ''}")
    else:
        print(f"  - k={k_val}: Identical performance (p=1.0)")

---
## 7. Publication-Quality Graphics

## 7. Publication-Quality Graphics

### Graphic 1: Global Performance Comparison (Boxplot)
Compare the overall success distribution of K-Means vs. Hierarchical Clustering across all conditions.

In [ ]:
# ============================================================================
# GRAPHIC 1: Global Performance Comparison (Boxplot)
# Goal: Compare overall success distribution of K-Means vs Hierarchical Clustering
# ============================================================================

# Reshape data to long format for boxplot
df_long = pd.melt(
    df,
    id_vars=['N', 'p', 'k', 'rho', 'sep', 'rep', 'syn_idx'],
    value_vars=['success_kmeans', 'success_hc'],
    var_name='Algorithm',
    value_name='Success'
)

# Clean algorithm names
df_long['Algorithm'] = df_long['Algorithm'].map({
    'success_kmeans': 'K-Means',
    'success_hc': 'Hierarchical'
})

# Publication-quality color palette (colorblind-friendly)
colors = {'K-Means': '#E64B35', 'Hierarchical': '#4DBBD5'}

# Create figure
fig, ax = plt.subplots(figsize=(10, 7))

# Create boxplot with individual k as hue
sns.boxplot(
    data=df_long,
    x='Algorithm',
    y='Success',
    hue='k',
    palette='Set2',
    width=0.6,
    linewidth=1.5,
    ax=ax
)

# Overlay strip plot for individual points
sns.stripplot(
    data=df_long.groupby(['Algorithm', 'k', 'sep', 'rho']).mean().reset_index(),
    x='Algorithm',
    y='Success',
    hue='k',
    palette='Set2',
    dodge=True,
    alpha=0.4,
    size=4,
    ax=ax,
    legend=False
)

# Styling
ax.set_xlabel('Clustering Algorithm', fontsize=14, fontweight='bold')
ax.set_ylabel('Detection Success Rate', fontsize=14, fontweight='bold')
ax.set_title('Global Performance Comparison:\nK-Means vs Hierarchical Clustering', 
             fontsize=16, fontweight='bold', pad=15)
ax.legend(title='True Clusters ($k$)', title_fontsize=11, fontsize=10, 
          loc='lower right', framealpha=0.95)
ax.set_ylim(-0.05, 1.1)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.tick_params(axis='both', labelsize=11)

# Add statistical annotation
from scipy.stats import mannwhitneyu
km_success = df['success_kmeans'].values
hc_success = df['success_hc'].values
stat, pval = mannwhitneyu(km_success, hc_success, alternative='two-sided')
significance = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."

# Add annotation
ax.annotate(f'Mann-Whitney U: p = {pval:.2e} ({significance})', 
            xy=(0.5, 0.02), xycoords='axes fraction',
            ha='center', fontsize=10, style='italic', 
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### Graphic 2: Confusion Matrix with Hungarian Algorithm Alignment
Visualize precise misclassifications for a representative run (N=1000, k=3, rho=0.4) using optimal cluster alignment.

In [ ]:
# ============================================================================
# GRAPHIC 2: Confusion Matrix with Hungarian Algorithm Alignment
# Goal: Visualize misclassifications with optimal cluster-to-label matching
# ============================================================================

from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import os

# Step 1: Load a representative dataset (N=1000, k=3, rho=0.4, high separation)
target_file = "../data/original/OD_N1000_p10_k3_rho0.4_sep2_normal.parquet"

if os.path.exists(target_file):
    sample_data = pd.read_parquet(target_file) if target_file.endswith('.parquet') else pd.read_csv(target_file)
    
    # Extract features and true labels
    X = sample_data.drop(columns=['group', 'rep'], errors='ignore').values
    true_labels = sample_data['group'].astype(int).values
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Perform K-Means clustering
    kmeans = KMeans(n_clusters=3, n_init=25, random_state=42)
    pred_labels_km = kmeans.fit_predict(X_scaled)
    
    # Perform Hierarchical clustering
    hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
    pred_labels_hc = hc.fit_predict(X_scaled)
    
    def align_labels_hungarian(true_labels, pred_labels):
        """
        Use Hungarian Algorithm to optimally align predicted clusters to true labels.
        """
        # Step 2: Create raw confusion matrix
        cm = confusion_matrix(true_labels, pred_labels)
        
        # Step 3: Create cost matrix (we want to maximize matches, algorithm minimizes)
        cost_matrix = cm.max() - cm
        
        # Step 4: Apply Hungarian algorithm
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        
        # Step 5: Reorder columns to align diagonal
        cm_aligned = cm[:, col_ind]
        
        return cm_aligned, col_ind
    
    # Align both algorithms
    cm_km_aligned, mapping_km = align_labels_hungarian(true_labels, pred_labels_km)
    cm_hc_aligned, mapping_hc = align_labels_hungarian(true_labels, pred_labels_hc)
    
    # Create side-by-side confusion matrices
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # K-Means Confusion Matrix
    sns.heatmap(cm_km_aligned, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'Pred {i}' for i in range(3)],
                yticklabels=[f'True {i+1}' for i in range(3)],
                ax=axes[0], cbar_kws={'label': 'Count'},
                linewidths=1, linecolor='white', annot_kws={'size': 14, 'weight': 'bold'})
    axes[0].set_xlabel('Predicted Cluster (Aligned)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('True Cluster Label', fontsize=12, fontweight='bold')
    axes[0].set_title('K-Means\nOptimal Cluster Alignment (Hungarian Method)', 
                      fontsize=14, fontweight='bold')
    
    # Calculate accuracy for K-Means
    accuracy_km = np.diag(cm_km_aligned).sum() / cm_km_aligned.sum()
    axes[0].text(0.5, -0.12, f'Alignment Accuracy: {accuracy_km:.1%}', 
                 transform=axes[0].transAxes, ha='center', fontsize=11, style='italic')
    
    # Hierarchical Confusion Matrix
    sns.heatmap(cm_hc_aligned, annot=True, fmt='d', cmap='Oranges',
                xticklabels=[f'Pred {i}' for i in range(3)],
                yticklabels=[f'True {i+1}' for i in range(3)],
                ax=axes[1], cbar_kws={'label': 'Count'},
                linewidths=1, linecolor='white', annot_kws={'size': 14, 'weight': 'bold'})
    axes[1].set_xlabel('Predicted Cluster (Aligned)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('True Cluster Label', fontsize=12, fontweight='bold')
    axes[1].set_title('Hierarchical Clustering\nOptimal Cluster Alignment (Hungarian Method)', 
                      fontsize=14, fontweight='bold')
    
    # Calculate accuracy for HC
    accuracy_hc = np.diag(cm_hc_aligned).sum() / cm_hc_aligned.sum()
    axes[1].text(0.5, -0.12, f'Alignment Accuracy: {accuracy_hc:.1%}', 
                 transform=axes[1].transAxes, ha='center', fontsize=11, style='italic')
    
    fig.suptitle(f'Confusion Matrices with Hungarian Algorithm Alignment\n(N=1000, k=3, ρ=0.4, sep=2)', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()
    print(f"\n📊 Results Summary:")
    print(f"   K-Means Accuracy:      {accuracy_km:.2%}")
    print(f"   Hierarchical Accuracy: {accuracy_hc:.2%}")
    print(f"   Cluster mapping (K-Means): {mapping_km}")
    print(f"   Cluster mapping (HC):      {mapping_hc}")
else:
    print(f"⚠️ File not found: {target_file}")

### Graphic 3: Scalability Analysis (Execution Time vs N)
Demonstrate computational cost difference as sample size increases. K-Means should show linear scaling while Hierarchical shows exponential growth.

In [ ]:
# ============================================================================
# GRAPHIC 3: Scalability Analysis (Execution Time vs N)
# Goal: Show computational cost difference as sample size increases
# ============================================================================

import time
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

# Since execution time wasn't recorded in original data, we'll benchmark it now
# Using representative synthetic datasets

sample_sizes = [100, 250, 500, 750, 1000]
n_trials = 5  # Number of trials per configuration
k_test = 3  # Fixed number of clusters for benchmarking
p_test = 10  # Fixed dimensionality

# Storage for timing results
timing_results = []

print("🔄 Running scalability benchmark...")
print("=" * 50)

for N in sample_sizes:
    for trial in range(n_trials):
        # Generate synthetic data with similar properties
        np.random.seed(trial * 100 + N)
        X = np.random.randn(N, p_test)
        
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Time K-Means
        start = time.perf_counter()
        kmeans = KMeans(n_clusters=k_test, n_init=10, random_state=42)
        kmeans.fit(X_scaled)
        time_km = time.perf_counter() - start
        
        # Time Hierarchical Clustering
        start = time.perf_counter()
        hc = AgglomerativeClustering(n_clusters=k_test, linkage='ward')
        hc.fit(X_scaled)
        time_hc = time.perf_counter() - start
        
        timing_results.append({
            'N': N,
            'trial': trial,
            'time_kmeans': time_km,
            'time_hc': time_hc
        })
    
    print(f"  N={N:4d}: K-Means={np.mean([r['time_kmeans'] for r in timing_results if r['N']==N]):.4f}s, "
          f"HC={np.mean([r['time_hc'] for r in timing_results if r['N']==N]):.4f}s")

# Create DataFrame
df_timing = pd.DataFrame(timing_results)

# Aggregate by N
timing_agg = df_timing.groupby('N').agg({
    'time_kmeans': ['mean', 'std'],
    'time_hc': ['mean', 'std']
}).reset_index()
timing_agg.columns = ['N', 'km_mean', 'km_std', 'hc_mean', 'hc_std']

# Create publication-quality plot
fig, ax = plt.subplots(figsize=(10, 7))

# Plot K-Means (solid line)
ax.errorbar(timing_agg['N'], timing_agg['km_mean'], yerr=timing_agg['km_std'],
            fmt='o-', color='#E64B35', linewidth=2.5, markersize=10, 
            capsize=5, capthick=2, label='K-Means', markeredgecolor='white', markeredgewidth=1.5)

# Plot Hierarchical (dashed line)
ax.errorbar(timing_agg['N'], timing_agg['hc_mean'], yerr=timing_agg['hc_std'],
            fmt='s--', color='#4DBBD5', linewidth=2.5, markersize=10,
            capsize=5, capthick=2, label='Hierarchical Clustering', 
            markeredgecolor='white', markeredgewidth=1.5)

# Add trend annotations
ax.annotate('O(n)', xy=(800, timing_agg[timing_agg['N']==1000]['km_mean'].values[0]),
            fontsize=11, color='#E64B35', fontweight='bold',
            xytext=(15, 0), textcoords='offset points')
ax.annotate('O(n²)', xy=(800, timing_agg[timing_agg['N']==1000]['hc_mean'].values[0]),
            fontsize=11, color='#4DBBD5', fontweight='bold',
            xytext=(15, 0), textcoords='offset points')

# Styling
ax.set_xlabel('Sample Size ($N$)', fontsize=14, fontweight='bold')
ax.set_ylabel('Execution Time (seconds)', fontsize=14, fontweight='bold')
ax.set_title('Scalability Analysis:\nComputational Cost vs Sample Size', 
             fontsize=16, fontweight='bold', pad=15)
ax.legend(fontsize=12, loc='upper left', framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(axis='both', labelsize=11)
ax.set_xlim(50, 1100)

# Add ratio annotation at N=1000
ratio = timing_agg[timing_agg['N']==1000]['hc_mean'].values[0] / timing_agg[timing_agg['N']==1000]['km_mean'].values[0]
ax.text(0.95, 0.05, f'HC/K-Means ratio at N=1000: {ratio:.1f}×',
        transform=ax.transAxes, ha='right', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### Graphic 4: Sensitivity Analysis (Stability vs ρ)
Show algorithm robustness to noise/correlation (ρ). Highlights performance at the new parameter ρ = 0.4.

In [ ]:
# ============================================================================
# GRAPHIC 4: Sensitivity Analysis (Stability vs ρ - Noise/Correlation)
# Goal: Show algorithm robustness to noise parameter rho
# ============================================================================

# Aggregate by rho
rho_analysis = df.groupby('rho').agg({
    'success_kmeans': ['mean', 'std'],
    'success_hc': ['mean', 'std'],
    'sil_real_km': ['mean'],  # Using silhouette as additional quality metric
}).reset_index()

rho_analysis.columns = ['rho', 'km_mean', 'km_std', 'hc_mean', 'hc_std', 'sil_mean']

print("📊 Performance by Correlation/Noise Level (ρ):")
print(rho_analysis.to_string(index=False))

# Create publication-quality plot
fig, ax = plt.subplots(figsize=(10, 7))

# Get unique rho values
rho_values = rho_analysis['rho'].values

# Plot K-Means with error ribbon
ax.fill_between(rho_values, 
                rho_analysis['km_mean'] - rho_analysis['km_std'],
                rho_analysis['km_mean'] + rho_analysis['km_std'],
                alpha=0.2, color='#E64B35')
ax.plot(rho_values, rho_analysis['km_mean'], 'o-', color='#E64B35', 
        linewidth=2.5, markersize=12, label='K-Means',
        markeredgecolor='white', markeredgewidth=2)

# Plot Hierarchical with error ribbon
ax.fill_between(rho_values,
                rho_analysis['hc_mean'] - rho_analysis['hc_std'],
                rho_analysis['hc_mean'] + rho_analysis['hc_std'],
                alpha=0.2, color='#4DBBD5')
ax.plot(rho_values, rho_analysis['hc_mean'], 's--', color='#4DBBD5',
        linewidth=2.5, markersize=12, label='Hierarchical Clustering',
        markeredgecolor='white', markeredgewidth=2)

# Highlight rho = 0.4 with vertical line
ax.axvline(x=0.4, color='#7E6148', linestyle=':', linewidth=2.5, alpha=0.8)
ax.annotate('ρ = 0.4\n(High Noise)', xy=(0.4, ax.get_ylim()[1] * 0.15),
            fontsize=11, ha='center', color='#7E6148', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#7E6148', alpha=0.9))

# Add data labels for key points
for idx, row in rho_analysis.iterrows():
    ax.annotate(f'{row["km_mean"]:.2f}', 
                xy=(row['rho'], row['km_mean']), 
                xytext=(0, 12), textcoords='offset points',
                fontsize=9, ha='center', color='#E64B35')
    ax.annotate(f'{row["hc_mean"]:.2f}', 
                xy=(row['rho'], row['hc_mean']), 
                xytext=(0, -18), textcoords='offset points',
                fontsize=9, ha='center', color='#4DBBD5')

# Styling
ax.set_xlabel('Correlation Parameter ($\\rho$)', fontsize=14, fontweight='bold')
ax.set_ylabel('Detection Success Rate (Mean ± SD)', fontsize=14, fontweight='bold')
ax.set_title('Sensitivity Analysis:\nAlgorithm Robustness to Feature Correlation', 
             fontsize=16, fontweight='bold', pad=15)
ax.legend(fontsize=12, loc='lower left', framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle='--')
ax.tick_params(axis='both', labelsize=11)
ax.set_xlim(-0.05, max(rho_values) + 0.05)
ax.set_ylim(0, 1.15)

# Add performance drop annotation
if len(rho_values) > 1:
    drop_km = rho_analysis[rho_analysis['rho']==0]['km_mean'].values[0] - rho_analysis[rho_analysis['rho']==0.4]['km_mean'].values[0]
    drop_hc = rho_analysis[rho_analysis['rho']==0]['hc_mean'].values[0] - rho_analysis[rho_analysis['rho']==0.4]['hc_mean'].values[0]
    
    ax.text(0.97, 0.97, 
            f'Performance drop (ρ=0 → ρ=0.4):\n  K-Means: {drop_km:+.1%}\n  HC: {drop_hc:+.1%}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray', alpha=0.9))

plt.tight_layout()
plt.show()

### Summary: All Publication Graphics Generated

| Graphic | Description | File |
|---------|-------------|------|
| 1 | Global Performance Boxplot (K-Means vs HC by k) | `graphic1_global_boxplot.png` |
| 2 | Confusion Matrix with Hungarian Algorithm | `graphic2_confusion_matrix_hungarian.png` |
| 3 | Scalability Analysis (Time vs N) | `graphic3_scalability_time_vs_n.png` |
| 4 | Sensitivity Analysis (Success vs ρ) | `graphic4_sensitivity_rho.png` |

All graphics saved at **300 DPI** for publication quality.

---
## 8. Advanced Cluster Validation Graphics

### Graphic A: Cluster Number Matching (Confusion Matrix with Hungarian Alignment)
Visualize the alignment and specific misclassifications using optimal cluster-to-label matching.

In [ ]:
# ============================================================================
# GRAPHIC A: Cluster Number Matching (Confusion Matrix with Hungarian Alignment)
# Goal: Visualize alignment and misclassifications for a representative run
# Uses Hungarian Algorithm for optimal cluster-to-label matching
# ============================================================================

import os
import glob
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set academic whitegrid style
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# Find a representative dataset: N=1000, k=3, rho=0.4, moderate separation
data_dir = "../data/original"
target_params = {'N': 1000, 'k': 3, 'rho': 0.4, 'sep': 2}

# Search for matching file
pattern = f"OD_N{target_params['N']}_p*_k{target_params['k']}_rho{target_params['rho']}_sep{target_params['sep']}_*.parquet"
files = glob.glob(os.path.join(data_dir, pattern))

if files:
    target_file = files[0]
    print(f"📂 Loading: {target_file}")
    sample_data = pd.read_parquet(target_file) if target_file.endswith('.parquet') else pd.read_csv(target_file)
    
    # Extract features and true labels
    X = sample_data.drop(columns=['group', 'rep'], errors='ignore').values
    true_labels = sample_data['group'].astype(int).values
    
    # Convert to 0-indexed for consistency
    true_labels_0idx = true_labels - true_labels.min()
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    k_true = len(np.unique(true_labels))
    
    # Perform K-Means clustering
    kmeans = KMeans(n_clusters=k_true, n_init=25, random_state=42)
    pred_km = kmeans.fit_predict(X_scaled)
    
    # Perform Hierarchical clustering
    hc = AgglomerativeClustering(n_clusters=k_true, linkage='ward')
    pred_hc = hc.fit_predict(X_scaled)
    
    def hungarian_align_confusion_matrix(true_labels, pred_labels):
        """
        Create confusion matrix and apply Hungarian Algorithm for optimal alignment.
        Returns aligned confusion matrix and the optimal column ordering.
        """
        # Create raw confusion matrix
        cm = confusion_matrix(true_labels, pred_labels)
        
        # Hungarian algorithm minimizes cost, so we negate to maximize matches
        cost_matrix = -cm
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        
        # Reorder columns according to Hungarian solution
        cm_aligned = cm[:, col_ind]
        
        return cm_aligned, col_ind
    
    # Align both algorithms
    cm_km_aligned, mapping_km = hungarian_align_confusion_matrix(true_labels_0idx, pred_km)
    cm_hc_aligned, mapping_hc = hungarian_align_confusion_matrix(true_labels_0idx, pred_hc)
    
    # Calculate accuracies
    accuracy_km = np.diag(cm_km_aligned).sum() / cm_km_aligned.sum()
    accuracy_hc = np.diag(cm_hc_aligned).sum() / cm_hc_aligned.sum()
    
    # Create publication-quality figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # K-Means Confusion Matrix
    sns.heatmap(cm_km_aligned, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'Cluster {i}' for i in range(k_true)],
                yticklabels=[f'True {i+1}' for i in range(k_true)],
                ax=axes[0], cbar_kws={'label': 'Count'},
                linewidths=1.5, linecolor='white', 
                annot_kws={'size': 16, 'weight': 'bold'})
    axes[0].set_xlabel('Predicted Cluster (Hungarian-Aligned)', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('True Group Label', fontsize=12, fontweight='bold')
    axes[0].set_title(f'K-Means\nAccuracy: {accuracy_km:.1%}', fontsize=14, fontweight='bold')
    
    # Hierarchical Confusion Matrix
    sns.heatmap(cm_hc_aligned, annot=True, fmt='d', cmap='Oranges',
                xticklabels=[f'Cluster {i}' for i in range(k_true)],
                yticklabels=[f'True {i+1}' for i in range(k_true)],
                ax=axes[1], cbar_kws={'label': 'Count'},
                linewidths=1.5, linecolor='white',
                annot_kws={'size': 16, 'weight': 'bold'})
    axes[1].set_xlabel('Predicted Cluster (Hungarian-Aligned)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('True Group Label', fontsize=12, fontweight='bold')
    axes[1].set_title(f'Hierarchical Clustering\nAccuracy: {accuracy_hc:.1%}', fontsize=14, fontweight='bold')
    
    fig.suptitle(f'Graphic A: Confusion Matrix with Hungarian Algorithm Alignment\n'
                 f'(N={target_params["N"]}, k={target_params["k"]}, ρ={target_params["rho"]}, sep={target_params["sep"]})', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()
    print(f"\n📊 Results:")
    print(f"   K-Means Accuracy: {accuracy_km:.2%}")
    print(f"   Hierarchical Accuracy: {accuracy_hc:.2%}")
    print(f"   K-Means mapping (pred→true): {mapping_km}")
    print(f"   HC mapping (pred→true): {mapping_hc}")
else:
    print(f"⚠️ No files found matching pattern: {pattern}")

### Graphic B: Gini Coefficient (Cluster Size Balance)
Measure cluster size inequality - a Gini of 0 means perfect balance, approaching 1 means severe "cluster collapse".

In [ ]:
# ============================================================================
# GRAPHIC B: Gini Coefficient (Cluster Size Balance)
# Goal: Detect "cluster collapse" (one giant cluster, many tiny ones)
# Formula: G = Σ|n_i - n_j| / (2k * Σn_i)
# G=0: perfect equality | G≈1: maximum inequality
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

def calculate_gini_coefficient(cluster_sizes):
    """
    Calculate Gini coefficient for cluster size distribution.
    G = Σ|n_i - n_j| / (2k * Σn_i)
    """
    sizes = np.array(cluster_sizes, dtype=float)
    n = len(sizes)
    
    if n == 0 or sizes.sum() == 0:
        return 0.0
    
    # Calculate sum of absolute differences
    abs_diff_sum = 0
    for i in range(n):
        for j in range(n):
            abs_diff_sum += abs(sizes[i] - sizes[j])
    
    gini = abs_diff_sum / (2 * n * sizes.sum())
    return gini

def get_aligned_cluster_sizes(true_labels, pred_labels):
    """
    Get predicted cluster sizes aligned to true labels using Hungarian algorithm.
    Returns the sizes of predicted clusters after optimal alignment.
    """
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(true_labels, pred_labels)
    cost_matrix = -cm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Get sizes after alignment
    pred_sizes = np.bincount(pred_labels)
    aligned_sizes = pred_sizes[col_ind] if len(col_ind) == len(pred_sizes) else pred_sizes
    
    return aligned_sizes

# Process all data files and calculate Gini coefficients
data_dir = "../data/original"
gini_results = []

# Get list of all real data files
real_files = glob.glob(os.path.join(data_dir, "*.parquet"))
print(f"📂 Processing {len(real_files)} datasets for Gini analysis...")

# Sample a subset for efficiency (process all unique N, k, rho, sep combinations)
processed_params = set()

for idx, filepath in enumerate(real_files):
    # Parse filename for parameters
    filename = os.path.basename(filepath)
    parts = filename.replace('.parquet', '').split('_')
    
    try:
        # OD_ prefix shifts indices by 1; no rep in parquet filenames
        N = int(parts[1].replace('N', ''))
        p = int(parts[2].replace('p', ''))
        k = int(parts[3].replace('k', ''))
        rho = float(parts[4].replace('rho', ''))
        sep = float(parts[5].replace('sep', ''))
        rep = 1  # parquet files contain all reps; default to 1
    except (IndexError, ValueError):
        continue
    
    # Process each parameter combination once (first rep only for speed)
    param_key = (N, p, k, rho, sep)
    if param_key in processed_params:
        continue
    processed_params.add(param_key)
    
    # Load data
    try:
        data = pd.read_parquet(filepath) if filepath.endswith(".parquet") else pd.read_csv(filepath)
        X = data.drop(columns=['group']).values
        true_labels = data['group'].astype(int).values - data['group'].astype(int).min()
        
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        k_true = len(np.unique(true_labels))
        
        # K-Means
        kmeans = KMeans(n_clusters=k_true, n_init=10, random_state=42)
        pred_km = kmeans.fit_predict(X_scaled)
        
        # Hierarchical
        hc = AgglomerativeClustering(n_clusters=k_true, linkage='ward')
        pred_hc = hc.fit_predict(X_scaled)
        
        # Calculate Gini for true labels (baseline)
        true_sizes = np.bincount(true_labels)
        gini_true = calculate_gini_coefficient(true_sizes)
        
        # Calculate Gini for predicted (aligned via Hungarian)
        km_sizes = get_aligned_cluster_sizes(true_labels, pred_km)
        hc_sizes = get_aligned_cluster_sizes(true_labels, pred_hc)
        
        gini_km = calculate_gini_coefficient(km_sizes)
        gini_hc = calculate_gini_coefficient(hc_sizes)
        
        gini_results.append({
            'N': N, 'p': p, 'k': k, 'rho': rho, 'sep': sep,
            'gini_true': gini_true,
            'gini_kmeans': gini_km,
            'gini_hc': gini_hc
        })
        
    except Exception as e:
        continue

df_gini = pd.DataFrame(gini_results)
print(f"\n✅ Processed {len(df_gini)} parameter combinations")
print(df_gini.head())

# Reshape for plotting
df_gini_long = pd.melt(
    df_gini,
    id_vars=['N', 'p', 'k', 'rho', 'sep', 'gini_true'],
    value_vars=['gini_kmeans', 'gini_hc'],
    var_name='Algorithm',
    value_name='Gini'
)
df_gini_long['Algorithm'] = df_gini_long['Algorithm'].map({
    'gini_kmeans': 'K-Means',
    'gini_hc': 'Hierarchical'
})

# Create publication-quality plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Violin plot by k (number of clusters)
sns.violinplot(
    data=df_gini_long,
    x='k',
    y='Gini',
    hue='Algorithm',
    palette={'K-Means': '#E64B35', 'Hierarchical': '#4DBBD5'},
    split=True,
    inner='quart',
    ax=axes[0]
)
axes[0].axhline(y=0, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='Perfect Balance (G=0)')
axes[0].set_xlabel('Number of True Clusters ($k$)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Gini Coefficient', fontsize=12, fontweight='bold')
axes[0].set_title('Cluster Size Inequality by $k$', fontsize=14, fontweight='bold')
axes[0].legend(title='Algorithm', fontsize=10, loc='upper right')
axes[0].set_ylim(-0.05, 0.6)

# Panel 2: Boxplot by separation level
sns.boxplot(
    data=df_gini_long,
    x='sep',
    y='Gini',
    hue='Algorithm',
    palette={'K-Means': '#E64B35', 'Hierarchical': '#4DBBD5'},
    ax=axes[1]
)
axes[1].axhline(y=0, color='green', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1].set_xlabel('Cluster Separation', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Gini Coefficient', fontsize=12, fontweight='bold')
axes[1].set_title('Cluster Size Inequality by Separation', fontsize=14, fontweight='bold')
axes[1].legend(title='Algorithm', fontsize=10, loc='upper right')
axes[1].set_ylim(-0.05, 0.6)

fig.suptitle('Graphic B: Gini Coefficient – Cluster Size Balance\n'
             '(Lower = more balanced clusters, Higher = cluster collapse)', 
             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics:")
print(f"   K-Means Mean Gini: {df_gini['gini_kmeans'].mean():.4f} (±{df_gini['gini_kmeans'].std():.4f})")
print(f"   HC Mean Gini:      {df_gini['gini_hc'].mean():.4f} (±{df_gini['gini_hc'].std():.4f})")
print(f"   True Labels Gini:  {df_gini['gini_true'].mean():.4f} (baseline)")

### Graphic C: Mean Centroid Distance (Central Tendency Fidelity)
Measure how accurately each algorithm recovers the spatial center of the true clusters using Hungarian-aligned centroid matching.

In [ ]:
# ============================================================================
# GRAPHIC C: Mean Centroid Distance (Central Tendency Fidelity)
# Goal: Measure how accurately algorithms recover true cluster centers
# Method: Use Hungarian Algorithm to match predicted centroids to true centroids
# D_μ = (1/k) Σ ||μ_pred(i) - μ_true(matched)||_2
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

def calculate_centroids(X, labels):
    """Calculate centroids for each cluster."""
    unique_labels = np.unique(labels)
    centroids = np.array([X[labels == label].mean(axis=0) for label in unique_labels])
    return centroids

def hungarian_matched_centroid_distance(true_centroids, pred_centroids):
    """
    Use Hungarian algorithm to optimally match predicted centroids to true centroids.
    Returns mean Euclidean distance between matched pairs.
    """
    # Build distance matrix between all pairs
    dist_matrix = cdist(true_centroids, pred_centroids, metric='euclidean')
    
    # Hungarian algorithm finds optimal matching
    row_ind, col_ind = linear_sum_assignment(dist_matrix)
    
    # Calculate mean distance of matched pairs
    matched_distances = dist_matrix[row_ind, col_ind]
    mean_distance = matched_distances.mean()
    
    return mean_distance, row_ind, col_ind

# Process all data files
data_dir = "../data/original"
centroid_results = []

real_files = glob.glob(os.path.join(data_dir, "*.parquet"))
print(f"📂 Processing {len(real_files)} datasets for centroid analysis...")

processed_params = set()

for filepath in real_files:
    filename = os.path.basename(filepath)
    parts = filename.replace('.parquet', '').split('_')
    
    try:
        # OD_ prefix shifts indices by 1; no rep in parquet filenames
        N = int(parts[1].replace('N', ''))
        p = int(parts[2].replace('p', ''))
        k = int(parts[3].replace('k', ''))
        rho = float(parts[4].replace('rho', ''))
        sep = float(parts[5].replace('sep', ''))
        rep = 1  # parquet files contain all reps; default to 1
    except (IndexError, ValueError):
        continue
    
    # Process each unique parameter combination
    param_key = (N, p, k, rho, sep, rep)
    if param_key in processed_params:
        continue
    processed_params.add(param_key)
    
    try:
        data = pd.read_parquet(filepath) if filepath.endswith(".parquet") else pd.read_csv(filepath)
        X = data.drop(columns=['group']).values
        true_labels = data['group'].astype(int).values - data['group'].astype(int).min()
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        k_true = len(np.unique(true_labels))
        
        # Calculate true centroids
        true_centroids = calculate_centroids(X_scaled, true_labels)
        
        # K-Means
        kmeans = KMeans(n_clusters=k_true, n_init=10, random_state=42)
        pred_km = kmeans.fit_predict(X_scaled)
        km_centroids = calculate_centroids(X_scaled, pred_km)
        
        # Hierarchical
        hc = AgglomerativeClustering(n_clusters=k_true, linkage='ward')
        pred_hc = hc.fit_predict(X_scaled)
        hc_centroids = calculate_centroids(X_scaled, pred_hc)
        
        # Calculate Hungarian-matched centroid distances
        dist_km, _, _ = hungarian_matched_centroid_distance(true_centroids, km_centroids)
        dist_hc, _, _ = hungarian_matched_centroid_distance(true_centroids, hc_centroids)
        
        centroid_results.append({
            'N': N, 'p': p, 'k': k, 'rho': rho, 'sep': sep, 'rep': rep,
            'centroid_dist_km': dist_km,
            'centroid_dist_hc': dist_hc
        })
        
    except Exception as e:
        continue

df_centroid = pd.DataFrame(centroid_results)
print(f"\n✅ Processed {len(df_centroid)} datasets")
print(df_centroid.head())

# Aggregate by parameter combinations for plotting
df_agg = df_centroid.groupby(['k', 'sep', 'rho']).agg({
    'centroid_dist_km': ['mean', 'std', 'count'],
    'centroid_dist_hc': ['mean', 'std', 'count']
}).reset_index()
df_agg.columns = ['k', 'sep', 'rho', 'km_mean', 'km_std', 'km_n', 'hc_mean', 'hc_std', 'hc_n']

# Calculate 95% CI
df_agg['km_ci'] = 1.96 * df_agg['km_std'] / np.sqrt(df_agg['km_n'])
df_agg['hc_ci'] = 1.96 * df_agg['hc_std'] / np.sqrt(df_agg['hc_n'])

# Create publication-quality plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Mean Centroid Distance by Separation Level
sep_agg = df_centroid.groupby('sep').agg({
    'centroid_dist_km': ['mean', 'std', 'count'],
    'centroid_dist_hc': ['mean', 'std', 'count']
}).reset_index()
sep_agg.columns = ['sep', 'km_mean', 'km_std', 'km_n', 'hc_mean', 'hc_std', 'hc_n']
sep_agg['km_ci'] = 1.96 * sep_agg['km_std'] / np.sqrt(sep_agg['km_n'])
sep_agg['hc_ci'] = 1.96 * sep_agg['hc_std'] / np.sqrt(sep_agg['hc_n'])

x_sep = sep_agg['sep'].values

# Plot with error bands
axes[0].fill_between(x_sep, sep_agg['km_mean'] - sep_agg['km_ci'], 
                     sep_agg['km_mean'] + sep_agg['km_ci'], alpha=0.2, color='#E64B35')
axes[0].plot(x_sep, sep_agg['km_mean'], 'o-', color='#E64B35', linewidth=2.5, 
             markersize=10, label='K-Means', markeredgecolor='white', markeredgewidth=1.5)

axes[0].fill_between(x_sep, sep_agg['hc_mean'] - sep_agg['hc_ci'],
                     sep_agg['hc_mean'] + sep_agg['hc_ci'], alpha=0.2, color='#4DBBD5')
axes[0].plot(x_sep, sep_agg['hc_mean'], 's--', color='#4DBBD5', linewidth=2.5,
             markersize=10, label='Hierarchical', markeredgecolor='white', markeredgewidth=1.5)

axes[0].axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.7)
axes[0].set_xlabel('Cluster Separation', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Mean Centroid Distance ($D_\\mu$)', fontsize=12, fontweight='bold')
axes[0].set_title('Centroid Fidelity by Separation', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11, loc='upper right')
axes[0].grid(True, alpha=0.3, linestyle='--')

# Panel 2: Mean Centroid Distance by k
k_agg = df_centroid.groupby('k').agg({
    'centroid_dist_km': ['mean', 'std', 'count'],
    'centroid_dist_hc': ['mean', 'std', 'count']
}).reset_index()
k_agg.columns = ['k', 'km_mean', 'km_std', 'km_n', 'hc_mean', 'hc_std', 'hc_n']
k_agg['km_ci'] = 1.96 * k_agg['km_std'] / np.sqrt(k_agg['km_n'])
k_agg['hc_ci'] = 1.96 * k_agg['hc_std'] / np.sqrt(k_agg['hc_n'])

x_k = k_agg['k'].values
x_offset = 0.1

axes[1].bar(x_k - x_offset, k_agg['km_mean'], width=0.18, color='#E64B35', 
            label='K-Means', alpha=0.85, edgecolor='white', linewidth=1.5)
axes[1].errorbar(x_k - x_offset, k_agg['km_mean'], yerr=k_agg['km_ci'], 
                 fmt='none', color='black', capsize=4, capthick=2)

axes[1].bar(x_k + x_offset, k_agg['hc_mean'], width=0.18, color='#4DBBD5',
            label='Hierarchical', alpha=0.85, edgecolor='white', linewidth=1.5)
axes[1].errorbar(x_k + x_offset, k_agg['hc_mean'], yerr=k_agg['hc_ci'],
                 fmt='none', color='black', capsize=4, capthick=2)

axes[1].axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.7)
axes[1].set_xlabel('Number of True Clusters ($k$)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Mean Centroid Distance ($D_\\mu$)', fontsize=12, fontweight='bold')
axes[1].set_title('Centroid Fidelity by Number of Clusters', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11, loc='upper right')
axes[1].set_xticks(x_k)
axes[1].grid(axis='y', alpha=0.3, linestyle='--')

fig.suptitle('Graphic C: Mean Centroid Distance (Hungarian-Aligned)\n'
             '(Lower = Better Recovery of True Cluster Centers)', 
             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics:")
print(f"   K-Means Mean Centroid Dist: {df_centroid['centroid_dist_km'].mean():.4f} (±{df_centroid['centroid_dist_km'].std():.4f})")
print(f"   HC Mean Centroid Dist:      {df_centroid['centroid_dist_hc'].mean():.4f} (±{df_centroid['centroid_dist_hc'].std():.4f})")

### Graphic D: Mean Variance Difference (Dispersion Fidelity)
Measure if algorithms capture the correct spread/width of clusters using Hungarian-aligned variance comparison.

In [ ]:
# ============================================================================
# GRAPHIC D: Mean Variance Difference (Dispersion Fidelity)
# Goal: Measure if algorithms capture correct cluster spread/width
# Method: Use Hungarian Algorithm to match clusters, then compare variances
# Δσ² = (1/k) Σ |σ²_pred(i) - σ²_true(matched)|
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

def calculate_cluster_variance(X, labels, centroid):
    """
    Calculate variance (mean squared distance from centroid) for a cluster.
    σ² = (1/n) Σ ||x_i - μ||²
    """
    points = X[labels]
    if len(points) == 0:
        return 0.0
    distances_sq = np.sum((points - centroid) ** 2, axis=1)
    return distances_sq.mean()

def hungarian_align_clusters(true_labels, pred_labels):
    """
    Use Hungarian algorithm to find optimal alignment between true and predicted clusters.
    Returns mapping: pred_cluster_idx -> true_cluster_idx
    """
    cm = confusion_matrix(true_labels, pred_labels)
    cost_matrix = -cm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Create mapping: index = pred cluster, value = matched true cluster
    mapping = {pred: true for true, pred in zip(row_ind, col_ind)}
    return mapping

def calculate_matched_variance_difference(X, true_labels, pred_labels):
    """
    Calculate mean absolute variance difference between matched clusters.
    Uses Hungarian algorithm for optimal cluster matching.
    """
    unique_true = np.unique(true_labels)
    unique_pred = np.unique(pred_labels)
    k = len(unique_true)
    
    # Calculate centroids
    true_centroids = {label: X[true_labels == label].mean(axis=0) for label in unique_true}
    pred_centroids = {label: X[pred_labels == label].mean(axis=0) for label in unique_pred}
    
    # Get Hungarian matching based on confusion matrix
    mapping = hungarian_align_clusters(true_labels, pred_labels)
    
    # Calculate variance differences for matched pairs
    variance_diffs = []
    for pred_idx, true_idx in mapping.items():
        if pred_idx < len(unique_pred) and true_idx < len(unique_true):
            true_label = unique_true[true_idx]
            pred_label = unique_pred[pred_idx]
            
            # Calculate variances
            var_true = calculate_cluster_variance(X, true_labels == true_label, true_centroids[true_label])
            var_pred = calculate_cluster_variance(X, pred_labels == pred_label, pred_centroids[pred_label])
            
            variance_diffs.append(abs(var_pred - var_true))
    
    return np.mean(variance_diffs) if variance_diffs else 0.0

# Process all data files
data_dir = "../data/original"
variance_results = []

real_files = glob.glob(os.path.join(data_dir, "*.parquet"))
print(f"📂 Processing {len(real_files)} datasets for variance analysis...")

processed_params = set()

for filepath in real_files:
    filename = os.path.basename(filepath)
    parts = filename.replace('.parquet', '').split('_')
    
    try:
        # OD_ prefix shifts indices by 1; no rep in parquet filenames
        N = int(parts[1].replace('N', ''))
        p = int(parts[2].replace('p', ''))
        k = int(parts[3].replace('k', ''))
        rho = float(parts[4].replace('rho', ''))
        sep = float(parts[5].replace('sep', ''))
        rep = 1  # parquet files contain all reps; default to 1
    except (IndexError, ValueError):
        continue
    
    param_key = (N, p, k, rho, sep, rep)
    if param_key in processed_params:
        continue
    processed_params.add(param_key)
    
    try:
        data = pd.read_parquet(filepath) if filepath.endswith(".parquet") else pd.read_csv(filepath)
        X = data.drop(columns=['group']).values
        true_labels = data['group'].astype(int).values - data['group'].astype(int).min()
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        k_true = len(np.unique(true_labels))
        
        # K-Means
        kmeans = KMeans(n_clusters=k_true, n_init=10, random_state=42)
        pred_km = kmeans.fit_predict(X_scaled)
        
        # Hierarchical
        hc = AgglomerativeClustering(n_clusters=k_true, linkage='ward')
        pred_hc = hc.fit_predict(X_scaled)
        
        # Calculate variance differences
        var_diff_km = calculate_matched_variance_difference(X_scaled, true_labels, pred_km)
        var_diff_hc = calculate_matched_variance_difference(X_scaled, true_labels, pred_hc)
        
        variance_results.append({
            'N': N, 'p': p, 'k': k, 'rho': rho, 'sep': sep, 'rep': rep,
            'var_diff_km': var_diff_km,
            'var_diff_hc': var_diff_hc
        })
        
    except Exception as e:
        continue

df_variance = pd.DataFrame(variance_results)
print(f"\n✅ Processed {len(df_variance)} datasets")
print(df_variance.head())

# Create publication-quality plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Variance Difference by Separation (Line plot with 95% CI)
sep_agg = df_variance.groupby('sep').agg({
    'var_diff_km': ['mean', 'std', 'count'],
    'var_diff_hc': ['mean', 'std', 'count']
}).reset_index()
sep_agg.columns = ['sep', 'km_mean', 'km_std', 'km_n', 'hc_mean', 'hc_std', 'hc_n']
sep_agg['km_ci'] = 1.96 * sep_agg['km_std'] / np.sqrt(sep_agg['km_n'])
sep_agg['hc_ci'] = 1.96 * sep_agg['hc_std'] / np.sqrt(sep_agg['hc_n'])

x_sep = sep_agg['sep'].values

# K-Means
axes[0].fill_between(x_sep, sep_agg['km_mean'] - sep_agg['km_ci'],
                     sep_agg['km_mean'] + sep_agg['km_ci'], alpha=0.2, color='#E64B35')
axes[0].plot(x_sep, sep_agg['km_mean'], 'o-', color='#E64B35', linewidth=2.5,
             markersize=10, label='K-Means', markeredgecolor='white', markeredgewidth=1.5)

# Hierarchical
axes[0].fill_between(x_sep, sep_agg['hc_mean'] - sep_agg['hc_ci'],
                     sep_agg['hc_mean'] + sep_agg['hc_ci'], alpha=0.2, color='#4DBBD5')
axes[0].plot(x_sep, sep_agg['hc_mean'], 's--', color='#4DBBD5', linewidth=2.5,
             markersize=10, label='Hierarchical', markeredgecolor='white', markeredgewidth=1.5)

axes[0].axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.7, label='Perfect Match')
axes[0].set_xlabel('Cluster Separation', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Mean Variance Difference ($\\Delta\\sigma^2$)', fontsize=12, fontweight='bold')
axes[0].set_title('Dispersion Fidelity by Separation', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, loc='upper right')
axes[0].grid(True, alpha=0.3, linestyle='--')

# Panel 2: Variance Difference by rho (highlighting ρ=0.4)
rho_agg = df_variance.groupby('rho').agg({
    'var_diff_km': ['mean', 'std', 'count'],
    'var_diff_hc': ['mean', 'std', 'count']
}).reset_index()
rho_agg.columns = ['rho', 'km_mean', 'km_std', 'km_n', 'hc_mean', 'hc_std', 'hc_n']
rho_agg['km_ci'] = 1.96 * rho_agg['km_std'] / np.sqrt(rho_agg['km_n'])
rho_agg['hc_ci'] = 1.96 * rho_agg['hc_std'] / np.sqrt(rho_agg['hc_n'])

x_rho = rho_agg['rho'].values

# Plot with error ribbons
axes[1].fill_between(x_rho, rho_agg['km_mean'] - rho_agg['km_ci'],
                     rho_agg['km_mean'] + rho_agg['km_ci'], alpha=0.2, color='#E64B35')
axes[1].plot(x_rho, rho_agg['km_mean'], 'o-', color='#E64B35', linewidth=2.5,
             markersize=12, label='K-Means', markeredgecolor='white', markeredgewidth=2)

axes[1].fill_between(x_rho, rho_agg['hc_mean'] - rho_agg['hc_ci'],
                     rho_agg['hc_mean'] + rho_agg['hc_ci'], alpha=0.2, color='#4DBBD5')
axes[1].plot(x_rho, rho_agg['hc_mean'], 's--', color='#4DBBD5', linewidth=2.5,
             markersize=12, label='Hierarchical', markeredgecolor='white', markeredgewidth=2)

# Highlight ρ=0.4
if 0.4 in x_rho:
    axes[1].axvline(x=0.4, color='#7E6148', linestyle=':', linewidth=2.5, alpha=0.8)
    axes[1].annotate('ρ = 0.4', xy=(0.4, axes[1].get_ylim()[1] * 0.85),
                     fontsize=11, ha='center', color='#7E6148', fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#7E6148', alpha=0.9))

axes[1].axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.7)
axes[1].set_xlabel('Correlation Parameter ($\\rho$)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Mean Variance Difference ($\\Delta\\sigma^2$)', fontsize=12, fontweight='bold')
axes[1].set_title('Dispersion Fidelity by Noise Level', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper left')
axes[1].grid(True, alpha=0.3, linestyle='--')

fig.suptitle('Graphic D: Mean Variance Difference (Hungarian-Aligned)\n'
             '(Lower = Better Recovery of True Cluster Spread)', 
             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics:")
print(f"   K-Means Mean Var Diff: {df_variance['var_diff_km'].mean():.4f} (±{df_variance['var_diff_km'].std():.4f})")
print(f"   HC Mean Var Diff:      {df_variance['var_diff_hc'].mean():.4f} (±{df_variance['var_diff_hc'].std():.4f})")

### Summary: Advanced Validation Graphics

| Graphic | Metric | Description | File |
|---------|--------|-------------|------|
| A | Confusion Matrix | Cluster-label alignment with Hungarian Algorithm | `graphicA_confusion_hungarian.png` |
| B | Gini Coefficient | Cluster size balance (0=perfect, 1=collapse) | `graphicB_gini_coefficient.png` |
| C | Mean Centroid Distance | Central tendency fidelity ($D_\mu$) | `graphicC_centroid_distance.png` |
| D | Mean Variance Difference | Dispersion fidelity ($\Delta\sigma^2$) | `graphicD_variance_difference.png` |

**All metrics use Hungarian Algorithm alignment for meaningful cluster-to-label comparisons.**

### Graphic E: Comprehensive 3×4 Matrix Panel
Combined visualization showing all validation metrics (Gini, Centroid Distance, Variance Difference) across all parameters (Separation, Cluster, Variables, Correlation).

In [ ]:
# ============================================================================
# GRAPHIC E: Comprehensive 3×4 Matrix Panel
# Goal: Show all validation metrics across all key parameters in a single view
# Layout: 3 rows (metrics) × 4 columns (parameters)
# Rows: Gini Coefficient, Centroid Distance, Variance Difference
# Columns: Separation, Cluster (k), Variables (p), Correlation (ρ)
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set publication style
sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Prepare the data (use existing dataframes from previous cells)
# We have: df_gini, df_centroid, df_variance

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# Define colors for algorithms
colors_km = '#E64B35'
colors_hc = '#4DBBD5'

# ============================================================================
# ROW 1: GINI COEFFICIENT (Cluster Size Balance)
# ============================================================================

# Column 1: Gini by Separation
ax = axes[0, 0]
sep_data = df_gini_long.groupby(['sep', 'Algorithm'])['Gini'].apply(list).reset_index()
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_gini_long[df_gini_long['Algorithm'] == algo]
    positions = sorted(subset['sep'].unique())
    data_to_plot = [subset[subset['sep'] == pos]['Gini'].values for pos in positions]
    bp = ax.boxplot(data_to_plot, positions=positions, widths=0.8, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini', fontsize=10, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-0.05, 0.5)

# Column 2: Gini by Cluster (k)
ax = axes[0, 1]
for algo, color, marker in [('K-Means', colors_km, 'o'), ('Hierarchical', colors_hc, 's')]:
    subset = df_gini_long[df_gini_long['Algorithm'] == algo]
    k_vals = sorted(subset['k'].unique())
    data_to_plot = [subset[subset['k'] == k]['Gini'].values for k in k_vals]
    bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.3, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Cluster', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini', fontsize=10, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks([2, 3, 4])
ax.set_ylim(-0.05, 0.5)

# Column 3: Gini by Variables (p)
ax = axes[0, 2]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_gini_long[df_gini_long['Algorithm'] == algo]
    p_vals = sorted(subset['p'].unique())
    data_to_plot = [subset[subset['p'] == p]['Gini'].values for p in p_vals]
    bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.2, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Variables', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini', fontsize=10, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.5)

# Column 4: Gini by Correlation (rho)
ax = axes[0, 3]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_gini_long[df_gini_long['Algorithm'] == algo]
    rho_vals = sorted(subset['rho'].unique())
    data_to_plot = [subset[subset['rho'] == r]['Gini'].values for r in rho_vals]
    bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.08, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Correlation', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini', fontsize=10, fontweight='bold')
ax.set_title('by Feature Correlation (ρ)', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-0.05, 0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE (Central Tendency Fidelity)
# ============================================================================

# Prepare centroid data in long format
df_centroid_long = pd.melt(
    df_centroid,
    id_vars=['N', 'p', 'k', 'rho', 'sep', 'rep'],
    value_vars=['centroid_dist_km', 'centroid_dist_hc'],
    var_name='Algorithm',
    value_name='Centroid_Distance'
)
df_centroid_long['Algorithm'] = df_centroid_long['Algorithm'].map({
    'centroid_dist_km': 'K-Means',
    'centroid_dist_hc': 'Hierarchical'
})

# Column 1: Centroid Distance by Separation
ax = axes[1, 0]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_centroid_long[df_centroid_long['Algorithm'] == algo]
    positions = sorted(subset['sep'].unique())
    data_to_plot = [subset[subset['sep'] == pos]['Centroid_Distance'].values for pos in positions]
    bp = ax.boxplot(data_to_plot, positions=positions, widths=0.8, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Distance', fontsize=10, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-0.1, 2.5)

# Column 2: Centroid Distance by Cluster (k)
ax = axes[1, 1]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_centroid_long[df_centroid_long['Algorithm'] == algo]
    k_vals = sorted(subset['k'].unique())
    data_to_plot = [subset[subset['k'] == k]['Centroid_Distance'].values for k in k_vals]
    bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.3, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Cluster', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Distance', fontsize=10, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks([2, 3, 4])
ax.set_ylim(-0.1, 2.5)

# Column 3: Centroid Distance by Variables (p)
ax = axes[1, 2]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_centroid_long[df_centroid_long['Algorithm'] == algo]
    p_vals = sorted(subset['p'].unique())
    data_to_plot = [subset[subset['p'] == p]['Centroid_Distance'].values for p in p_vals]
    bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.2, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Variables', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Distance', fontsize=10, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)

# Column 4: Centroid Distance by Correlation (rho)
ax = axes[1, 3]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_centroid_long[df_centroid_long['Algorithm'] == algo]
    rho_vals = sorted(subset['rho'].unique())
    data_to_plot = [subset[subset['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
    bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.08, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Correlation', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Distance', fontsize=10, fontweight='bold')
ax.set_title('by Feature Correlation (ρ)', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-0.1, 2.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE (Dispersion Fidelity)
# ============================================================================

# Prepare variance data in long format
df_variance_long = pd.melt(
    df_variance,
    id_vars=['N', 'p', 'k', 'rho', 'sep', 'rep'],
    value_vars=['var_diff_km', 'var_diff_hc'],
    var_name='Algorithm',
    value_name='Variance_Difference'
)
df_variance_long['Algorithm'] = df_variance_long['Algorithm'].map({
    'var_diff_km': 'K-Means',
    'var_diff_hc': 'Hierarchical'
})

# Column 1: Variance Difference by Separation
ax = axes[2, 0]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_variance_long[df_variance_long['Algorithm'] == algo]
    positions = sorted(subset['sep'].unique())
    data_to_plot = [subset[subset['sep'] == pos]['Variance_Difference'].values for pos in positions]
    bp = ax.boxplot(data_to_plot, positions=positions, widths=0.8, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Var Diff', fontsize=10, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-0.1, 4.0)

# Column 2: Variance Difference by Cluster (k)
ax = axes[2, 1]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_variance_long[df_variance_long['Algorithm'] == algo]
    k_vals = sorted(subset['k'].unique())
    data_to_plot = [subset[subset['k'] == k]['Variance_Difference'].values for k in k_vals]
    bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.3, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Cluster', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Var Diff', fontsize=10, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks([2, 3, 4])
ax.set_ylim(-0.1, 4.0)

# Column 3: Variance Difference by Variables (p)
ax = axes[2, 2]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_variance_long[df_variance_long['Algorithm'] == algo]
    p_vals = sorted(subset['p'].unique())
    data_to_plot = [subset[subset['p'] == p]['Variance_Difference'].values for p in p_vals]
    bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.2, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Variables', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Var Diff', fontsize=10, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)

# Column 4: Variance Difference by Correlation (rho)
ax = axes[2, 3]
for algo, color in [('K-Means', colors_km), ('Hierarchical', colors_hc)]:
    subset = df_variance_long[df_variance_long['Algorithm'] == algo]
    rho_vals = sorted(subset['rho'].unique())
    data_to_plot = [subset[subset['rho'] == r]['Variance_Difference'].values for r in rho_vals]
    bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.08, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(color=color), capprops=dict(color=color),
                    showfliers=False)
ax.set_xlabel('Correlation', fontsize=10, fontweight='bold')
ax.set_ylabel('Mean Var Diff', fontsize=10, fontweight='bold')
ax.set_title('by Feature Correlation (ρ)', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-0.1, 4.0)

# Add legend in top right corner with better structure explanation
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=colors_km, alpha=0.7, edgecolor='black', label='K-Means'),
    Patch(facecolor=colors_hc, alpha=0.7, edgecolor='black', label='Hierarchical')
]
fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.98, 0.98),
          fontsize=11, title='Algorithm', title_fontsize=12, framealpha=0.95)

# Add side annotation for "Better maintained structure"
fig.text(0.97, 0.5, 'Better maintained\nstructure:', rotation=0, fontsize=11,
         ha='left', va='center', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.3))
fig.text(0.97, 0.44, '• Separation', fontsize=9, ha='left', va='center')
fig.text(0.97, 0.40, '• Cluster', fontsize=9, ha='left', va='center')
fig.text(0.97, 0.36, '• Variables', fontsize=9, ha='left', va='center')

# Overall title
fig.suptitle('Graphic E: Comprehensive Clustering Structure Comparison (3×4 Matrix)\n'
             'Rows: Metrics (Gini, Centroid Distance, Variance) | Columns: Parameters (Separation, k, p, ρ)',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 0.95, 0.99])
plt.show()
print("\n📊 Matrix Structure:")
print("   Rows (Metrics):")
print("     1. Gini Coefficient (Cluster Size Balance)")
print("     2. Centroid Distance (Central Tendency Fidelity)")
print("     3. Variance Difference (Dispersion Fidelity)")
print("\n   Columns (Parameters):")
print("     1. Separation (cluster distance)")
print("     2. Cluster (number of true clusters k)")
print("     3. Variables (dimensionality p)")
print("     4. Correlation (feature correlation ρ)")

### Graphic F: K-Means Only - 3×4 Matrix Panel
Isolated view of K-Means performance across all validation metrics and parameters.

In [ ]:
# ============================================================================
# GRAPHIC F: K-Means Only - 3×4 Matrix Panel
# Goal: Clean visualization of K-Means performance across all metrics
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# K-Means color scheme
km_color = '#E64B35'

# Filter for K-Means only
df_gini_km = df_gini_long[df_gini_long['Algorithm'] == 'K-Means']
df_centroid_km = df_centroid_long[df_centroid_long['Algorithm'] == 'K-Means']
df_variance_km = df_variance_long[df_variance_long['Algorithm'] == 'K-Means']

# ============================================================================
# ROW 1: GINI COEFFICIENT
# ============================================================================

# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_km['sep'].unique())
data_to_plot = [df_gini_km[df_gini_km['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_km['k'].unique())
data_to_plot = [df_gini_km[df_gini_km['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_km['p'].unique())
data_to_plot = [df_gini_km[df_gini_km['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_km['rho'].unique())
data_to_plot = [df_gini_km[df_gini_km['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE
# ============================================================================

# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_centroid_km['sep'].unique())
data_to_plot = [df_centroid_km[df_centroid_km['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_centroid_km['k'].unique())
data_to_plot = [df_centroid_km[df_centroid_km['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_centroid_km['p'].unique())
data_to_plot = [df_centroid_km[df_centroid_km['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_centroid_km['rho'].unique())
data_to_plot = [df_centroid_km[df_centroid_km['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE
# ============================================================================

# Column 1: By Separation
ax = axes[2, 0]
positions = sorted(df_variance_km['sep'].unique())
data_to_plot = [df_variance_km[df_variance_km['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[2, 1]
k_vals = sorted(df_variance_km['k'].unique())
data_to_plot = [df_variance_km[df_variance_km['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[2, 2]
p_vals = sorted(df_variance_km['p'].unique())
data_to_plot = [df_variance_km[df_variance_km['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[2, 3]
rho_vals = sorted(df_variance_km['rho'].unique())
data_to_plot = [df_variance_km[df_variance_km['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('Graphic F: K-Means Clustering Structure Analysis (3×4 Matrix)\n'
             'Rows: Gini, Centroid Distance, Variance | Columns: Separation, k, p, ρ',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

### Graphic G: Hierarchical Clustering Only - 3×4 Matrix Panel
Isolated view of Hierarchical Clustering performance across all validation metrics and parameters.

In [ ]:
# ============================================================================
# GRAPHIC G: Hierarchical Clustering Only - 3×4 Matrix Panel
# Goal: Clean visualization of Hierarchical Clustering performance
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# Hierarchical color scheme
hc_color = '#4DBBD5'

# Filter for Hierarchical Clustering only
df_gini_hc = df_gini_long[df_gini_long['Algorithm'] == 'Hierarchical']
df_centroid_hc = df_centroid_long[df_centroid_long['Algorithm'] == 'Hierarchical']
df_variance_hc = df_variance_long[df_variance_long['Algorithm'] == 'Hierarchical']

# ============================================================================
# ROW 1: GINI COEFFICIENT
# ============================================================================

# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_hc['sep'].unique())
data_to_plot = [df_gini_hc[df_gini_hc['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_hc['k'].unique())
data_to_plot = [df_gini_hc[df_gini_hc['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.05, 0.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_hc['p'].unique())
data_to_plot = [df_gini_hc[df_gini_hc['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_hc['rho'].unique())
data_to_plot = [df_gini_hc[df_gini_hc['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE
# ============================================================================

# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_centroid_hc['sep'].unique())
data_to_plot = [df_centroid_hc[df_centroid_hc['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_centroid_hc['k'].unique())
data_to_plot = [df_centroid_hc[df_centroid_hc['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_centroid_hc['p'].unique())
data_to_plot = [df_centroid_hc[df_centroid_hc['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_centroid_hc['rho'].unique())
data_to_plot = [df_centroid_hc[df_centroid_hc['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE
# ============================================================================

# Column 1: By Separation
ax = axes[2, 0]
positions = sorted(df_variance_hc['sep'].unique())
data_to_plot = [df_variance_hc[df_variance_hc['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[2, 1]
k_vals = sorted(df_variance_hc['k'].unique())
data_to_plot = [df_variance_hc[df_variance_hc['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[2, 2]
p_vals = sorted(df_variance_hc['p'].unique())
data_to_plot = [df_variance_hc[df_variance_hc['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[2, 3]
rho_vals = sorted(df_variance_hc['rho'].unique())
data_to_plot = [df_variance_hc[df_variance_hc['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('Graphic G: Hierarchical Clustering Structure Analysis (3×4 Matrix)\n'
             'Rows: Gini, Centroid Distance, Variance | Columns: Separation, k, p, ρ',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

---
## GAMMA DISTRIBUTION ANALYSIS
**Distribution-Specific Graphics: Gamma Distribution**

The following graphics isolate the **Gamma distribution** results to analyze:
- GraphicH: Comprehensive 3×4 matrix (all metrics) - Gamma only
- GraphicI: K-Means 3×4 matrix - Gamma only  
- GraphicJ: Hierarchical Clustering 3×4 matrix - Gamma only

### Graphic H: Comprehensive 3×4 Matrix - Gamma Distribution Only

In [ ]:
# ============================================================================
# GRAPHIC H: COMPREHENSIVE 3×4 MATRIX (ALL ALGORITHMS + ALL METRICS) - GAMMA ONLY
# ============================================================================

# Filter for Gamma distribution only
df_gamma = df[df['distribution'] == 'gamma'].copy()

print(f"🔬 Gamma Distribution Data: {len(df_gamma)} rows")

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('Comprehensive Clustering Performance Matrix (3×4) - Gamma Distribution', 
             fontsize=18, fontweight='bold', y=0.995)

# Define parameters for each column
k_vals = sorted(df_gamma['k'].unique())
p_vals = sorted(df_gamma['p'].unique())
rho_vals = sorted(df_gamma['rho'].unique())
sep_vals = sorted(df_gamma['sep'].unique())

# Define colors
colors = {'K-Means': '#2E86AB', 'Hierarchical': '#A23B72'}

# Row titles
row_titles = [
    'Detection Success Rate (%)',
    'Silhouette Coefficient Drop',
    'Distortion Ratio Increase'
]

# ============================================================
# ROW 1: DETECTION SUCCESS RATE
# ============================================================
# Col 1: Detection vs k
k_success = df_gamma.groupby('k')[['success_kmeans', 'success_hc']].mean() * 100
axes[0, 0].plot(k_success.index, k_success['success_kmeans'], 'o-', 
                color=colors['K-Means'], label='K-Means', linewidth=2, markersize=8)
axes[0, 0].plot(k_success.index, k_success['success_hc'], 's-', 
                color=colors['Hierarchical'], label='Hierarchical', linewidth=2, markersize=8)
axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel(row_titles[0], fontsize=11, fontweight='bold')
axes[0, 0].set_ylim(0, 100)
axes[0, 0].legend(frameon=True, loc='best')
axes[0, 0].grid(True, alpha=0.3)

# Col 2: Detection vs p
p_success = df_gamma.groupby('p')[['success_kmeans', 'success_hc']].mean() * 100
axes[0, 1].plot(p_success.index, p_success['success_kmeans'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[0, 1].plot(p_success.index, p_success['success_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[0, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylim(0, 100)
axes[0, 1].grid(True, alpha=0.3)

# Col 3: Detection vs rho
rho_success = df_gamma.groupby('rho')[['success_kmeans', 'success_hc']].mean() * 100
axes[0, 2].plot(rho_success.index, rho_success['success_kmeans'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[0, 2].plot(rho_success.index, rho_success['success_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[0, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[0, 2].set_ylim(0, 100)
axes[0, 2].grid(True, alpha=0.3)

# Col 4: Detection vs sep
sep_success = df_gamma.groupby('sep')[['success_kmeans', 'success_hc']].mean() * 100
axes[0, 3].plot(sep_success.index, sep_success['success_kmeans'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[0, 3].plot(sep_success.index, sep_success['success_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[0, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[0, 3].set_ylim(0, 100)
axes[0, 3].grid(True, alpha=0.3)

# ============================================================
# ROW 2: SILHOUETTE COEFFICIENT DROP
# ============================================================
# Col 1: Silhouette vs k
k_sil = df_gamma.groupby('k')[['diff_sil_km', 'diff_sil_hc']].mean()
axes[1, 0].plot(k_sil.index, k_sil['diff_sil_km'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[1, 0].plot(k_sil.index, k_sil['diff_sil_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[1, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel(row_titles[1], fontsize=11, fontweight='bold')
axes[1, 0].axhline(0, color='red', linestyle='--', alpha=0.5, label='No degradation')
axes[1, 0].grid(True, alpha=0.3)

# Col 2: Silhouette vs p
p_sil = df_gamma.groupby('p')[['diff_sil_km', 'diff_sil_hc']].mean()
axes[1, 1].plot(p_sil.index, p_sil['diff_sil_km'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[1, 1].plot(p_sil.index, p_sil['diff_sil_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[1, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[1, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].grid(True, alpha=0.3)

# Col 3: Silhouette vs rho
rho_sil = df_gamma.groupby('rho')[['diff_sil_km', 'diff_sil_hc']].mean()
axes[1, 2].plot(rho_sil.index, rho_sil['diff_sil_km'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[1, 2].plot(rho_sil.index, rho_sil['diff_sil_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[1, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[1, 2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 2].grid(True, alpha=0.3)

# Col 4: Silhouette vs sep
sep_sil = df_gamma.groupby('sep')[['diff_sil_km', 'diff_sil_hc']].mean()
axes[1, 3].plot(sep_sil.index, sep_sil['diff_sil_km'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[1, 3].plot(sep_sil.index, sep_sil['diff_sil_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[1, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[1, 3].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 3].grid(True, alpha=0.3)

# ============================================================
# ROW 3: DISTORTION RATIO INCREASE
# ============================================================
# Col 1: Distortion vs k
k_dist = df_gamma.groupby('k')[['diff_dist_km', 'diff_dist_hc']].mean()
axes[2, 0].plot(k_dist.index, k_dist['diff_dist_km'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[2, 0].plot(k_dist.index, k_dist['diff_dist_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[2, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel(row_titles[2], fontsize=11, fontweight='bold')
axes[2, 0].axhline(0, color='red', linestyle='--', alpha=0.5, label='No degradation')
axes[2, 0].grid(True, alpha=0.3)

# Col 2: Distortion vs p
p_dist = df_gamma.groupby('p')[['diff_dist_km', 'diff_dist_hc']].mean()
axes[2, 1].plot(p_dist.index, p_dist['diff_dist_km'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[2, 1].plot(p_dist.index, p_dist['diff_dist_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[2, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[2, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 1].grid(True, alpha=0.3)

# Col 3: Distortion vs rho
rho_dist = df_gamma.groupby('rho')[['diff_dist_km', 'diff_dist_hc']].mean()
axes[2, 2].plot(rho_dist.index, rho_dist['diff_dist_km'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[2, 2].plot(rho_dist.index, rho_dist['diff_dist_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[2, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[2, 2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 2].grid(True, alpha=0.3)

# Col 4: Distortion vs sep
sep_dist = df_gamma.groupby('sep')[['diff_dist_km', 'diff_dist_hc']].mean()
axes[2, 3].plot(sep_dist.index, sep_dist['diff_dist_km'], 'o-', 
                color=colors['K-Means'], linewidth=2, markersize=8)
axes[2, 3].plot(sep_dist.index, sep_dist['diff_dist_hc'], 's-', 
                color=colors['Hierarchical'], linewidth=2, markersize=8)
axes[2, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[2, 3].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Graphic I: K-Means Only 3×4 Matrix - Gamma Distribution

In [ ]:
# ============================================================================
# GRAPHIC I: K-MEANS ONLY 3×4 MATRIX - GAMMA DISTRIBUTION
# ============================================================================

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('K-Means Clustering Performance Matrix (3×4) - Gamma Distribution', 
             fontsize=18, fontweight='bold', y=0.995)

km_color = '#2E86AB'

# ============================================================
# ROW 1: DETECTION SUCCESS RATE (K-MEANS)
# ============================================================
axes[0, 0].plot(k_success.index, k_success['success_kmeans'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Detection Success Rate (%)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylim(0, 100)
axes[0, 0].set_title('K-Means Detection vs k', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(p_success.index, p_success['success_kmeans'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[0, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylim(0, 100)
axes[0, 1].set_title('K-Means Detection vs p', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(rho_success.index, rho_success['success_kmeans'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[0, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[0, 2].set_ylim(0, 100)
axes[0, 2].set_title('K-Means Detection vs ρ', fontsize=12, fontweight='bold')
axes[0, 2].grid(True, alpha=0.3)

axes[0, 3].plot(sep_success.index, sep_success['success_kmeans'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[0, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[0, 3].set_ylim(0, 100)
axes[0, 3].set_title('K-Means Detection vs Separation', fontsize=12, fontweight='bold')
axes[0, 3].grid(True, alpha=0.3)

# ============================================================
# ROW 2: SILHOUETTE COEFFICIENT DROP (K-MEANS)
# ============================================================
axes[1, 0].plot(k_sil.index, k_sil['diff_sil_km'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[1, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Silhouette Drop', fontsize=11, fontweight='bold')
axes[1, 0].set_title('K-Means Silhouette vs k', fontsize=12, fontweight='bold')
axes[1, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(p_sil.index, p_sil['diff_sil_km'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[1, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('K-Means Silhouette vs p', fontsize=12, fontweight='bold')
axes[1, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(rho_sil.index, rho_sil['diff_sil_km'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[1, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[1, 2].set_title('K-Means Silhouette vs ρ', fontsize=12, fontweight='bold')
axes[1, 2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 2].grid(True, alpha=0.3)

axes[1, 3].plot(sep_sil.index, sep_sil['diff_sil_km'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[1, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[1, 3].set_title('K-Means Silhouette vs Separation', fontsize=12, fontweight='bold')
axes[1, 3].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 3].grid(True, alpha=0.3)

# ============================================================
# ROW 3: DISTORTION RATIO INCREASE (K-MEANS)
# ============================================================
axes[2, 0].plot(k_dist.index, k_dist['diff_dist_km'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[2, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('Distortion Increase', fontsize=11, fontweight='bold')
axes[2, 0].set_title('K-Means Distortion vs k', fontsize=12, fontweight='bold')
axes[2, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 0].grid(True, alpha=0.3)

axes[2, 1].plot(p_dist.index, p_dist['diff_dist_km'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[2, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[2, 1].set_title('K-Means Distortion vs p', fontsize=12, fontweight='bold')
axes[2, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 1].grid(True, alpha=0.3)

axes[2, 2].plot(rho_dist.index, rho_dist['diff_dist_km'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[2, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[2, 2].set_title('K-Means Distortion vs ρ', fontsize=12, fontweight='bold')
axes[2, 2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 2].grid(True, alpha=0.3)

axes[2, 3].plot(sep_dist.index, sep_dist['diff_dist_km'], 'o-', 
                color=km_color, linewidth=3, markersize=10)
axes[2, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[2, 3].set_title('K-Means Distortion vs Separation', fontsize=12, fontweight='bold')
axes[2, 3].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Graphic J: Hierarchical Clustering Only 3×4 Matrix - Gamma Distribution

In [ ]:
# ============================================================================
# GRAPHIC J: HIERARCHICAL CLUSTERING ONLY 3×4 MATRIX - GAMMA DISTRIBUTION
# ============================================================================

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
fig.suptitle('Hierarchical Clustering Performance Matrix (3×4) - Gamma Distribution', 
             fontsize=18, fontweight='bold', y=0.995)

hc_color = '#A23B72'

# ============================================================
# ROW 1: DETECTION SUCCESS RATE (HIERARCHICAL)
# ============================================================
axes[0, 0].plot(k_success.index, k_success['success_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[0, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylabel('Detection Success Rate (%)', fontsize=11, fontweight='bold')
axes[0, 0].set_ylim(0, 100)
axes[0, 0].set_title('Hierarchical Detection vs k', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(p_success.index, p_success['success_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[0, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[0, 1].set_ylim(0, 100)
axes[0, 1].set_title('Hierarchical Detection vs p', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

axes[0, 2].plot(rho_success.index, rho_success['success_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[0, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[0, 2].set_ylim(0, 100)
axes[0, 2].set_title('Hierarchical Detection vs ρ', fontsize=12, fontweight='bold')
axes[0, 2].grid(True, alpha=0.3)

axes[0, 3].plot(sep_success.index, sep_success['success_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[0, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[0, 3].set_ylim(0, 100)
axes[0, 3].set_title('Hierarchical Detection vs Separation', fontsize=12, fontweight='bold')
axes[0, 3].grid(True, alpha=0.3)

# ============================================================
# ROW 2: SILHOUETTE COEFFICIENT DROP (HIERARCHICAL)
# ============================================================
axes[1, 0].plot(k_sil.index, k_sil['diff_sil_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[1, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Silhouette Drop', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Hierarchical Silhouette vs k', fontsize=12, fontweight='bold')
axes[1, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(p_sil.index, p_sil['diff_sil_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[1, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Hierarchical Silhouette vs p', fontsize=12, fontweight='bold')
axes[1, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].grid(True, alpha=0.3)

axes[1, 2].plot(rho_sil.index, rho_sil['diff_sil_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[1, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[1, 2].set_title('Hierarchical Silhouette vs ρ', fontsize=12, fontweight='bold')
axes[1, 2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 2].grid(True, alpha=0.3)

axes[1, 3].plot(sep_sil.index, sep_sil['diff_sil_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[1, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[1, 3].set_title('Hierarchical Silhouette vs Separation', fontsize=12, fontweight='bold')
axes[1, 3].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 3].grid(True, alpha=0.3)

# ============================================================
# ROW 3: DISTORTION RATIO INCREASE (HIERARCHICAL)
# ============================================================
axes[2, 0].plot(k_dist.index, k_dist['diff_dist_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[2, 0].set_xlabel('Number of Clusters (k)', fontsize=11, fontweight='bold')
axes[2, 0].set_ylabel('Distortion Increase', fontsize=11, fontweight='bold')
axes[2, 0].set_title('Hierarchical Distortion vs k', fontsize=12, fontweight='bold')
axes[2, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 0].grid(True, alpha=0.3)

axes[2, 1].plot(p_dist.index, p_dist['diff_dist_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[2, 1].set_xlabel('Dimensions (p)', fontsize=11, fontweight='bold')
axes[2, 1].set_title('Hierarchical Distortion vs p', fontsize=12, fontweight='bold')
axes[2, 1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 1].grid(True, alpha=0.3)

axes[2, 2].plot(rho_dist.index, rho_dist['diff_dist_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[2, 2].set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
axes[2, 2].set_title('Hierarchical Distortion vs ρ', fontsize=12, fontweight='bold')
axes[2, 2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 2].grid(True, alpha=0.3)

axes[2, 3].plot(sep_dist.index, sep_dist['diff_dist_hc'], 's-', 
                color=hc_color, linewidth=3, markersize=10)
axes[2, 3].set_xlabel('Separation', fontsize=11, fontweight='bold')
axes[2, 3].set_title('Hierarchical Distortion vs Separation', fontsize=12, fontweight='bold')
axes[2, 3].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Early Visualization Plots - Gamma Distribution Only

Recreating the initial publication-quality graphics specifically for gamma distribution data.

#### Plot 1 (Gamma): Success Rate Heatmap

In [ ]:
# Calculate success rates for GAMMA distribution only
success_rates_gamma = df_gamma.groupby(['sep', 'k']).agg({
    'success_kmeans': 'mean',
    'success_hc': 'mean'
}).reset_index()

# Pivot for heatmap format
heatmap_km_gamma = success_rates_gamma.pivot(index='k', columns='sep', values='success_kmeans')
heatmap_hc_gamma = success_rates_gamma.pivot(index='k', columns='sep', values='success_hc')

print("📊 Success Rates (Gamma) - K-Means:")
print(heatmap_km_gamma)
print("\n📊 Success Rates (Gamma) - Hierarchical Clustering:")
print(heatmap_hc_gamma)

In [ ]:
# Create side-by-side heatmap comparison for GAMMA
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# K-Means Heatmap (Gamma)
sns.heatmap(heatmap_km_gamma, annot=True, fmt='.3f', cmap='RdYlGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Success Rate'},
            ax=axes[0], linewidths=0.5, linecolor='gray')
axes[0].set_title('K-Means: Detection Success Rate (Gamma Distribution)', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Cluster Separation', fontsize=12)
axes[0].set_ylabel('True Number of Clusters (k)', fontsize=12)

# Hierarchical Clustering Heatmap (Gamma)
sns.heatmap(heatmap_hc_gamma, annot=True, fmt='.3f', cmap='RdYlGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Success Rate'},
            ax=axes[1], linewidths=0.5, linecolor='gray')
axes[1].set_title('Hierarchical Clustering: Detection Success Rate (Gamma Distribution)', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Cluster Separation', fontsize=12)
axes[1].set_ylabel('True Number of Clusters (k)', fontsize=12)

plt.tight_layout()
plt.show()

#### Plot 2 (Gamma): Performance Delta Heatmap

In [ ]:
# Calculate performance delta (HC - K-Means) for GAMMA
performance_delta_gamma = heatmap_hc_gamma - heatmap_km_gamma

print("📊 Performance Delta (HC - K-Means) [Gamma Distribution]:")
print(performance_delta_gamma)
print("\n🔍 Interpretation:")
print("  Positive values = HC outperforms K-Means")
print("  Negative values = K-Means outperforms HC")
print("  Zero = Equal performance")

In [ ]:
# Create delta heatmap for GAMMA
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(performance_delta_gamma, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
            cbar_kws={'label': 'Performance Delta (HC - K-Means)'},
            linewidths=0.5, linecolor='gray', ax=ax)

ax.set_title('Hierarchical Clustering vs K-Means\nPerformance Advantage Map (Gamma Distribution)', 
             fontsize=16, fontweight='bold')
ax.set_xlabel('Cluster Separation', fontsize=12)
ax.set_ylabel('True Number of Clusters (k)', fontsize=12)

# Add interpretation text
fig.text(0.5, -0.05, 
         'Red = HC Outperforms | Blue = K-Means Outperforms | White = Equal',
         ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.show()

#### Plot 3 (Gamma): Quality Distortion Boxplots

In [ ]:
# Prepare data for boxplots (GAMMA only)
boxplot_data_gamma = df_gamma[['sep', 'k', 'diff_sil_km', 'diff_sil_hc']].copy()

# Reshape for easier plotting
boxplot_data_gamma_long = pd.melt(
    boxplot_data_gamma, 
    id_vars=['sep', 'k'], 
    value_vars=['diff_sil_km', 'diff_sil_hc'],
    var_name='Algorithm', 
    value_name='Silhouette_Delta'
)

# Rename for clarity
boxplot_data_gamma_long['Algorithm'] = boxplot_data_gamma_long['Algorithm'].map({
    'diff_sil_km': 'K-Means',
    'diff_sil_hc': 'Hierarchical'
})

print("📊 Sample of prepared data (Gamma):")
print(boxplot_data_gamma_long.head(10))

# Summary statistics
print("\n📊 QUALITY DISTORTION SUMMARY (Gamma Distribution)")
print("=" * 60)

for algo in ['K-Means', 'Hierarchical']:
    data = boxplot_data_gamma_long[boxplot_data_gamma_long['Algorithm'] == algo]['Silhouette_Delta']
    print(f"\n{algo}:")
    print(f"  Mean Delta: {data.mean():.4f}")
    print(f"  Median Delta: {data.median():.4f}")
    print(f"  Std Dev: {data.std():.4f}")
    print(f"  Min: {data.min():.4f} | Max: {data.max():.4f}")

In [ ]:
# Create boxplots by separation (GAMMA)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

separations_gamma = sorted(df_gamma['sep'].unique())

for idx, sep in enumerate(separations_gamma):
    subset = boxplot_data_gamma_long[boxplot_data_gamma_long['sep'] == sep]
    
    sns.boxplot(
        data=subset, 
        x='k', 
        y='Silhouette_Delta', 
        hue='Algorithm',
        palette={'K-Means': '#FF6B6B', 'Hierarchical': '#4ECDC4'},
        ax=axes[idx]
    )
    
    # Add reference line at 0
    axes[idx].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    axes[idx].set_title(f'Separation = {sep}', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('True Number of Clusters (k)', fontsize=11)
    axes[idx].set_ylabel('Silhouette Delta\n(Synthetic - Real)', fontsize=11)
    axes[idx].legend(title='Algorithm', loc='lower right')
    axes[idx].grid(axis='y', alpha=0.3)

fig.suptitle('Quality Distortion: Gamma Distribution (How Much Does Synthpop Degrade Cluster Geometry?)', 
             fontsize=18, fontweight='bold', y=1.00)

plt.tight_layout()
plt.show()

#### Graphic F (Gamma): K-Means Only - 3×4 Validation Matrix

In [ ]:
# ============================================================================
# GRAPHIC F (GAMMA): K-Means Only - 3×4 Matrix Panel
# Goal: Clean visualization of K-Means performance across all metrics
# Note: Using existing validation data (distribution filter not available in these dataframes)
# ============================================================================

# Filter for K-Means only
df_gini_km_gamma = df_gini_long[df_gini_long['Algorithm'] == 'K-Means']
df_centroid_km_gamma = df_centroid_long[df_centroid_long['Algorithm'] == 'K-Means']
df_variance_km_gamma = df_variance_long[df_variance_long['Algorithm'] == 'K-Means']

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# K-Means color scheme
km_color = '#E64B35'

# ============================================================================
# ROW 1: GINI COEFFICIENT
# ============================================================================

# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_km_gamma['sep'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_km_gamma['k'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_km_gamma['p'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_km_gamma['rho'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE
# ============================================================================

# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_centroid_km_gamma['sep'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_centroid_km_gamma['k'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_centroid_km_gamma['p'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_centroid_km_gamma['rho'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE
# ============================================================================

# Column 1: By Separation
ax = axes[2, 0]
positions = sorted(df_variance_km_gamma['sep'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[2, 1]
k_vals = sorted(df_variance_km_gamma['k'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[2, 2]
p_vals = sorted(df_variance_km_gamma['p'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[2, 3]
rho_vals = sorted(df_variance_km_gamma['rho'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('Graphic F (Gamma): K-Means Clustering Structure Analysis (3×4 Matrix)\n'
             'Rows: Gini, Centroid Distance, Variance | Columns: Separation, k, p, ρ',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

#### Graphic G (Gamma): Hierarchical Clustering Only - 3×4 Validation Matrix

In [ ]:
# ============================================================================
# GRAPHIC G (GAMMA): Hierarchical Clustering Only - 3×4 Matrix Panel
# Goal: Clean visualization of Hierarchical Clustering performance across all metrics
# Note: Using existing validation data (distribution filter not available in these dataframes)
# ============================================================================

# Filter for Hierarchical Clustering only
df_gini_hc_gamma = df_gini_long[df_gini_long['Algorithm'] == 'Hierarchical']
df_centroid_hc_gamma = df_centroid_long[df_centroid_long['Algorithm'] == 'Hierarchical']
df_variance_hc_gamma = df_variance_long[df_variance_long['Algorithm'] == 'Hierarchical']

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# Hierarchical Clustering color scheme
hc_color = '#4DBBD5'

# ============================================================================
# ROW 1: GINI COEFFICIENT
# ============================================================================

# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_hc_gamma['sep'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_hc_gamma['k'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_hc_gamma['p'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_hc_gamma['rho'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE
# ============================================================================

# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_centroid_hc_gamma['sep'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_centroid_hc_gamma['k'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_centroid_hc_gamma['p'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_centroid_hc_gamma['rho'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE
# ============================================================================

# Column 1: By Separation
ax = axes[2, 0]
positions = sorted(df_variance_hc_gamma['sep'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[2, 1]
k_vals = sorted(df_variance_hc_gamma['k'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[2, 2]
p_vals = sorted(df_variance_hc_gamma['p'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[2, 3]
rho_vals = sorted(df_variance_hc_gamma['rho'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('Graphic G (Gamma): Hierarchical Clustering Structure Analysis (3×4 Matrix)\n'
             'Rows: Gini, Centroid Distance, Variance | Columns: Separation, k, p, ρ',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

---
## DISTRIBUTION COMPARISON GRAPHICS
**Comparing Normal vs Gamma Distributions**

Visual comparison of the underlying data distributions used in the simulation.

### Graphic K: Normal Distribution - Data Visualization

In [ ]:
# ============================================================================
# GRAPHIC K: Normal vs Gamma Distribution Comparison
# Pooled distribution of ALL features - side by side comparison
# ============================================================================

import glob
import os
from scipy import stats
from scipy.stats import gaussian_kde

# Find representative datasets
data_dir = "../data/original"
normal_files = glob.glob(os.path.join(data_dir, "*_normal.parquet"))
gamma_files = glob.glob(os.path.join(data_dir, "*_gamma.parquet"))

if normal_files and gamma_files:
    # Load sample files
    normal_file = normal_files[0]
    gamma_file = gamma_files[0]
    
    print(f"📂 Normal: {os.path.basename(normal_file)}")
    print(f"📂 Gamma:  {os.path.basename(gamma_file)}")
    
    df_normal = pd.read_parquet(normal_file) if normal_file.endswith(".parquet") else pd.read_csv(normal_file)
    df_gamma_dist = pd.read_parquet(gamma_file) if gamma_file.endswith(".parquet") else pd.read_csv(gamma_file)
    
    # Extract ALL feature columns and pool them
    X_normal = df_normal.select_dtypes(include=[np.number]).drop(columns=['group'], errors='ignore')
    X_gamma = df_gamma_dist.select_dtypes(include=[np.number]).drop(columns=['group'], errors='ignore')
    
    # Pool ALL values across ALL features
    all_normal = X_normal.values.flatten()
    all_gamma = X_gamma.values.flatten()
    
    print(f"\n📊 Normal: {len(all_normal):,} total values (N={X_normal.shape[0]}, p={X_normal.shape[1]})")
    print(f"📊 Gamma:  {len(all_gamma):,} total values (N={X_gamma.shape[0]}, p={X_gamma.shape[1]})")
    
    # Create side-by-side comparison
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Graphic K: Normal vs Gamma Distribution Comparison\n(Pooled Across All Features)', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    # ============================================================
    # TOP ROW: Histograms with KDE
    # ============================================================
    
    # Normal Distribution
    ax = axes[0, 0]
    ax.hist(all_normal, bins=50, density=True, alpha=0.6, color='steelblue', 
            edgecolor='black', linewidth=0.3, label='Data')
    
    kde_normal = gaussian_kde(all_normal)
    x_range = np.linspace(all_normal.min(), all_normal.max(), 200)
    ax.plot(x_range, kde_normal(x_range), 'darkblue', linewidth=2.5, label='KDE')
    
    # Fit theoretical normal
    mu, sigma = stats.norm.fit(all_normal)
    normal_fit = stats.norm.pdf(x_range, mu, sigma)
    ax.plot(x_range, normal_fit, 'r--', linewidth=2, label=f'Normal Fit (μ={mu:.2f}, σ={sigma:.2f})')
    
    ax.set_title('Normal Distribution (mvtnorm)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Value', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Gamma Distribution
    ax = axes[0, 1]
    ax.hist(all_gamma, bins=50, density=True, alpha=0.6, color='coral', 
            edgecolor='black', linewidth=0.3, label='Data')
    
    kde_gamma = gaussian_kde(all_gamma)
    x_range_g = np.linspace(all_gamma.min(), all_gamma.max(), 200)
    ax.plot(x_range_g, kde_gamma(x_range_g), 'darkred', linewidth=2.5, label='KDE')
    
    # Fit normal for comparison (should be poor fit)
    mu_g, sigma_g = stats.norm.fit(all_gamma)
    normal_fit_g = stats.norm.pdf(x_range_g, mu_g, sigma_g)
    ax.plot(x_range_g, normal_fit_g, 'b--', linewidth=2, alpha=0.7, 
            label=f'Normal Fit (μ={mu_g:.2f}, σ={sigma_g:.2f})')
    
    ax.set_title('Gamma Distribution (Z-Score Normalized)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Value', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # ============================================================
    # BOTTOM ROW: Q-Q Plots
    # ============================================================
    
    # Normal Q-Q
    ax = axes[1, 0]
    stats.probplot(all_normal, dist="norm", plot=ax)
    ax.set_title('Normal Data Q-Q Plot', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    _, (slope, intercept, r) = stats.probplot(all_normal, dist="norm")
    ax.text(0.05, 0.95, f'R² = {r**2:.4f}', transform=ax.transAxes, fontsize=11,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    
    # Gamma Q-Q
    ax = axes[1, 1]
    stats.probplot(all_gamma, dist="norm", plot=ax)
    ax.set_title('Gamma Data Q-Q Plot', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    _, (slope, intercept, r) = stats.probplot(all_gamma, dist="norm")
    ax.text(0.05, 0.95, f'R² = {r**2:.4f}', transform=ax.transAxes, fontsize=11,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
    
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # SUMMARY STATISTICS TABLE
    # ============================================================
    print("\n" + "="*70)
    print("DISTRIBUTION SUMMARY STATISTICS (Pooled All Features)")
    print("="*70)
    print(f"{'Metric':<25} {'Normal':>20} {'Gamma':>20}")
    print("-"*70)
    print(f"{'Mean':<25} {np.mean(all_normal):>20.4f} {np.mean(all_gamma):>20.4f}")
    print(f"{'Std Dev':<25} {np.std(all_normal):>20.4f} {np.std(all_gamma):>20.4f}")
    print(f"{'Skewness':<25} {stats.skew(all_normal):>20.4f} {stats.skew(all_gamma):>20.4f}")
    print(f"{'Kurtosis':<25} {stats.kurtosis(all_normal):>20.4f} {stats.kurtosis(all_gamma):>20.4f}")
    print(f"{'Min':<25} {np.min(all_normal):>20.4f} {np.min(all_gamma):>20.4f}")
    print(f"{'Max':<25} {np.max(all_normal):>20.4f} {np.max(all_gamma):>20.4f}")
    
    # Shapiro-Wilk test (sample for large N)
    sample_n = min(5000, len(all_normal))
    _, p_normal = stats.shapiro(np.random.choice(all_normal, sample_n, replace=False))
    _, p_gamma = stats.shapiro(np.random.choice(all_gamma, sample_n, replace=False))
    print(f"{'Shapiro-Wilk p-value':<25} {p_normal:>20.6f} {p_gamma:>20.6f}")
    print("="*70)
else:
    print("⚠️ Distribution files not found!")

### Graphic L: Distribution Overlay - Normal vs Gamma Statistical Comparison

In [ ]:
# ============================================================================
# GRAPHIC L: Distribution Overlay Comparison - Normal vs Gamma
# Statistical comparison with hypothesis tests and effect sizes
# ============================================================================

# Load all feature values from both distributions
pattern_normal = "*_normal.parquet"
pattern_gamma = "*_gamma.parquet"

normal_files = glob.glob(os.path.join(data_dir, pattern_normal))
gamma_files = glob.glob(os.path.join(data_dir, pattern_gamma))

if normal_files and gamma_files:
    # Pool ALL feature values from ALL files for each distribution
    all_normal = []
    all_gamma = []
    
    for nf in normal_files:
        df_n = pd.read_parquet(nf) if nf.endswith(".parquet") else pd.read_csv(nf)
        X_n = df_n.select_dtypes(include=[np.number]).drop(columns=['group'], errors='ignore')
        all_normal.extend(X_n.values.flatten())
    
    for gf in gamma_files:
        df_g = pd.read_parquet(gf) if gf.endswith(".parquet") else pd.read_csv(gf)
        X_g = df_g.select_dtypes(include=[np.number]).drop(columns=['group'], errors='ignore')
        all_gamma.extend(X_g.values.flatten())
    
    all_normal = np.array(all_normal)
    all_gamma = np.array(all_gamma)
    
    print(f"📊 Pooled Normal: {len(all_normal):,} values from {len(normal_files)} files")
    print(f"📊 Pooled Gamma: {len(all_gamma):,} values from {len(gamma_files)} files")
    
    # Create comprehensive statistical comparison
    fig = plt.figure(figsize=(16, 14))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    fig.suptitle('Graphic L: Normal vs Gamma Distribution - Statistical Comparison', 
                 fontsize=18, fontweight='bold', y=0.995)
    
    # ---- Row 1: Overlaid Distributions ----
    
    # Panel 1: Overlaid Histograms with KDE
    ax1 = fig.add_subplot(gs[0, 0])
    
    # Common bins for fair comparison
    min_val = min(all_normal.min(), all_gamma.min())
    max_val = max(all_normal.max(), all_gamma.max())
    bins = np.linspace(min_val, max_val, 50)
    
    ax1.hist(all_normal, bins=bins, density=True, alpha=0.5, color='steelblue', 
             label='Normal', edgecolor='navy', linewidth=0.3)
    ax1.hist(all_gamma, bins=bins, density=True, alpha=0.5, color='coral', 
             label='Gamma', edgecolor='darkred', linewidth=0.3)
    
    # KDE overlays
    from scipy.stats import gaussian_kde
    x_range = np.linspace(min_val, max_val, 500)
    
    kde_normal = gaussian_kde(all_normal)
    kde_gamma = gaussian_kde(all_gamma)
    
    ax1.plot(x_range, kde_normal(x_range), 'navy', linewidth=2.5, label='Normal KDE')
    ax1.plot(x_range, kde_gamma(x_range), 'darkred', linewidth=2.5, label='Gamma KDE')
    
    ax1.set_title('Overlaid Distributions', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Value (Z-Score Normalized)', fontsize=10)
    ax1.set_ylabel('Density', fontsize=10)
    ax1.legend(fontsize=9, loc='upper right')
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: CDF Comparison
    ax2 = fig.add_subplot(gs[0, 1])
    
    # Sort for CDF
    normal_sorted = np.sort(all_normal)
    gamma_sorted = np.sort(all_gamma)
    
    # Use subsample if too large
    step_n = max(1, len(normal_sorted) // 5000)
    step_g = max(1, len(gamma_sorted) // 5000)
    
    cdf_n = np.arange(1, len(normal_sorted)+1) / len(normal_sorted)
    cdf_g = np.arange(1, len(gamma_sorted)+1) / len(gamma_sorted)
    
    ax2.plot(normal_sorted[::step_n], cdf_n[::step_n], 'steelblue', linewidth=2, label='Normal')
    ax2.plot(gamma_sorted[::step_g], cdf_g[::step_g], 'coral', linewidth=2, label='Gamma')
    
    ax2.set_title('Cumulative Distribution Functions', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Value (Z-Score Normalized)', fontsize=10)
    ax2.set_ylabel('Cumulative Probability', fontsize=10)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Box-Violin Comparison
    ax3 = fig.add_subplot(gs[0, 2])
    
    # Subsample for violin plot if needed
    n_sample = min(10000, len(all_normal), len(all_gamma))
    sample_normal = np.random.choice(all_normal, n_sample, replace=False)
    sample_gamma = np.random.choice(all_gamma, n_sample, replace=False)
    
    parts = ax3.violinplot([sample_normal, sample_gamma], positions=[1, 2], showmeans=True, showmedians=True)
    
    colors = ['steelblue', 'coral']
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(colors[i])
        pc.set_alpha(0.6)
    
    ax3.set_xticks([1, 2])
    ax3.set_xticklabels(['Normal', 'Gamma'], fontsize=11)
    ax3.set_title('Violin Plot Comparison', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Value (Z-Score Normalized)', fontsize=10)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # ---- Row 2: Q-Q Plots ----
    
    # Panel 4: Q-Q Normal vs Standard Normal
    ax4 = fig.add_subplot(gs[1, 0])
    stats.probplot(all_normal[::max(1, len(all_normal)//5000)], dist="norm", plot=ax4)
    ax4.set_title('Normal Data Q-Q Plot\n(vs Standard Normal)', fontsize=11, fontweight='bold')
    ax4.get_lines()[0].set_markerfacecolor('steelblue')
    ax4.get_lines()[0].set_markeredgecolor('navy')
    ax4.grid(True, alpha=0.3)
    
    # Panel 5: Q-Q Gamma vs Standard Normal
    ax5 = fig.add_subplot(gs[1, 1])
    stats.probplot(all_gamma[::max(1, len(all_gamma)//5000)], dist="norm", plot=ax5)
    ax5.set_title('Gamma Data Q-Q Plot\n(vs Standard Normal)', fontsize=11, fontweight='bold')
    ax5.get_lines()[0].set_markerfacecolor('coral')
    ax5.get_lines()[0].set_markeredgecolor('darkred')
    ax5.grid(True, alpha=0.3)
    
    # Panel 6: Q-Q Direct Comparison (Normal vs Gamma)
    ax6 = fig.add_subplot(gs[1, 2])
    
    # Quantile-Quantile between the two distributions
    n_quantiles = 1000
    quantiles = np.linspace(0.001, 0.999, n_quantiles)
    normal_quantiles = np.quantile(all_normal, quantiles)
    gamma_quantiles = np.quantile(all_gamma, quantiles)
    
    ax6.scatter(normal_quantiles, gamma_quantiles, alpha=0.5, s=10, c='purple')
    
    # Add reference line
    min_q = min(normal_quantiles.min(), gamma_quantiles.min())
    max_q = max(normal_quantiles.max(), gamma_quantiles.max())
    ax6.plot([min_q, max_q], [min_q, max_q], 'k--', linewidth=2, label='y = x')
    
    ax6.set_title('Q-Q: Normal vs Gamma\n(Direct Comparison)', fontsize=11, fontweight='bold')
    ax6.set_xlabel('Normal Quantiles', fontsize=10)
    ax6.set_ylabel('Gamma Quantiles', fontsize=10)
    ax6.legend()
    ax6.grid(True, alpha=0.3)
    
    # ---- Row 3: Statistical Tests ----
    
    # Panel 7: Moment Comparison
    ax7 = fig.add_subplot(gs[2, 0])
    
    moments = ['Mean', 'Std Dev', 'Skewness', 'Kurtosis']
    normal_moments = [
        np.mean(all_normal),
        np.std(all_normal),
        stats.skew(all_normal),
        stats.kurtosis(all_normal)
    ]
    gamma_moments = [
        np.mean(all_gamma),
        np.std(all_gamma),
        stats.skew(all_gamma),
        stats.kurtosis(all_gamma)
    ]
    
    x_pos = np.arange(len(moments))
    width = 0.35
    
    bars1 = ax7.bar(x_pos - width/2, normal_moments, width, label='Normal', 
                    color='steelblue', alpha=0.7, edgecolor='navy')
    bars2 = ax7.bar(x_pos + width/2, gamma_moments, width, label='Gamma', 
                    color='coral', alpha=0.7, edgecolor='darkred')
    
    ax7.set_xticks(x_pos)
    ax7.set_xticklabels(moments, fontsize=10)
    ax7.set_title('Distribution Moments', fontsize=12, fontweight='bold')
    ax7.set_ylabel('Value', fontsize=10)
    ax7.legend()
    ax7.grid(True, alpha=0.3, axis='y')
    ax7.axhline(y=0, color='black', linewidth=0.5)
    
    # Panel 8-9: Statistical Test Results Table
    ax8 = fig.add_subplot(gs[2, 1:])
    ax8.axis('off')
    
    # Perform statistical tests
    # 1. Kolmogorov-Smirnov test
    ks_stat, ks_p = stats.ks_2samp(all_normal[:50000], all_gamma[:50000])
    
    # 2. Two-sample t-test
    t_stat, t_p = stats.ttest_ind(all_normal[:50000], all_gamma[:50000])
    
    # 3. Levene's test (variance equality)
    levene_stat, levene_p = stats.levene(all_normal[:50000], all_gamma[:50000])
    
    # 4. Shapiro-Wilk (on subsample)
    shapiro_n_stat, shapiro_n_p = stats.shapiro(all_normal[:5000])
    shapiro_g_stat, shapiro_g_p = stats.shapiro(all_gamma[:5000])
    
    # 5. Cohen's d (effect size)
    pooled_std = np.sqrt((np.var(all_normal) + np.var(all_gamma)) / 2)
    cohens_d = (np.mean(all_normal) - np.mean(all_gamma)) / pooled_std
    
    test_data = [
        ['Kolmogorov-Smirnov', f'{ks_stat:.4f}', f'{ks_p:.2e}', 
         'Significant' if ks_p < 0.05 else 'Not Sig.'],
        ['Two-Sample t-test', f'{t_stat:.4f}', f'{t_p:.2e}', 
         'Significant' if t_p < 0.05 else 'Not Sig.'],
        ["Levene's Test (Variance)", f'{levene_stat:.4f}', f'{levene_p:.2e}', 
         'Different' if levene_p < 0.05 else 'Equal'],
        ['Shapiro-Wilk (Normal)', f'{shapiro_n_stat:.4f}', f'{shapiro_n_p:.2e}', 
         'Normal' if shapiro_n_p > 0.05 else 'Non-Normal'],
        ['Shapiro-Wilk (Gamma)', f'{shapiro_g_stat:.4f}', f'{shapiro_g_p:.2e}', 
         'Normal' if shapiro_g_p > 0.05 else 'Non-Normal'],
        ["Cohen's d Effect Size", f'{cohens_d:.4f}', '—', 
         f'{"Small" if abs(cohens_d) < 0.5 else "Medium" if abs(cohens_d) < 0.8 else "Large"}']
    ]
    
    table = ax8.table(cellText=test_data,
                     colLabels=['Test', 'Statistic', 'p-value', 'Interpretation'],
                     cellLoc='center',
                     loc='center',
                     bbox=[0, 0, 1, 0.9])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.2)
    
    # Style header
    for i in range(4):
        table[(0, i)].set_facecolor('#4A90A4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    ax8.set_title('Statistical Hypothesis Tests', fontsize=12, fontweight='bold', pad=20)
    
    plt.show()
    
    # Print summary
    print("\n" + "="*70)
    print("📊 STATISTICAL COMPARISON SUMMARY")
    print("="*70)
    print(f"\nNormal Distribution (n={len(all_normal):,}):")
    print(f"  Mean: {np.mean(all_normal):.4f}, Std: {np.std(all_normal):.4f}")
    print(f"  Skewness: {stats.skew(all_normal):.4f}, Kurtosis: {stats.kurtosis(all_normal):.4f}")
    print(f"\nGamma Distribution (n={len(all_gamma):,}):")
    print(f"  Mean: {np.mean(all_gamma):.4f}, Std: {np.std(all_gamma):.4f}")
    print(f"  Skewness: {stats.skew(all_gamma):.4f}, Kurtosis: {stats.kurtosis(all_gamma):.4f}")
    print(f"\nEffect Size (Cohen's d): {cohens_d:.4f}")
    print("="*70)
else:
    print("⚠️ Distribution files not found!")

## PCA and t-SNE Visualization: Real vs Synthetic Data

Compare the structure of real and synthetic datasets using dimensionality reduction techniques.

In [ ]:
# ============================================================================
# GRAPHIC M: PCA and t-SNE Visualization - Real vs Synthetic Data
# Compare real and synthetic data distributions using dimensionality reduction
# Uses ALL synthetic files to show overall comparison with 2 colors
# ============================================================================

import glob
import os
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Define data directory
data_dir_real = "../data/original"
data_dir_synthetic = "../data/synthetic"

# Select a specific parameter set for detailed comparison
# Choose: N=1000, p=10, k=2, rho=0, sep=2, rep=1
target_params = {
    'N': 1000, 
    'p': 10, 
    'k': 2, 
    'rho': 0, 
    'sep': 2, 
    'rep': 1
}

# Pattern for real data (both normal and gamma)
real_pattern_normal = f"OD_N{target_params['N']}_p{target_params['p']}_k{target_params['k']}_rho{target_params['rho']}_sep{target_params['sep']}_normal.parquet"
real_pattern_gamma  = f"OD_N{target_params['N']}_p{target_params['p']}_k{target_params['k']}_rho{target_params['rho']}_sep{target_params['sep']}_gamma.parquet"

# Pattern for synthetic data (new naming: SD_cart_N{N}_p{p}_k{k}_rho{rho}_sep{sep}_{dist}_syn{id}.parquet)
syn_base = f"SD_cart_N{target_params['N']}_p{target_params['p']}_k{target_params['k']}_rho{target_params['rho']}_sep{target_params['sep']}"

# Load real data paths
real_file_normal = os.path.join(data_dir_real, real_pattern_normal)
real_file_gamma  = os.path.join(data_dir_real, real_pattern_gamma)

# Load synthetic data file lists
syn_files_normal = sorted(glob.glob(os.path.join(data_dir_synthetic, syn_base + "_normal_syn*.parquet")))
syn_files_gamma  = sorted(glob.glob(os.path.join(data_dir_synthetic, syn_base + "_gamma_syn*.parquet")))

print(f"Found {len(syn_files_normal)} synthetic normal files")
print(f"Found {len(syn_files_gamma)} synthetic gamma files")

# Load and combine data for normal distribution
if os.path.exists(real_file_normal) and len(syn_files_normal) > 0:
    print("Loading normal distribution data...")
    
    # Load real data
    df_real_normal = pd.read_parquet(real_file_normal)
    df_real_normal = df_real_normal[df_real_normal['rep'] == target_params['rep']]
    X_real_normal = df_real_normal.select_dtypes(include=[np.number]).drop(columns=['group', 'rep'], errors='ignore').values
    
    # Load ALL synthetic data (filter to rep=1 per file)
    X_syn_normal_all = []
    for i, syn_file in enumerate(syn_files_normal):
        df_syn = pd.read_parquet(syn_file)
        df_syn = df_syn[df_syn['rep'] == target_params['rep']]
        X_syn = df_syn.select_dtypes(include=[np.number]).drop(columns=['group', 'rep'], errors='ignore').values
        X_syn_normal_all.append(X_syn)
        if (i + 1) % 2 == 0:  # Progress update every 2 files
            print(f"  Loaded {i+1}/{len(syn_files_normal)} synthetic files", end='\r')
    
    print(f"  Loaded {len(syn_files_normal)}/{len(syn_files_normal)} synthetic files")
    
    # Combine all synthetic data
    X_syn_normal_combined = np.vstack(X_syn_normal_all)
    
    # Combine real and synthetic
    X_combined_normal = np.vstack([X_real_normal, X_syn_normal_combined])
    
    # Create binary labels: 0 = Real, 1 = Synthetic
    y_labels_normal = np.concatenate([
        np.zeros(len(X_real_normal)),
        np.ones(len(X_syn_normal_combined))
    ])
    
    print(f"✓ Normal data loaded: Real={len(X_real_normal):,}, Synthetic={len(X_syn_normal_combined):,}, Total={len(X_combined_normal):,}")
else:
    print("⚠️ Normal distribution data not found!")
    X_combined_normal = None

# Load and combine data for gamma distribution
if os.path.exists(real_file_gamma) and len(syn_files_gamma) > 0:
    print("Loading gamma distribution data...")
    
    # Load real data
    df_real_gamma = pd.read_parquet(real_file_gamma)
    df_real_gamma = df_real_gamma[df_real_gamma['rep'] == target_params['rep']]
    X_real_gamma = df_real_gamma.select_dtypes(include=[np.number]).drop(columns=['group', 'rep'], errors='ignore').values
    
    # Load ALL synthetic data (filter to rep=1 per file)
    X_syn_gamma_all = []
    for i, syn_file in enumerate(syn_files_gamma):
        df_syn = pd.read_parquet(syn_file)
        df_syn = df_syn[df_syn['rep'] == target_params['rep']]
        X_syn = df_syn.select_dtypes(include=[np.number]).drop(columns=['group', 'rep'], errors='ignore').values
        X_syn_gamma_all.append(X_syn)
        if (i + 1) % 2 == 0:  # Progress update every 2 files
            print(f"  Loaded {i+1}/{len(syn_files_gamma)} synthetic files", end='\r')
    
    print(f"  Loaded {len(syn_files_gamma)}/{len(syn_files_gamma)} synthetic files")
    
    # Combine all synthetic data
    X_syn_gamma_combined = np.vstack(X_syn_gamma_all)
    
    # Combine real and synthetic
    X_combined_gamma = np.vstack([X_real_gamma, X_syn_gamma_combined])
    
    # Create binary labels: 0 = Real, 1 = Synthetic
    y_labels_gamma = np.concatenate([
        np.zeros(len(X_real_gamma)),
        np.ones(len(X_syn_gamma_combined))
    ])
    
    print(f"✓ Gamma data loaded: Real={len(X_real_gamma):,}, Synthetic={len(X_syn_gamma_combined):,}, Total={len(X_combined_gamma):,}")
else:
    print("⚠️ Gamma distribution data not found!")
    X_combined_gamma = None

# Create visualization
if X_combined_normal is not None or X_combined_gamma is not None:
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.25)
    
    fig.suptitle(f'Graphic M: Real vs Synthetic Data Comparison (PCA & t-SNE)\nParameters: N={target_params["N"]}, p={target_params["p"]}, k={target_params["k"]}, sep={target_params["sep"]} | All {len(syn_files_normal)} Synthetic Replicates', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Color palette: Real (blue) vs Synthetic (orange)
    color_real = '#1f77b4'      # Blue
    color_synthetic = '#ff7f0e'  # Orange
    
    # ============================================================================
    # NORMAL DISTRIBUTION
    # ============================================================================
    
    if X_combined_normal is not None:
        print("\nProcessing Normal Distribution...")
        
        # Standardize data
        scaler_normal = StandardScaler()
        X_scaled_normal = scaler_normal.fit_transform(X_combined_normal)
        
        # --- PCA for Normal Distribution ---
        print("  Computing PCA...")
        ax1 = fig.add_subplot(gs[0, 0])
        
        pca_normal = PCA(n_components=2)
        X_pca_normal = pca_normal.fit_transform(X_scaled_normal)
        
        # Plot synthetic first (in background), then real (in foreground)
        mask_syn = y_labels_normal == 1
        mask_real = y_labels_normal == 0
        
        ax1.scatter(X_pca_normal[mask_syn, 0], X_pca_normal[mask_syn, 1], 
                   c=color_synthetic, label='Synthetic', alpha=0.3, s=8, edgecolors='none')
        ax1.scatter(X_pca_normal[mask_real, 0], X_pca_normal[mask_real, 1], 
                   c=color_real, label='Real', alpha=0.7, s=15, edgecolors='darkblue', linewidth=0.3)
        
        ax1.set_title(f'PCA: Normal Distribution\nExplained Variance: {pca_normal.explained_variance_ratio_[0]:.2%} & {pca_normal.explained_variance_ratio_[1]:.2%}', 
                     fontsize=12, fontweight='bold')
        ax1.set_xlabel(f'PC1 ({pca_normal.explained_variance_ratio_[0]:.1%})', fontsize=10)
        ax1.set_ylabel(f'PC2 ({pca_normal.explained_variance_ratio_[1]:.1%})', fontsize=10)
        ax1.legend(loc='best', fontsize=10, framealpha=0.9)
        ax1.grid(True, alpha=0.3)
        
        # --- t-SNE for Normal Distribution ---
        print("  Computing t-SNE...")
        ax2 = fig.add_subplot(gs[0, 1])
        
        # Use subsample for t-SNE if data is too large
        max_samples = 5000
        if len(X_scaled_normal) > max_samples:
            # Stratified sampling to maintain real/synthetic ratio
            n_real = int(max_samples * len(X_real_normal) / len(X_combined_normal))
            n_syn = max_samples - n_real
            
            indices_real = np.random.choice(np.where(y_labels_normal == 0)[0], n_real, replace=False)
            indices_syn = np.random.choice(np.where(y_labels_normal == 1)[0], n_syn, replace=False)
            indices = np.concatenate([indices_real, indices_syn])
            
            X_tsne_input = X_scaled_normal[indices]
            y_tsne_labels = y_labels_normal[indices]
        else:
            X_tsne_input = X_scaled_normal
            y_tsne_labels = y_labels_normal
        
        tsne_normal = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
        X_tsne_normal = tsne_normal.fit_transform(X_tsne_input)
        
        # Plot synthetic first (in background), then real (in foreground)
        mask_syn = y_tsne_labels == 1
        mask_real = y_tsne_labels == 0
        
        ax2.scatter(X_tsne_normal[mask_syn, 0], X_tsne_normal[mask_syn, 1], 
                   c=color_synthetic, label='Synthetic', alpha=0.3, s=8, edgecolors='none')
        ax2.scatter(X_tsne_normal[mask_real, 0], X_tsne_normal[mask_real, 1], 
                   c=color_real, label='Real', alpha=0.7, s=15, edgecolors='darkblue', linewidth=0.3)
        
        ax2.set_title('t-SNE: Normal Distribution', fontsize=12, fontweight='bold')
        ax2.set_xlabel('t-SNE Component 1', fontsize=10)
        ax2.set_ylabel('t-SNE Component 2', fontsize=10)
        ax2.legend(loc='best', fontsize=10, framealpha=0.9)
        ax2.grid(True, alpha=0.3)
    
    # ============================================================================
    # GAMMA DISTRIBUTION
    # ============================================================================
    
    if X_combined_gamma is not None:
        print("\nProcessing Gamma Distribution...")
        
        # Standardize data
        scaler_gamma = StandardScaler()
        X_scaled_gamma = scaler_gamma.fit_transform(X_combined_gamma)
        
        # --- PCA for Gamma Distribution ---
        print("  Computing PCA...")
        ax3 = fig.add_subplot(gs[1, 0])
        
        pca_gamma = PCA(n_components=2)
        X_pca_gamma = pca_gamma.fit_transform(X_scaled_gamma)
        
        # Plot synthetic first (in background), then real (in foreground)
        mask_syn = y_labels_gamma == 1
        mask_real = y_labels_gamma == 0
        
        ax3.scatter(X_pca_gamma[mask_syn, 0], X_pca_gamma[mask_syn, 1], 
                   c=color_synthetic, label='Synthetic', alpha=0.3, s=8, edgecolors='none')
        ax3.scatter(X_pca_gamma[mask_real, 0], X_pca_gamma[mask_real, 1], 
                   c=color_real, label='Real', alpha=0.7, s=15, edgecolors='darkblue', linewidth=0.3)
        
        ax3.set_title(f'PCA: Gamma Distribution\nExplained Variance: {pca_gamma.explained_variance_ratio_[0]:.2%} & {pca_gamma.explained_variance_ratio_[1]:.2%}', 
                     fontsize=12, fontweight='bold')
        ax3.set_xlabel(f'PC1 ({pca_gamma.explained_variance_ratio_[0]:.1%})', fontsize=10)
        ax3.set_ylabel(f'PC2 ({pca_gamma.explained_variance_ratio_[1]:.1%})', fontsize=10)
        ax3.legend(loc='best', fontsize=10, framealpha=0.9)
        ax3.grid(True, alpha=0.3)
        
        # --- t-SNE for Gamma Distribution ---
        print("  Computing t-SNE...")
        ax4 = fig.add_subplot(gs[1, 1])
        
        # Use subsample for t-SNE if data is too large
        max_samples = 5000
        if len(X_scaled_gamma) > max_samples:
            # Stratified sampling to maintain real/synthetic ratio
            n_real = int(max_samples * len(X_real_gamma) / len(X_combined_gamma))
            n_syn = max_samples - n_real
            
            indices_real = np.random.choice(np.where(y_labels_gamma == 0)[0], n_real, replace=False)
            indices_syn = np.random.choice(np.where(y_labels_gamma == 1)[0], n_syn, replace=False)
            indices = np.concatenate([indices_real, indices_syn])
            
            X_tsne_input = X_scaled_gamma[indices]
            y_tsne_labels = y_labels_gamma[indices]
        else:
            X_tsne_input = X_scaled_gamma
            y_tsne_labels = y_labels_gamma
        
        tsne_gamma = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
        X_tsne_gamma = tsne_gamma.fit_transform(X_tsne_input)
        
        # Plot synthetic first (in background), then real (in foreground)
        mask_syn = y_tsne_labels == 1
        mask_real = y_tsne_labels == 0
        
        ax4.scatter(X_tsne_gamma[mask_syn, 0], X_tsne_gamma[mask_syn, 1], 
                   c=color_synthetic, label='Synthetic', alpha=0.3, s=8, edgecolors='none')
        ax4.scatter(X_tsne_gamma[mask_real, 0], X_tsne_gamma[mask_real, 1], 
                   c=color_real, label='Real', alpha=0.7, s=15, edgecolors='darkblue', linewidth=0.3)
        
        ax4.set_title('t-SNE: Gamma Distribution', fontsize=12, fontweight='bold')
        ax4.set_xlabel('t-SNE Component 1', fontsize=10)
        ax4.set_ylabel('t-SNE Component 2', fontsize=10)
        ax4.legend(loc='best', fontsize=10, framealpha=0.9)
        ax4.grid(True, alpha=0.3)
    
    plt.show()
    
    print("\n" + "="*70)
    print("📊 PCA & t-SNE VISUALIZATION SUMMARY")
    print("="*70)
    print(f"Parameters: N={target_params['N']}, p={target_params['p']}, k={target_params['k']}, sep={target_params['sep']}")
    print(f"Synthetic files per distribution: {len(syn_files_normal)}")
    if X_combined_normal is not None:
        print(f"\nNormal Distribution:")
        print(f"  Real samples: {len(X_real_normal):,}")
        print(f"  Synthetic samples: {len(X_syn_normal_combined):,}")
        print(f"  Total samples: {len(X_combined_normal):,}")
        print(f"  PCA variance explained: {sum(pca_normal.explained_variance_ratio_[:2]):.2%}")
    if X_combined_gamma is not None:
        print(f"\nGamma Distribution:")
        print(f"  Real samples: {len(X_real_gamma):,}")
        print(f"  Synthetic samples: {len(X_syn_gamma_combined):,}")
        print(f"  Total samples: {len(X_combined_gamma):,}")
        print(f"  PCA variance explained: {sum(pca_gamma.explained_variance_ratio_[:2]):.2%}")
    print("="*70)
else:
    print("⚠️ No data available for visualization!")

## PCA and t-SNE Visualization: K-Means and Hierarchical Clustering Results

Visualize how the clustering algorithms partition the data in reduced dimensional space.

In [ ]:
# ============================================================================
# GRAPHIC N: PCA and t-SNE with Clustering Results
# Visualize K-Means and Hierarchical clustering results in reduced dimensions
# Shows how clustering algorithms partition the data
# ============================================================================

from sklearn.cluster import AgglomerativeClustering

# Use the same data and parameters as before
target_params = {
    'N': 1000, 
    'p': 10, 
    'k': 2, 
    'rho': 0, 
    'sep': 2, 
    'rep': 1
}

# We'll use the real data that we already loaded
if 'X_real_normal' in locals() and 'X_real_gamma' in locals():
    print("Using previously loaded real data...")
    
    # Create visualization
    fig = plt.figure(figsize=(18, 16))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.25)
    
    fig.suptitle(f'Graphic N: Clustering Results Visualization (PCA & t-SNE)\nParameters: N={target_params["N"]}, p={target_params["p"]}, k={target_params["k"]}, sep={target_params["sep"]}', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Color palette for clusters
    cluster_colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    
    # ============================================================================
    # NORMAL DISTRIBUTION
    # ============================================================================
    
    print("\nProcessing Normal Distribution...")
    
    # Standardize data
    scaler_normal = StandardScaler()
    X_scaled_normal = scaler_normal.fit_transform(X_real_normal)
    
    # Apply K-Means and Hierarchical Clustering
    k = target_params['k']
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    hc = AgglomerativeClustering(n_clusters=k)
    
    labels_km_normal = kmeans.fit_predict(X_scaled_normal)
    labels_hc_normal = hc.fit_predict(X_scaled_normal)
    
    # Compute PCA
    print("  Computing PCA...")
    pca_normal = PCA(n_components=2)
    X_pca_normal = pca_normal.fit_transform(X_scaled_normal)
    
    # --- PCA with K-Means ---
    ax1 = fig.add_subplot(gs[0, 0])
    for cluster_id in range(k):
        mask = labels_km_normal == cluster_id
        ax1.scatter(X_pca_normal[mask, 0], X_pca_normal[mask, 1], 
                   c=cluster_colors[cluster_id], label=f'Cluster {cluster_id+1}', 
                   alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    
    ax1.set_title(f'PCA + K-Means: Normal Distribution\nExplained Variance: {sum(pca_normal.explained_variance_ratio_[:2]):.2%}', 
                 fontsize=12, fontweight='bold')
    ax1.set_xlabel(f'PC1 ({pca_normal.explained_variance_ratio_[0]:.1%})', fontsize=10)
    ax1.set_ylabel(f'PC2 ({pca_normal.explained_variance_ratio_[1]:.1%})', fontsize=10)
    ax1.legend(loc='best', fontsize=10, framealpha=0.9)
    ax1.grid(True, alpha=0.3)
    
    # --- PCA with Hierarchical Clustering ---
    ax2 = fig.add_subplot(gs[0, 1])
    for cluster_id in range(k):
        mask = labels_hc_normal == cluster_id
        ax2.scatter(X_pca_normal[mask, 0], X_pca_normal[mask, 1], 
                   c=cluster_colors[cluster_id], label=f'Cluster {cluster_id+1}', 
                   alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    
    ax2.set_title(f'PCA + Hierarchical: Normal Distribution\nExplained Variance: {sum(pca_normal.explained_variance_ratio_[:2]):.2%}', 
                 fontsize=12, fontweight='bold')
    ax2.set_xlabel(f'PC1 ({pca_normal.explained_variance_ratio_[0]:.1%})', fontsize=10)
    ax2.set_ylabel(f'PC2 ({pca_normal.explained_variance_ratio_[1]:.1%})', fontsize=10)
    ax2.legend(loc='best', fontsize=10, framealpha=0.9)
    ax2.grid(True, alpha=0.3)
    
    # Compute t-SNE (with sampling if needed)
    print("  Computing t-SNE...")
    max_samples = 5000
    if len(X_scaled_normal) > max_samples:
        indices = np.random.choice(len(X_scaled_normal), max_samples, replace=False)
        X_tsne_input = X_scaled_normal[indices]
        labels_km_tsne = labels_km_normal[indices]
        labels_hc_tsne = labels_hc_normal[indices]
    else:
        X_tsne_input = X_scaled_normal
        labels_km_tsne = labels_km_normal
        labels_hc_tsne = labels_hc_normal
    
    tsne_normal = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
    X_tsne_normal = tsne_normal.fit_transform(X_tsne_input)
    
    # --- t-SNE with K-Means ---
    ax3 = fig.add_subplot(gs[1, 0])
    for cluster_id in range(k):
        mask = labels_km_tsne == cluster_id
        ax3.scatter(X_tsne_normal[mask, 0], X_tsne_normal[mask, 1], 
                   c=cluster_colors[cluster_id], label=f'Cluster {cluster_id+1}', 
                   alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    
    ax3.set_title('t-SNE + K-Means: Normal Distribution', fontsize=12, fontweight='bold')
    ax3.set_xlabel('t-SNE Component 1', fontsize=10)
    ax3.set_ylabel('t-SNE Component 2', fontsize=10)
    ax3.legend(loc='best', fontsize=10, framealpha=0.9)
    ax3.grid(True, alpha=0.3)
    
    # --- t-SNE with Hierarchical Clustering ---
    ax4 = fig.add_subplot(gs[1, 1])
    for cluster_id in range(k):
        mask = labels_hc_tsne == cluster_id
        ax4.scatter(X_tsne_normal[mask, 0], X_tsne_normal[mask, 1], 
                   c=cluster_colors[cluster_id], label=f'Cluster {cluster_id+1}', 
                   alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    
    ax4.set_title('t-SNE + Hierarchical: Normal Distribution', fontsize=12, fontweight='bold')
    ax4.set_xlabel('t-SNE Component 1', fontsize=10)
    ax4.set_ylabel('t-SNE Component 2', fontsize=10)
    ax4.legend(loc='best', fontsize=10, framealpha=0.9)
    ax4.grid(True, alpha=0.3)
    
    plt.show()
    
    # ============================================================================
    # GAMMA DISTRIBUTION
    # ============================================================================
    
    print("\nProcessing Gamma Distribution...")
    
    # Standardize data
    scaler_gamma = StandardScaler()
    X_scaled_gamma = scaler_gamma.fit_transform(X_real_gamma)
    
    # Apply K-Means and Hierarchical Clustering
    kmeans_gamma = KMeans(n_clusters=k, random_state=42, n_init=10)
    hc_gamma = AgglomerativeClustering(n_clusters=k)
    
    labels_km_gamma = kmeans_gamma.fit_predict(X_scaled_gamma)
    labels_hc_gamma = hc_gamma.fit_predict(X_scaled_gamma)
    
    # Create second figure for gamma
    fig2 = plt.figure(figsize=(18, 16))
    gs2 = fig2.add_gridspec(2, 2, hspace=0.3, wspace=0.25)
    
    fig2.suptitle(f'Graphic N: Clustering Results Visualization (PCA & t-SNE)\nGamma Distribution | Parameters: N={target_params["N"]}, p={target_params["p"]}, k={target_params["k"]}, sep={target_params["sep"]}', 
                  fontsize=16, fontweight='bold', y=0.995)
    
    # Compute PCA
    print("  Computing PCA...")
    pca_gamma = PCA(n_components=2)
    X_pca_gamma = pca_gamma.fit_transform(X_scaled_gamma)
    
    # --- PCA with K-Means ---
    ax5 = fig2.add_subplot(gs2[0, 0])
    for cluster_id in range(k):
        mask = labels_km_gamma == cluster_id
        ax5.scatter(X_pca_gamma[mask, 0], X_pca_gamma[mask, 1], 
                   c=cluster_colors[cluster_id], label=f'Cluster {cluster_id+1}', 
                   alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    
    ax5.set_title(f'PCA + K-Means: Gamma Distribution\nExplained Variance: {sum(pca_gamma.explained_variance_ratio_[:2]):.2%}', 
                 fontsize=12, fontweight='bold')
    ax5.set_xlabel(f'PC1 ({pca_gamma.explained_variance_ratio_[0]:.1%})', fontsize=10)
    ax5.set_ylabel(f'PC2 ({pca_gamma.explained_variance_ratio_[1]:.1%})', fontsize=10)
    ax5.legend(loc='best', fontsize=10, framealpha=0.9)
    ax5.grid(True, alpha=0.3)
    
    # --- PCA with Hierarchical Clustering ---
    ax6 = fig2.add_subplot(gs2[0, 1])
    for cluster_id in range(k):
        mask = labels_hc_gamma == cluster_id
        ax6.scatter(X_pca_gamma[mask, 0], X_pca_gamma[mask, 1], 
                   c=cluster_colors[cluster_id], label=f'Cluster {cluster_id+1}', 
                   alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    
    ax6.set_title(f'PCA + Hierarchical: Gamma Distribution\nExplained Variance: {sum(pca_gamma.explained_variance_ratio_[:2]):.2%}', 
                 fontsize=12, fontweight='bold')
    ax6.set_xlabel(f'PC1 ({pca_gamma.explained_variance_ratio_[0]:.1%})', fontsize=10)
    ax6.set_ylabel(f'PC2 ({pca_gamma.explained_variance_ratio_[1]:.1%})', fontsize=10)
    ax6.legend(loc='best', fontsize=10, framealpha=0.9)
    ax6.grid(True, alpha=0.3)
    
    # Compute t-SNE (with sampling if needed)
    print("  Computing t-SNE...")
    if len(X_scaled_gamma) > max_samples:
        indices = np.random.choice(len(X_scaled_gamma), max_samples, replace=False)
        X_tsne_input = X_scaled_gamma[indices]
        labels_km_tsne_gamma = labels_km_gamma[indices]
        labels_hc_tsne_gamma = labels_hc_gamma[indices]
    else:
        X_tsne_input = X_scaled_gamma
        labels_km_tsne_gamma = labels_km_gamma
        labels_hc_tsne_gamma = labels_hc_gamma
    
    tsne_gamma = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
    X_tsne_gamma = tsne_gamma.fit_transform(X_tsne_input)
    
    # --- t-SNE with K-Means ---
    ax7 = fig2.add_subplot(gs2[1, 0])
    for cluster_id in range(k):
        mask = labels_km_tsne_gamma == cluster_id
        ax7.scatter(X_tsne_gamma[mask, 0], X_tsne_gamma[mask, 1], 
                   c=cluster_colors[cluster_id], label=f'Cluster {cluster_id+1}', 
                   alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    
    ax7.set_title('t-SNE + K-Means: Gamma Distribution', fontsize=12, fontweight='bold')
    ax7.set_xlabel('t-SNE Component 1', fontsize=10)
    ax7.set_ylabel('t-SNE Component 2', fontsize=10)
    ax7.legend(loc='best', fontsize=10, framealpha=0.9)
    ax7.grid(True, alpha=0.3)
    
    # --- t-SNE with Hierarchical Clustering ---
    ax8 = fig2.add_subplot(gs2[1, 1])
    for cluster_id in range(k):
        mask = labels_hc_tsne_gamma == cluster_id
        ax8.scatter(X_tsne_gamma[mask, 0], X_tsne_gamma[mask, 1], 
                   c=cluster_colors[cluster_id], label=f'Cluster {cluster_id+1}', 
                   alpha=0.6, s=20, edgecolors='black', linewidth=0.3)
    
    ax8.set_title('t-SNE + Hierarchical: Gamma Distribution', fontsize=12, fontweight='bold')
    ax8.set_xlabel('t-SNE Component 1', fontsize=10)
    ax8.set_ylabel('t-SNE Component 2', fontsize=10)
    ax8.legend(loc='best', fontsize=10, framealpha=0.9)
    ax8.grid(True, alpha=0.3)
    
    plt.show()
    
    print("\n" + "="*70)
    print("📊 CLUSTERING VISUALIZATION SUMMARY")
    print("="*70)
    print(f"Parameters: N={target_params['N']}, p={target_params['p']}, k={target_params['k']}, sep={target_params['sep']}")
    print(f"\nNormal Distribution:")
    print(f"  Samples: {len(X_real_normal):,}")
    print(f"  PCA variance explained: {sum(pca_normal.explained_variance_ratio_[:2]):.2%}")
    print(f"\nGamma Distribution:")
    print(f"  Samples: {len(X_real_gamma):,}")
    print(f"  PCA variance explained: {sum(pca_gamma.explained_variance_ratio_[:2]):.2%}")
    print("="*70)
    
else:
    print("⚠️ Real data not found. Please run the previous cells first to load the data.")

---
## 8. Metric Definitions & Computation

Helper functions for Gini coefficient, mean centroid distance,
and mean variance difference — used in the per-distribution breakdowns below.

## 3-Relevant Metrics

### 3.0.1-Gini Definition

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

def calculate_gini_coefficient(cluster_sizes):
    """
    Calculate Gini coefficient for cluster size distribution.
    G = Σ|n_i - n_j| / (2k * Σn_i)
    """
    sizes = np.array(cluster_sizes, dtype=float)
    n = len(sizes)
    
    if n == 0 or sizes.sum() == 0:
        return 0.0
    
    # Calculate sum of absolute differences
    abs_diff_sum = 0
    for i in range(n):
        for j in range(n):
            abs_diff_sum += abs(sizes[i] - sizes[j])
    
    gini = abs_diff_sum / (2 * n * sizes.sum())
    return gini

def get_aligned_cluster_sizes(true_labels, pred_labels):
    """
    Get predicted cluster sizes aligned to true labels using Hungarian algorithm.
    Returns the sizes of predicted clusters after optimal alignment.
    """
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(true_labels, pred_labels)
    cost_matrix = -cm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Get sizes after alignment
    pred_sizes = np.bincount(pred_labels)
    aligned_sizes = pred_sizes[col_ind] if len(col_ind) == len(pred_sizes) else pred_sizes
    
    return aligned_sizes

# Process all data files and calculate Gini coefficients
data_dir = "../data/original"
gini_results = []

# Get list of all real data files
real_files = glob.glob(os.path.join(data_dir, "*.parquet"))
print(f"📂 Processing {len(real_files)} datasets for Gini analysis...")

# Sample a subset for efficiency (process all unique N, k, rho, sep combinations)
processed_params = set()

for idx, filepath in enumerate(real_files):
    # Parse filename for parameters
    filename = os.path.basename(filepath)
    parts = filename.replace('.parquet', '').split('_')
    
    try:
        # OD_ prefix shifts indices by 1; no rep in parquet filenames
        N = int(parts[1].replace('N', ''))
        p = int(parts[2].replace('p', ''))
        k = int(parts[3].replace('k', ''))
        rho = float(parts[4].replace('rho', ''))
        sep = float(parts[5].replace('sep', ''))
        rep = 1  # parquet files contain all reps; default to 1
    except (IndexError, ValueError):
        continue
    
    # Process each parameter combination once (first rep only for speed)
    param_key = (N, p, k, rho, sep)
    if param_key in processed_params:
        continue
    processed_params.add(param_key)
    
    # Load data
    try:
        data = pd.read_parquet(filepath) if filepath.endswith(".parquet") else pd.read_csv(filepath)
        X = data.drop(columns=['group']).values
        true_labels = data['group'].astype(int).values - data['group'].astype(int).min()
        
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        k_true = len(np.unique(true_labels))
        
        # K-Means
        kmeans = KMeans(n_clusters=k_true, n_init=10, random_state=42)
        pred_km = kmeans.fit_predict(X_scaled)
        
        # Hierarchical
        hc = AgglomerativeClustering(n_clusters=k_true, linkage='ward')
        pred_hc = hc.fit_predict(X_scaled)
        
        # Calculate Gini for true labels (baseline)
        true_sizes = np.bincount(true_labels)
        gini_true = calculate_gini_coefficient(true_sizes)
        
        # Calculate Gini for predicted (aligned via Hungarian)
        km_sizes = get_aligned_cluster_sizes(true_labels, pred_km)
        hc_sizes = get_aligned_cluster_sizes(true_labels, pred_hc)
        
        gini_km = calculate_gini_coefficient(km_sizes)
        gini_hc = calculate_gini_coefficient(hc_sizes)
        
        gini_results.append({
            'N': N, 'p': p, 'k': k, 'rho': rho, 'sep': sep,
            'gini_true': gini_true,
            'gini_kmeans': gini_km,
            'gini_hc': gini_hc
        })
        
    except Exception as e:
        continue

df_gini = pd.DataFrame(gini_results)
print(f"\n✅ Processed {len(df_gini)} parameter combinations")
print(df_gini.head())

# Reshape for plotting
df_gini_long = pd.melt(
    df_gini,
    id_vars=['N', 'p', 'k', 'rho', 'sep', 'gini_true'],
    value_vars=['gini_kmeans', 'gini_hc'],
    var_name='Algorithm',
    value_name='Gini'
)
df_gini_long['Algorithm'] = df_gini_long['Algorithm'].map({
    'gini_kmeans': 'K-Means',
    'gini_hc': 'Hierarchical'
})

### 3.0.2-Mean Centroid Distance Definition

In [ ]:
# ============================================================================
# GRAPHIC C: Mean Centroid Distance (Central Tendency Fidelity)
# Goal: Measure how accurately algorithms recover true cluster centers
# Method: Use Hungarian Algorithm to match predicted centroids to true centroids
# D_μ = (1/k) Σ ||μ_pred(i) - μ_true(matched)||_2
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

def calculate_centroids(X, labels):
    """Calculate centroids for each cluster."""
    unique_labels = np.unique(labels)
    centroids = np.array([X[labels == label].mean(axis=0) for label in unique_labels])
    return centroids

def hungarian_matched_centroid_distance(true_centroids, pred_centroids):
    """
    Use Hungarian algorithm to optimally match predicted centroids to true centroids.
    Returns mean Euclidean distance between matched pairs.
    """
    # Build distance matrix between all pairs
    dist_matrix = cdist(true_centroids, pred_centroids, metric='euclidean')
    
    # Hungarian algorithm finds optimal matching
    row_ind, col_ind = linear_sum_assignment(dist_matrix)
    
    # Calculate mean distance of matched pairs
    matched_distances = dist_matrix[row_ind, col_ind]
    mean_distance = matched_distances.mean()
    
    return mean_distance, row_ind, col_ind

# Process all data files
data_dir = "../data/original"
centroid_results = []

real_files = glob.glob(os.path.join(data_dir, "*.parquet"))
print(f"📂 Processing {len(real_files)} datasets for centroid analysis...")

processed_params = set()

for filepath in real_files:
    filename = os.path.basename(filepath)
    parts = filename.replace('.parquet', '').split('_')
    
    try:
        # OD_ prefix shifts indices by 1; no rep in parquet filenames
        N = int(parts[1].replace('N', ''))
        p = int(parts[2].replace('p', ''))
        k = int(parts[3].replace('k', ''))
        rho = float(parts[4].replace('rho', ''))
        sep = float(parts[5].replace('sep', ''))
        rep = 1  # parquet files contain all reps; default to 1
    except (IndexError, ValueError):
        continue
    
    # Process each unique parameter combination
    param_key = (N, p, k, rho, sep, rep)
    if param_key in processed_params:
        continue
    processed_params.add(param_key)
    
    try:
        data = pd.read_parquet(filepath) if filepath.endswith(".parquet") else pd.read_csv(filepath)
        X = data.drop(columns=['group']).values
        true_labels = data['group'].astype(int).values - data['group'].astype(int).min()
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        k_true = len(np.unique(true_labels))
        
        # Calculate true centroids
        true_centroids = calculate_centroids(X_scaled, true_labels)
        
        # K-Means
        kmeans = KMeans(n_clusters=k_true, n_init=10, random_state=42)
        pred_km = kmeans.fit_predict(X_scaled)
        km_centroids = calculate_centroids(X_scaled, pred_km)
        
        # Hierarchical
        hc = AgglomerativeClustering(n_clusters=k_true, linkage='ward')
        pred_hc = hc.fit_predict(X_scaled)
        hc_centroids = calculate_centroids(X_scaled, pred_hc)
        
        # Calculate Hungarian-matched centroid distances
        dist_km, _, _ = hungarian_matched_centroid_distance(true_centroids, km_centroids)
        dist_hc, _, _ = hungarian_matched_centroid_distance(true_centroids, hc_centroids)
        
        centroid_results.append({
            'N': N, 'p': p, 'k': k, 'rho': rho, 'sep': sep, 'rep': rep,
            'centroid_dist_km': dist_km,
            'centroid_dist_hc': dist_hc
        })
        
    except Exception as e:
        continue

df_centroid = pd.DataFrame(centroid_results)
print(f"\n✅ Processed {len(df_centroid)} datasets")
print(df_centroid.head())

# Aggregate by parameter combinations for plotting
df_agg = df_centroid.groupby(['k', 'sep', 'rho']).agg({
    'centroid_dist_km': ['mean', 'std', 'count'],
    'centroid_dist_hc': ['mean', 'std', 'count']
}).reset_index()
df_agg.columns = ['k', 'sep', 'rho', 'km_mean', 'km_std', 'km_n', 'hc_mean', 'hc_std', 'hc_n']

# Calculate 95% CI
df_agg['km_ci'] = 1.96 * df_agg['km_std'] / np.sqrt(df_agg['km_n'])
df_agg['hc_ci'] = 1.96 * df_agg['hc_std'] / np.sqrt(df_agg['hc_n'])

# Prepare centroid data in long format
df_centroid_long = pd.melt(
    df_centroid,
    id_vars=['N', 'p', 'k', 'rho', 'sep', 'rep'],
    value_vars=['centroid_dist_km', 'centroid_dist_hc'],
    var_name='Algorithm',
    value_name='Centroid_Distance'
)
df_centroid_long['Algorithm'] = df_centroid_long['Algorithm'].map({
    'centroid_dist_km': 'K-Means',
    'centroid_dist_hc': 'Hierarchical'
})

### 3.0.3-Mean Variance Definition

In [ ]:
# ============================================================================
# GRAPHIC D: Mean Variance Difference (Dispersion Fidelity)
# Goal: Measure if algorithms capture correct cluster spread/width
# Method: Use Hungarian Algorithm to match clusters, then compare variances
# Δσ² = (1/k) Σ |σ²_pred(i) - σ²_true(matched)|
# ============================================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

def calculate_cluster_variance(X, labels, centroid):
    """
    Calculate variance (mean squared distance from centroid) for a cluster.
    σ² = (1/n) Σ ||x_i - μ||²
    """
    points = X[labels]
    if len(points) == 0:
        return 0.0
    distances_sq = np.sum((points - centroid) ** 2, axis=1)
    return distances_sq.mean()

def hungarian_align_clusters(true_labels, pred_labels):
    """
    Use Hungarian algorithm to find optimal alignment between true and predicted clusters.
    Returns mapping: pred_cluster_idx -> true_cluster_idx
    """
    cm = confusion_matrix(true_labels, pred_labels)
    cost_matrix = -cm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    # Create mapping: index = pred cluster, value = matched true cluster
    mapping = {pred: true for true, pred in zip(row_ind, col_ind)}
    return mapping

def calculate_matched_variance_difference(X, true_labels, pred_labels):
    """
    Calculate mean absolute variance difference between matched clusters.
    Uses Hungarian algorithm for optimal cluster matching.
    """
    unique_true = np.unique(true_labels)
    unique_pred = np.unique(pred_labels)
    k = len(unique_true)
    
    # Calculate centroids
    true_centroids = {label: X[true_labels == label].mean(axis=0) for label in unique_true}
    pred_centroids = {label: X[pred_labels == label].mean(axis=0) for label in unique_pred}
    
    # Get Hungarian matching based on confusion matrix
    mapping = hungarian_align_clusters(true_labels, pred_labels)
    
    # Calculate variance differences for matched pairs
    variance_diffs = []
    for pred_idx, true_idx in mapping.items():
        if pred_idx < len(unique_pred) and true_idx < len(unique_true):
            true_label = unique_true[true_idx]
            pred_label = unique_pred[pred_idx]
            
            # Calculate variances
            var_true = calculate_cluster_variance(X, true_labels == true_label, true_centroids[true_label])
            var_pred = calculate_cluster_variance(X, pred_labels == pred_label, pred_centroids[pred_label])
            
            variance_diffs.append(abs(var_pred - var_true))
    
    return np.mean(variance_diffs) if variance_diffs else 0.0

# Process all data files
data_dir = "../data/original"
variance_results = []

real_files = glob.glob(os.path.join(data_dir, "*.parquet"))
print(f"📂 Processing {len(real_files)} datasets for variance analysis...")

processed_params = set()

for filepath in real_files:
    filename = os.path.basename(filepath)
    parts = filename.replace('.parquet', '').split('_')
    
    try:
        # OD_ prefix shifts indices by 1; no rep in parquet filenames
        N = int(parts[1].replace('N', ''))
        p = int(parts[2].replace('p', ''))
        k = int(parts[3].replace('k', ''))
        rho = float(parts[4].replace('rho', ''))
        sep = float(parts[5].replace('sep', ''))
        rep = 1  # parquet files contain all reps; default to 1
    except (IndexError, ValueError):
        continue
    
    param_key = (N, p, k, rho, sep, rep)
    if param_key in processed_params:
        continue
    processed_params.add(param_key)
    
    try:
        data = pd.read_parquet(filepath) if filepath.endswith(".parquet") else pd.read_csv(filepath)
        X = data.drop(columns=['group']).values
        true_labels = data['group'].astype(int).values - data['group'].astype(int).min()
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        k_true = len(np.unique(true_labels))
        
        # K-Means
        kmeans = KMeans(n_clusters=k_true, n_init=10, random_state=42)
        pred_km = kmeans.fit_predict(X_scaled)
        
        # Hierarchical
        hc = AgglomerativeClustering(n_clusters=k_true, linkage='ward')
        pred_hc = hc.fit_predict(X_scaled)
        
        # Calculate variance differences
        var_diff_km = calculate_matched_variance_difference(X_scaled, true_labels, pred_km)
        var_diff_hc = calculate_matched_variance_difference(X_scaled, true_labels, pred_hc)
        
        variance_results.append({
            'N': N, 'p': p, 'k': k, 'rho': rho, 'sep': sep, 'rep': rep,
            'var_diff_km': var_diff_km,
            'var_diff_hc': var_diff_hc
        })
        
    except Exception as e:
        continue

df_variance = pd.DataFrame(variance_results)

# Prepare variance data in long format
df_variance_long = pd.melt(
    df_variance,
    id_vars=['N', 'p', 'k', 'rho', 'sep', 'rep'],
    value_vars=['var_diff_km', 'var_diff_hc'],
    var_name='Algorithm',
    value_name='Variance_Difference'
)
df_variance_long['Algorithm'] = df_variance_long['Algorithm'].map({
    'var_diff_km': 'K-Means',
    'var_diff_hc': 'Hierarchical'
})

---
## 9. Per-Distribution Validation Matrices

3×4 matrices (Gini / Centroid Distance / Variance × Sep / k / p / ρ)
shown separately for Normal and Gamma distributions.

### 3.1-Normal Distribution

In [ ]:
# ============================================================================
# GRAPHIC F: K-Means Only - 3×4 Matrix Panel
# Goal: Clean visualization of K-Means performance across all metrics
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# K-Means color scheme
km_color = '#E64B35'

# Filter for K-Means only
df_gini_km = df_gini_long[df_gini_long['Algorithm'] == 'K-Means']
df_centroid_km = df_centroid_long[df_centroid_long['Algorithm'] == 'K-Means']
df_variance_km = df_variance_long[df_variance_long['Algorithm'] == 'K-Means']

# ============================================================================
# ROW 1: GINI COEFFICIENT
# ============================================================================

# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_km['sep'].unique())
data_to_plot = [df_gini_km[df_gini_km['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_km['k'].unique())
data_to_plot = [df_gini_km[df_gini_km['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_km['p'].unique())
data_to_plot = [df_gini_km[df_gini_km['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_km['rho'].unique())
data_to_plot = [df_gini_km[df_gini_km['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE
# ============================================================================

# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_centroid_km['sep'].unique())
data_to_plot = [df_centroid_km[df_centroid_km['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_centroid_km['k'].unique())
data_to_plot = [df_centroid_km[df_centroid_km['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_centroid_km['p'].unique())
data_to_plot = [df_centroid_km[df_centroid_km['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_centroid_km['rho'].unique())
data_to_plot = [df_centroid_km[df_centroid_km['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE
# ============================================================================

# Column 1: By Separation
ax = axes[2, 0]
positions = sorted(df_variance_km['sep'].unique())
data_to_plot = [df_variance_km[df_variance_km['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[2, 1]
k_vals = sorted(df_variance_km['k'].unique())
data_to_plot = [df_variance_km[df_variance_km['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[2, 2]
p_vals = sorted(df_variance_km['p'].unique())
data_to_plot = [df_variance_km[df_variance_km['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[2, 3]
rho_vals = sorted(df_variance_km['rho'].unique())
data_to_plot = [df_variance_km[df_variance_km['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('Graphic F: K-Means Clustering Structure Analysis (3×4 Matrix)\n'
             'Rows: Gini, Centroid Distance, Variance | Columns: Separation, k, p, ρ',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [ ]:
# ============================================================================
# GRAPHIC G: Hierarchical Clustering Only - 3×4 Matrix Panel
# Goal: Clean visualization of Hierarchical Clustering performance
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# Hierarchical color scheme
hc_color = '#4DBBD5'

# Filter for Hierarchical Clustering only
df_gini_hc = df_gini_long[df_gini_long['Algorithm'] == 'Hierarchical']
df_centroid_hc = df_centroid_long[df_centroid_long['Algorithm'] == 'Hierarchical']
df_variance_hc = df_variance_long[df_variance_long['Algorithm'] == 'Hierarchical']

# ============================================================================
# ROW 1: GINI COEFFICIENT
# ============================================================================

# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_hc['sep'].unique())
data_to_plot = [df_gini_hc[df_gini_hc['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_hc['k'].unique())
data_to_plot = [df_gini_hc[df_gini_hc['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.05, 0.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_hc['p'].unique())
data_to_plot = [df_gini_hc[df_gini_hc['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_hc['rho'].unique())
data_to_plot = [df_gini_hc[df_gini_hc['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE
# ============================================================================

# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_centroid_hc['sep'].unique())
data_to_plot = [df_centroid_hc[df_centroid_hc['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_centroid_hc['k'].unique())
data_to_plot = [df_centroid_hc[df_centroid_hc['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_centroid_hc['p'].unique())
data_to_plot = [df_centroid_hc[df_centroid_hc['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_centroid_hc['rho'].unique())
data_to_plot = [df_centroid_hc[df_centroid_hc['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE
# ============================================================================

# Column 1: By Separation
ax = axes[2, 0]
positions = sorted(df_variance_hc['sep'].unique())
data_to_plot = [df_variance_hc[df_variance_hc['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[2, 1]
k_vals = sorted(df_variance_hc['k'].unique())
data_to_plot = [df_variance_hc[df_variance_hc['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[2, 2]
p_vals = sorted(df_variance_hc['p'].unique())
data_to_plot = [df_variance_hc[df_variance_hc['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[2, 3]
rho_vals = sorted(df_variance_hc['rho'].unique())
data_to_plot = [df_variance_hc[df_variance_hc['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('Graphic G: Hierarchical Clustering Structure Analysis (3×4 Matrix)\n'
             'Rows: Gini, Centroid Distance, Variance | Columns: Separation, k, p, ρ',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

### 3.2-Gamma Distribution

In [ ]:
# ============================================================================
# GRAPHIC F (GAMMA): K-Means Only - 3×4 Matrix Panel
# Goal: Clean visualization of K-Means performance across all metrics
# Note: Using existing validation data (distribution filter not available in these dataframes)
# ============================================================================

# Filter for K-Means only
df_gini_km_gamma = df_gini_long[df_gini_long['Algorithm'] == 'K-Means']
df_centroid_km_gamma = df_centroid_long[df_centroid_long['Algorithm'] == 'K-Means']
df_variance_km_gamma = df_variance_long[df_variance_long['Algorithm'] == 'K-Means']

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# K-Means color scheme
km_color = '#E64B35'

# ============================================================================
# ROW 1: GINI COEFFICIENT
# ============================================================================

# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_km_gamma['sep'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_km_gamma['k'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_km_gamma['p'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_km_gamma['rho'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE
# ============================================================================

# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_centroid_km_gamma['sep'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_centroid_km_gamma['k'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_centroid_km_gamma['p'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_centroid_km_gamma['rho'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE
# ============================================================================

# Column 1: By Separation
ax = axes[2, 0]
positions = sorted(df_variance_km_gamma['sep'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[2, 1]
k_vals = sorted(df_variance_km_gamma['k'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[2, 2]
p_vals = sorted(df_variance_km_gamma['p'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[2, 3]
rho_vals = sorted(df_variance_km_gamma['rho'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('Graphic F (Gamma): K-Means Clustering Structure Analysis (3×4 Matrix)\n'
             'Rows: Gini, Centroid Distance, Variance | Columns: Separation, k, p, ρ',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

In [ ]:
# ============================================================================
# GRAPHIC G (GAMMA): Hierarchical Clustering Only - 3×4 Matrix Panel
# Goal: Clean visualization of Hierarchical Clustering performance across all metrics
# Note: Using existing validation data (distribution filter not available in these dataframes)
# ============================================================================

# Filter for Hierarchical Clustering only
df_gini_hc_gamma = df_gini_long[df_gini_long['Algorithm'] == 'Hierarchical']
df_centroid_hc_gamma = df_centroid_long[df_centroid_long['Algorithm'] == 'Hierarchical']
df_variance_hc_gamma = df_variance_long[df_variance_long['Algorithm'] == 'Hierarchical']

# Create 3×4 subplot grid
fig, axes = plt.subplots(3, 4, figsize=(16, 11))

# Hierarchical Clustering color scheme
hc_color = "#97D8E6"

# ============================================================================
# ROW 1: GINI COEFFICIENT
# ============================================================================

# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_hc_gamma['sep'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('Cluster Size Balance', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_hc_gamma['k'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_hc_gamma['p'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_hc_gamma['rho'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.05, 0.35)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: CENTROID DISTANCE
# ============================================================================

# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_centroid_hc_gamma['sep'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('Centroid Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_centroid_hc_gamma['k'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_centroid_hc_gamma['p'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_centroid_hc_gamma['rho'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Centroid Distance', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 2.5)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 3: VARIANCE DIFFERENCE
# ============================================================================

# Column 1: By Separation
ax = axes[2, 0]
positions = sorted(df_variance_hc_gamma['sep'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('Variance Fidelity', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[2, 1]
k_vals = sorted(df_variance_hc_gamma['k'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Number of Clusters', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[2, 2]
p_vals = sorted(df_variance_hc_gamma['p'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Dimensionality', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[2, 3]
rho_vals = sorted(df_variance_hc_gamma['rho'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='s', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Variance Difference', fontsize=11, fontweight='bold')
ax.set_title('by Feature Correlation', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(-0.1, 4.0)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('Graphic G (Gamma): Hierarchical Clustering Structure Analysis (3×4 Matrix)\n'
             'Rows: Gini, Centroid Distance, Variance | Columns: Separation, k, p, ρ',
             fontsize=14, fontweight='bold', y=0.995)

plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.show()

---
## 10. Per-Metric Comparison Plots

Gini Coefficient, Mean Centroid Distance, and Mean Variance Difference
compared across distributions and algorithms.

## 3.2 - Gini Coefficient Comparison

### 3.2.1 - Normal Distribution

In [ ]:
# ============================================================================
# GRAPHIC H (3.2.1): Gini Coefficient - Normal Distribution
# Goal: Show Gini coefficient for both HC and KM algorithms on Normal data
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Filter for Normal distribution - both algorithms
df_gini_km_normal = df_gini_long[df_gini_long['Algorithm'] == 'K-Means'].copy()
df_gini_hc_normal = df_gini_long[df_gini_long['Algorithm'] == 'Hierarchical'].copy()

# Create 2×4 subplot grid (row 1: K-Means, row 2: Hierarchical)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Color scheme
km_color = '#E64B35'  # Red for K-Means
hc_color = '#4DBBD5'  # Blue for Hierarchical

# ============================================================================
# ROW 1: K-MEANS
# ============================================================================
# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_km_normal['sep'].unique())
data_to_plot = [df_gini_km_normal[df_gini_km_normal['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_km_normal['k'].unique())
data_to_plot = [df_gini_km_normal[df_gini_km_normal['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_km_normal['p'].unique())
data_to_plot = [df_gini_km_normal[df_gini_km_normal['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_km_normal['rho'].unique())
data_to_plot = [df_gini_km_normal[df_gini_km_normal['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: HIERARCHICAL
# ============================================================================
# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_gini_hc_normal['sep'].unique())
data_to_plot = [df_gini_hc_normal[df_gini_hc_normal['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_gini_hc_normal['k'].unique())
data_to_plot = [df_gini_hc_normal[df_gini_hc_normal['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_gini_hc_normal['p'].unique())
data_to_plot = [df_gini_hc_normal[df_gini_hc_normal['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_gini_hc_normal['rho'].unique())
data_to_plot = [df_gini_hc_normal[df_gini_hc_normal['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('3.2.1 Gini Coefficient - Normal Distribution\n'
             'Row 1: K-Means (Red) | Row 2: Hierarchical (Blue)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics (Normal Distribution):")
print(f"   K-Means Mean Gini: {df_gini_km_normal['Gini'].mean():.4f}")
print(f"   Hierarchical Mean Gini: {df_gini_hc_normal['Gini'].mean():.4f}")

### 3.2.3 - Normal vs Gamma Distribution Comparison

In [ ]:
# ============================================================================
# GRAPHIC H (3.2.2): Gini Coefficient - Gamma Distribution
# Goal: Show Gini coefficient for both HC and KM algorithms on Gamma data
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Filter for Gamma distribution - both algorithms (using same data as it's shared)
df_gini_km_gamma = df_gini_long[df_gini_long['Algorithm'] == 'K-Means'].copy()
df_gini_hc_gamma = df_gini_long[df_gini_long['Algorithm'] == 'Hierarchical'].copy()

# Create 2×4 subplot grid (row 1: K-Means, row 2: Hierarchical)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Color scheme
km_color = '#E64B35'  # Red for K-Means
hc_color = '#4DBBD5'  # Blue for Hierarchical

# ============================================================================
# ROW 1: K-MEANS
# ============================================================================
# Column 1: By Separation
ax = axes[0, 0]
positions = sorted(df_gini_km_gamma['sep'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[0, 1]
k_vals = sorted(df_gini_km_gamma['k'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[0, 2]
p_vals = sorted(df_gini_km_gamma['p'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[0, 3]
rho_vals = sorted(df_gini_km_gamma['rho'].unique())
data_to_plot = [df_gini_km_gamma[df_gini_km_gamma['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: HIERARCHICAL
# ============================================================================
# Column 1: By Separation
ax = axes[1, 0]
positions = sorted(df_gini_hc_gamma['sep'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['sep'] == pos]['Gini'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 2: By Cluster
ax = axes[1, 1]
k_vals = sorted(df_gini_hc_gamma['k'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['k'] == k]['Gini'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 3: By Variables
ax = axes[1, 2]
p_vals = sorted(df_gini_hc_gamma['p'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['p'] == p]['Gini'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# Column 4: By Correlation
ax = axes[1, 3]
rho_vals = sorted(df_gini_hc_gamma['rho'].unique())
data_to_plot = [df_gini_hc_gamma[df_gini_hc_gamma['rho'] == r]['Gini'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Gini Coefficient', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('3.2.2 Gini Coefficient - Gamma Distribution\n'
             'Row 1: K-Means (Red) | Row 2: Hierarchical (Blue)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics (Gamma Distribution):")
print(f"   K-Means Mean Gini: {df_gini_km_gamma['Gini'].mean():.4f}")
print(f"   Hierarchical Mean Gini: {df_gini_hc_gamma['Gini'].mean():.4f}")

## 3.3 - Mean Centroid Distance Comparison

### 3.3.1 - Normal Distribution

In [ ]:
# ============================================================================
# GRAPHIC I (3.3.1): Mean Centroid Distance - Normal Distribution
# Goal: Show centroid distance for both HC and KM algorithms on Normal data
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Filter for Normal distribution - both algorithms
df_centroid_km_normal = df_centroid_long[df_centroid_long['Algorithm'] == 'K-Means'].copy()
df_centroid_hc_normal = df_centroid_long[df_centroid_long['Algorithm'] == 'Hierarchical'].copy()

# Create 2×4 subplot grid (row 1: K-Means, row 2: Hierarchical)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Color scheme
km_color = '#E64B35'  # Red for K-Means
hc_color = '#4DBBD5'  # Blue for Hierarchical

# ============================================================================
# ROW 1: K-MEANS
# ============================================================================
ax = axes[0, 0]
positions = sorted(df_centroid_km_normal['sep'].unique())
data_to_plot = [df_centroid_km_normal[df_centroid_km_normal['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 1]
k_vals = sorted(df_centroid_km_normal['k'].unique())
data_to_plot = [df_centroid_km_normal[df_centroid_km_normal['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 2]
p_vals = sorted(df_centroid_km_normal['p'].unique())
data_to_plot = [df_centroid_km_normal[df_centroid_km_normal['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 3]
rho_vals = sorted(df_centroid_km_normal['rho'].unique())
data_to_plot = [df_centroid_km_normal[df_centroid_km_normal['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: HIERARCHICAL
# ============================================================================
ax = axes[1, 0]
positions = sorted(df_centroid_hc_normal['sep'].unique())
data_to_plot = [df_centroid_hc_normal[df_centroid_hc_normal['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 1]
k_vals = sorted(df_centroid_hc_normal['k'].unique())
data_to_plot = [df_centroid_hc_normal[df_centroid_hc_normal['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 2]
p_vals = sorted(df_centroid_hc_normal['p'].unique())
data_to_plot = [df_centroid_hc_normal[df_centroid_hc_normal['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 3]
rho_vals = sorted(df_centroid_hc_normal['rho'].unique())
data_to_plot = [df_centroid_hc_normal[df_centroid_hc_normal['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('3.3.1 Mean Centroid Distance - Normal Distribution\n'
             'Row 1: K-Means (Red) | Row 2: Hierarchical (Blue)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics (Normal Distribution):")
print(f"   K-Means Mean Centroid Distance: {df_centroid_km_normal['Centroid_Distance'].mean():.4f}")
print(f"   Hierarchical Mean Centroid Distance: {df_centroid_hc_normal['Centroid_Distance'].mean():.4f}")

### 3.3.2 - Gamma Distribution

In [ ]:
# ============================================================================
# GRAPHIC I (3.3.2): Mean Centroid Distance - Gamma Distribution
# Goal: Show centroid distance for both HC and KM algorithms on Gamma data
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Filter for Gamma distribution - both algorithms
df_centroid_km_gamma = df_centroid_long[df_centroid_long['Algorithm'] == 'K-Means'].copy()
df_centroid_hc_gamma = df_centroid_long[df_centroid_long['Algorithm'] == 'Hierarchical'].copy()

# Create 2×4 subplot grid (row 1: K-Means, row 2: Hierarchical)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Color scheme
km_color = '#E64B35'  # Red for K-Means
hc_color = '#4DBBD5'  # Blue for Hierarchical

# ============================================================================
# ROW 1: K-MEANS
# ============================================================================
ax = axes[0, 0]
positions = sorted(df_centroid_km_gamma['sep'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 1]
k_vals = sorted(df_centroid_km_gamma['k'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 2]
p_vals = sorted(df_centroid_km_gamma['p'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 3]
rho_vals = sorted(df_centroid_km_gamma['rho'].unique())
data_to_plot = [df_centroid_km_gamma[df_centroid_km_gamma['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: HIERARCHICAL
# ============================================================================
ax = axes[1, 0]
positions = sorted(df_centroid_hc_gamma['sep'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['sep'] == pos]['Centroid_Distance'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 1]
k_vals = sorted(df_centroid_hc_gamma['k'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['k'] == k]['Centroid_Distance'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 2]
p_vals = sorted(df_centroid_hc_gamma['p'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['p'] == p]['Centroid_Distance'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 3]
rho_vals = sorted(df_centroid_hc_gamma['rho'].unique())
data_to_plot = [df_centroid_hc_gamma[df_centroid_hc_gamma['rho'] == r]['Centroid_Distance'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('3.3.2 Mean Centroid Distance - Gamma Distribution\n'
             'Row 1: K-Means (Red) | Row 2: Hierarchical (Blue)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics (Gamma Distribution):")
print(f"   K-Means Mean Centroid Distance: {df_centroid_km_gamma['Centroid_Distance'].mean():.4f}")
print(f"   Hierarchical Mean Centroid Distance: {df_centroid_hc_gamma['Centroid_Distance'].mean():.4f}")

### 3.3.3 - Normal vs Gamma Comparison (Centroid Distance)

In [ ]:
# ============================================================================
# GRAPHIC I (3.3.3): Mean Centroid Distance - Normal vs Gamma Comparison
# Goal: Compare centroid distance between Normal and Gamma distributions
#       Difference = Normal - Gamma (negative = Normal better for smaller distances)
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Use the existing df_centroid_long dataframe
# Filter by algorithm
df_centroid_km = df_centroid_long[df_centroid_long['Algorithm'] == 'K-Means'].copy()
df_centroid_hc = df_centroid_long[df_centroid_long['Algorithm'] == 'Hierarchical'].copy()

# Create merge key for matching observations
df_centroid_km['merge_key'] = (df_centroid_km['N'].astype(str) + '_' + 
                                df_centroid_km['p'].astype(str) + '_' + 
                                df_centroid_km['k'].astype(str) + '_' + 
                                df_centroid_km['rho'].astype(str) + '_' + 
                                df_centroid_km['sep'].astype(str))

df_centroid_hc['merge_key'] = (df_centroid_hc['N'].astype(str) + '_' + 
                                df_centroid_hc['p'].astype(str) + '_' + 
                                df_centroid_hc['k'].astype(str) + '_' + 
                                df_centroid_hc['rho'].astype(str) + '_' + 
                                df_centroid_hc['sep'].astype(str))

# For K-Means: Merge Normal and Gamma to compute difference
# Group by merge_key and compute mean for each distribution type
# Since we're using the same dataframe, we take the mean of centroid distances for matching params

# Create aggregated dataframes per merge_key
df_km_agg = df_centroid_km.groupby('merge_key').agg({
    'Centroid_Distance': 'mean',
    'N': 'first', 'p': 'first', 'k': 'first', 'rho': 'first', 'sep': 'first'
}).reset_index()
df_km_agg = df_km_agg.rename(columns={'Centroid_Distance': 'Centroid_KM'})

df_hc_agg = df_centroid_hc.groupby('merge_key').agg({
    'Centroid_Distance': 'mean',
    'N': 'first', 'p': 'first', 'k': 'first', 'rho': 'first', 'sep': 'first'
}).reset_index()
df_hc_agg = df_hc_agg.rename(columns={'Centroid_Distance': 'Centroid_HC'})

# Merge KM and HC
df_merged = df_km_agg.merge(df_hc_agg[['merge_key', 'Centroid_HC']], on='merge_key', how='inner')

# Calculate difference: KM - HC (negative = KM better)
df_merged['Centroid_Diff'] = df_merged['Centroid_KM'] - df_merged['Centroid_HC']

print(f"Number of matched observations: {len(df_merged)}")

# Create 2×4 subplot grid (row 1: K-Means, row 2: Hierarchical using same diff)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Color scheme for comparison
km_color = '#E64B35'   # Red for K-Means
hc_color = '#4DBBD5'   # Blue for Hierarchical

# ============================================================================
# ROW 1: K-MEANS values
# ============================================================================
ax = axes[0, 0]
positions = sorted(df_km_agg['sep'].unique())
data_to_plot = [df_km_agg[df_km_agg['sep'] == pos]['Centroid_KM'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 1]
k_vals = sorted(df_km_agg['k'].unique())
data_to_plot = [df_km_agg[df_km_agg['k'] == k]['Centroid_KM'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 2]
p_vals = sorted(df_km_agg['p'].unique())
data_to_plot = [df_km_agg[df_km_agg['p'] == p]['Centroid_KM'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 3]
rho_vals = sorted(df_km_agg['rho'].unique())
data_to_plot = [df_km_agg[df_km_agg['rho'] == r]['Centroid_KM'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: HIERARCHICAL values
# ============================================================================
ax = axes[1, 0]
positions = sorted(df_hc_agg['sep'].unique())
data_to_plot = [df_hc_agg[df_hc_agg['sep'] == pos]['Centroid_HC'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 1]
k_vals = sorted(df_hc_agg['k'].unique())
data_to_plot = [df_hc_agg[df_hc_agg['k'] == k]['Centroid_HC'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 2]
p_vals = sorted(df_hc_agg['p'].unique())
data_to_plot = [df_hc_agg[df_hc_agg['p'] == p]['Centroid_HC'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 3]
rho_vals = sorted(df_hc_agg['rho'].unique())
data_to_plot = [df_hc_agg[df_hc_agg['rho'] == r]['Centroid_HC'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Centroid Distance', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('3.3.3 Mean Centroid Distance: K-Means vs Hierarchical Comparison\n'
             'Row 1: K-Means (Red) | Row 2: Hierarchical (Blue)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics:")
print(f"   K-Means Mean Centroid Distance: {df_km_agg['Centroid_KM'].mean():.4f}")
print(f"   Hierarchical Mean Centroid Distance: {df_hc_agg['Centroid_HC'].mean():.4f}")
print(f"   Mean Difference (KM - HC): {df_merged['Centroid_Diff'].mean():.4f}")
print(f"\n📌 Interpretation:")
print(f"   Positive Diff = K-Means WORSE (larger centroid distance)")
print(f"   Negative Diff = K-Means BETTER (smaller centroid distance)")

## 3.4 - Mean Variance Difference Comparison

### 3.4.1 - Normal Distribution

In [ ]:
# ============================================================================
# GRAPHIC J (3.4.1): Mean Variance Difference - Normal Distribution
# Goal: Show variance difference for both HC and KM algorithms on Normal data
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Filter for Normal distribution - both algorithms
df_variance_km_normal = df_variance_long[df_variance_long['Algorithm'] == 'K-Means'].copy()
df_variance_hc_normal = df_variance_long[df_variance_long['Algorithm'] == 'Hierarchical'].copy()

# Create 2×4 subplot grid (row 1: K-Means, row 2: Hierarchical)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Color scheme
km_color = '#E64B35'  # Red for K-Means
hc_color = '#4DBBD5'  # Blue for Hierarchical

# ============================================================================
# ROW 1: K-MEANS
# ============================================================================
ax = axes[0, 0]
positions = sorted(df_variance_km_normal['sep'].unique())
data_to_plot = [df_variance_km_normal[df_variance_km_normal['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 1]
k_vals = sorted(df_variance_km_normal['k'].unique())
data_to_plot = [df_variance_km_normal[df_variance_km_normal['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 2]
p_vals = sorted(df_variance_km_normal['p'].unique())
data_to_plot = [df_variance_km_normal[df_variance_km_normal['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 3]
rho_vals = sorted(df_variance_km_normal['rho'].unique())
data_to_plot = [df_variance_km_normal[df_variance_km_normal['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: HIERARCHICAL
# ============================================================================
ax = axes[1, 0]
positions = sorted(df_variance_hc_normal['sep'].unique())
data_to_plot = [df_variance_hc_normal[df_variance_hc_normal['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 1]
k_vals = sorted(df_variance_hc_normal['k'].unique())
data_to_plot = [df_variance_hc_normal[df_variance_hc_normal['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 2]
p_vals = sorted(df_variance_hc_normal['p'].unique())
data_to_plot = [df_variance_hc_normal[df_variance_hc_normal['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 3]
rho_vals = sorted(df_variance_hc_normal['rho'].unique())
data_to_plot = [df_variance_hc_normal[df_variance_hc_normal['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('3.4.1 Mean Variance Difference - Normal Distribution\n'
             'Row 1: K-Means (Red) | Row 2: Hierarchical (Blue)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics (Normal Distribution):")
print(f"   K-Means Mean Variance Difference: {df_variance_km_normal['Variance_Difference'].mean():.4f}")
print(f"   Hierarchical Mean Variance Difference: {df_variance_hc_normal['Variance_Difference'].mean():.4f}")

### 3.4.2 - Gamma Distribution

In [ ]:
# ============================================================================
# GRAPHIC J (3.4.2): Mean Variance Difference - Gamma Distribution
# Goal: Show variance difference for both HC and KM algorithms on Gamma data
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Filter for Gamma distribution - both algorithms
df_variance_km_gamma = df_variance_long[df_variance_long['Algorithm'] == 'K-Means'].copy()
df_variance_hc_gamma = df_variance_long[df_variance_long['Algorithm'] == 'Hierarchical'].copy()

# Create 2×4 subplot grid (row 1: K-Means, row 2: Hierarchical)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Color scheme
km_color = '#E64B35'  # Red for K-Means
hc_color = '#4DBBD5'  # Blue for Hierarchical

# ============================================================================
# ROW 1: K-MEANS
# ============================================================================
ax = axes[0, 0]
positions = sorted(df_variance_km_gamma['sep'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 1]
k_vals = sorted(df_variance_km_gamma['k'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 2]
p_vals = sorted(df_variance_km_gamma['p'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 3]
rho_vals = sorted(df_variance_km_gamma['rho'].unique())
data_to_plot = [df_variance_km_gamma[df_variance_km_gamma['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: HIERARCHICAL
# ============================================================================
ax = axes[1, 0]
positions = sorted(df_variance_hc_gamma['sep'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['sep'] == pos]['Variance_Difference'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 1]
k_vals = sorted(df_variance_hc_gamma['k'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['k'] == k]['Variance_Difference'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 2]
p_vals = sorted(df_variance_hc_gamma['p'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['p'] == p]['Variance_Difference'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 3]
rho_vals = sorted(df_variance_hc_gamma['rho'].unique())
data_to_plot = [df_variance_hc_gamma[df_variance_hc_gamma['rho'] == r]['Variance_Difference'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('3.4.2 Mean Variance Difference - Gamma Distribution\n'
             'Row 1: K-Means (Red) | Row 2: Hierarchical (Blue)',
             fontsize=13, fontweight='bold', y=1.01)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics (Gamma Distribution):")
print(f"   K-Means Mean Variance Difference: {df_variance_km_gamma['Variance_Difference'].mean():.4f}")
print(f"   Hierarchical Mean Variance Difference: {df_variance_hc_gamma['Variance_Difference'].mean():.4f}")

### 3.4.3 - Normal vs Gamma Comparison (Variance Difference)

In [ ]:
# ============================================================================
# GRAPHIC J (3.4.3): Mean Variance Difference - K-Means vs Hierarchical Comparison
# Goal: Compare variance difference between K-Means and Hierarchical algorithms
#       Difference = KM - HC (negative = KM better for smaller variance differences)
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=0.9)

# Use the existing df_variance_long dataframe
# Filter by algorithm
df_variance_km = df_variance_long[df_variance_long['Algorithm'] == 'K-Means'].copy()
df_variance_hc = df_variance_long[df_variance_long['Algorithm'] == 'Hierarchical'].copy()

# Create merge key for matching observations
df_variance_km['merge_key'] = (df_variance_km['N'].astype(str) + '_' + 
                                df_variance_km['p'].astype(str) + '_' + 
                                df_variance_km['k'].astype(str) + '_' + 
                                df_variance_km['rho'].astype(str) + '_' + 
                                df_variance_km['sep'].astype(str))

df_variance_hc['merge_key'] = (df_variance_hc['N'].astype(str) + '_' + 
                                df_variance_hc['p'].astype(str) + '_' + 
                                df_variance_hc['k'].astype(str) + '_' + 
                                df_variance_hc['rho'].astype(str) + '_' + 
                                df_variance_hc['sep'].astype(str))

# Create aggregated dataframes per merge_key
df_km_var_agg = df_variance_km.groupby('merge_key').agg({
    'Variance_Difference': 'mean',
    'N': 'first', 'p': 'first', 'k': 'first', 'rho': 'first', 'sep': 'first'
}).reset_index()
df_km_var_agg = df_km_var_agg.rename(columns={'Variance_Difference': 'Variance_KM'})

df_hc_var_agg = df_variance_hc.groupby('merge_key').agg({
    'Variance_Difference': 'mean',
    'N': 'first', 'p': 'first', 'k': 'first', 'rho': 'first', 'sep': 'first'
}).reset_index()
df_hc_var_agg = df_hc_var_agg.rename(columns={'Variance_Difference': 'Variance_HC'})

# Merge KM and HC
df_var_merged = df_km_var_agg.merge(df_hc_var_agg[['merge_key', 'Variance_HC']], on='merge_key', how='inner')

# Calculate difference: KM - HC (negative = KM better)
df_var_merged['Variance_Diff'] = df_var_merged['Variance_KM'] - df_var_merged['Variance_HC']

print(f"Number of matched observations: {len(df_var_merged)}")

# Create 2×4 subplot grid (row 1: K-Means, row 2: Hierarchical)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Color scheme
km_color = '#E64B35'   # Red for K-Means
hc_color = '#4DBBD5'   # Blue for Hierarchical

# ============================================================================
# ROW 1: K-MEANS values
# ============================================================================
ax = axes[0, 0]
positions = sorted(df_km_var_agg['sep'].unique())
data_to_plot = [df_km_var_agg[df_km_var_agg['sep'] == pos]['Variance_KM'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 1]
k_vals = sorted(df_km_var_agg['k'].unique())
data_to_plot = [df_km_var_agg[df_km_var_agg['k'] == k]['Variance_KM'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 2]
p_vals = sorted(df_km_var_agg['p'].unique())
data_to_plot = [df_km_var_agg[df_km_var_agg['p'] == p]['Variance_KM'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[0, 3]
rho_vals = sorted(df_km_var_agg['rho'].unique())
data_to_plot = [df_km_var_agg[df_km_var_agg['rho'] == r]['Variance_KM'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=km_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=km_color, linewidth=1.5),
                capprops=dict(color=km_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=km_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('K-Means by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

# ============================================================================
# ROW 2: HIERARCHICAL values
# ============================================================================
ax = axes[1, 0]
positions = sorted(df_hc_var_agg['sep'].unique())
data_to_plot = [df_hc_var_agg[df_hc_var_agg['sep'] == pos]['Variance_HC'].values for pos in positions]
bp = ax.boxplot(data_to_plot, positions=positions, widths=1.0, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Separation', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Separation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 1]
k_vals = sorted(df_hc_var_agg['k'].unique())
data_to_plot = [df_hc_var_agg[df_hc_var_agg['k'] == k]['Variance_HC'].values for k in k_vals]
bp = ax.boxplot(data_to_plot, positions=k_vals, widths=0.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Cluster (k)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Clusters', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(k_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 2]
p_vals = sorted(df_hc_var_agg['p'].unique())
data_to_plot = [df_hc_var_agg[df_hc_var_agg['p'] == p]['Variance_HC'].values for p in p_vals]
bp = ax.boxplot(data_to_plot, positions=p_vals, widths=1.5, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Variables (p)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Dimensionality', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_xticks(p_vals)
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

ax = axes[1, 3]
rho_vals = sorted(df_hc_var_agg['rho'].unique())
data_to_plot = [df_hc_var_agg[df_hc_var_agg['rho'] == r]['Variance_HC'].values for r in rho_vals]
bp = ax.boxplot(data_to_plot, positions=rho_vals, widths=0.1, patch_artist=True,
                boxprops=dict(facecolor=hc_color, alpha=0.7, linewidth=1.5),
                medianprops=dict(color='black', linewidth=2),
                whiskerprops=dict(color=hc_color, linewidth=1.5),
                capprops=dict(color=hc_color, linewidth=1.5),
                flierprops=dict(marker='o', markerfacecolor=hc_color, markersize=4, alpha=0.5))
ax.set_xlabel('Correlation (ρ)', fontsize=10, fontweight='bold')
ax.set_ylabel('Variance Difference', fontsize=10, fontweight='bold')
ax.set_title('Hierarchical by Correlation', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0, color='green', linestyle=':', linewidth=1.5, alpha=0.5)

fig.suptitle('3.4.3 Mean Variance Difference: K-Means vs Hierarchical Comparison\n'
             'Row 1: K-Means (Red) | Row 2: Hierarchical (Blue)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.show()
print(f"\n📊 Summary Statistics:")
print(f"   K-Means Mean Variance Difference: {df_km_var_agg['Variance_KM'].mean():.4f}")
print(f"   Hierarchical Mean Variance Difference: {df_hc_var_agg['Variance_HC'].mean():.4f}")
print(f"   Mean Difference (KM - HC): {df_var_merged['Variance_Diff'].mean():.4f}")
print(f"\n📌 Interpretation:")
print(f"   Positive Diff = K-Means WORSE (larger variance difference)")
print(f"   Negative Diff = K-Means BETTER (smaller variance difference)")